# Setup Workspace

In [1]:
!pip install tensorflow

  Using cached tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.5 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-manylinux2010_x86_64.whl.metadata (5.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-6.33.5-cp39-abi3-manylinux2014_x86_64.whl.metadata (593 bytes)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached setuptools-82.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached wrapt-2.1.1-cp31

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [2]:
!pip install keras

In [5]:
!pip install pandas

  Using cached pandas-3.0.1-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
Using cached pandas-3.0.1-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (10.9 MB)


In [3]:
!pip install ltn

In [14]:
import ltn
import keras
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
from collections import defaultdict
from tensorflow.keras.preprocessing import image

print(keras.__version__)
""" HYPERPARAMETERS """


DATA_ROOT = Path("data")
BATCH_SIZE = 64
COMPLEXITY_DATASET = 1
EPOCHS = 50
TOTAL_EXAMPLES = 15000
TRAIN_TEST_LIMIT = 12000
LEARNING_RATE = 0.001
csv_path = "results.csv"
txt_path = "stats.txt"
confusion_matrix_path = "conf_matrix.txt"

3.13.2


In [ ]:
!ls /content/drive/MyDrive/datasetVCB/c1

ls: cannot access '/content/drive/MyDrive/datasetVCB/C1': No such file or directory


In [ ]:
!unzip "./drive/MyDrive/Colab Notebooks/C1.zip" -d "/content"

In [8]:
EPOCHS = 100

# Testes para Reunião de 22.5

Teste 1

In [24]:

""" DATASET """
data_path = Path(DATA_ROOT, "C" + str(COMPLEXITY_DATASET))
labels = Path(DATA_ROOT, "all.csv")
images = Path(DATA_ROOT, "c1")

df = pd.read_csv(labels, index_col=0)
df["id"] = df["id"].apply(lambda x: f"{x}.jpg")
# CLASSES = df.columns.tolist()[1:]
CLASSES = [
    "Cafe",
    "Hotel",
    "Store",
    "MiscCommercial",
    "Awning",
    "Billboard",
    "VendingMachine",
    "Statue",
    "Table",
    "WallSign",
]

datagen = image.ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=0,
    width_shift_range=0,
    height_shift_range=0,
    shear_range=0,
    zoom_range=0,
    horizontal_flip=False,
    fill_mode="nearest",
)

train_generator = datagen.flow_from_dataframe(
    dataframe=df[:TRAIN_TEST_LIMIT],
    directory=str(images),
    x_col="id",
    y_col=CLASSES,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False,
)

test_generator = datagen.flow_from_dataframe(
    dataframe=df[TRAIN_TEST_LIMIT:TOTAL_EXAMPLES],
    directory=str(images),
    x_col="id",
    y_col=CLASSES,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False,
)


"""CNN MODEL"""


class CNNModel(keras.Model):

    def __init__(self, num_classes):
        super(CNNModel, self).__init__(name="cnn_model")
        self.conv1 = keras.layers.Conv2D(
            16, kernel_size=(3, 3), activation="relu", input_shape=(224, 224, 3)
        )
        self.conv2 = keras.layers.Conv2D(32, kernel_size=(3, 3), activation="relu")
        self.conv3 = keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu")
        self.maxpool = keras.layers.MaxPooling2D(pool_size=(2, 2))
        self.flatten = keras.layers.Flatten()
        self.dropout = keras.layers.Dropout(0.5)
        self.dense1 = keras.layers.Dense(128, activation="relu")
        self.dense2 = keras.layers.Dense(64, activation="relu")
        self.concept_layer = keras.layers.Dense(32, activation="relu")
        self.output_layer = keras.layers.Dense(num_classes)

    def call(self, inputs):
        x = self.conv1(inputs[0]) #thinha um 0
        x = self.maxpool(x)
        x = self.conv2(x)
        x = self.maxpool(x)
        x = self.conv3(x)
        x = self.maxpool(x)
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.dense1(x)
        x = self.dropout(x)
        x = self.dense2(x)
        y = self.concept_layer(x)
        x = self.output_layer(y)
        result = tf.concat([x, y], axis=1)
        return result


""" LTN MODEL """

# Predicates
logits_model = CNNModel(num_classes=4)
p = ltn.Predicate.FromLogits(
    logits_model, activation_function="softmax", with_class_indexing=True, name="p_model"
)
# pf = ltn.Predicate.FromLogits(logits_model, activation_function="softmax")

# Constants
class_cafe = ltn.Constant(CLASSES.index("Cafe"), trainable=False)
class_hotel = ltn.Constant(CLASSES.index("Hotel"), trainable=False)
class_store = ltn.Constant(CLASSES.index("Store"), trainable=False)
class_misc_commercial = ltn.Constant(CLASSES.index("MiscCommercial"), trainable=False)
class_awning = ltn.Constant(CLASSES.index("Awning"), trainable=False)
class_billboard = ltn.Constant(CLASSES.index("Billboard"), trainable=False)
class_vending_machine = ltn.Constant(CLASSES.index("VendingMachine"), trainable=False)
class_statue = ltn.Constant(CLASSES.index("Statue"), trainable=False)
class_table = ltn.Constant(CLASSES.index("Table"), trainable=False)
class_wall_sign = ltn.Constant(CLASSES.index("WallSign"), trainable=False)


# Operators
Not = ltn.Wrapper_Connective(ltn.fuzzy_ops.Not_Std())
And = ltn.Wrapper_Connective(ltn.fuzzy_ops.And_Prod())
Or = ltn.Wrapper_Connective(ltn.fuzzy_ops.Or_ProbSum())
Implies = ltn.Wrapper_Connective(ltn.fuzzy_ops.Implies_Reichenbach())
Equiv = ltn.Wrapper_Connective(
    ltn.fuzzy_ops.Equiv(ltn.fuzzy_ops.And_Prod(), ltn.fuzzy_ops.Implies_Reichenbach())
)
Forall = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMeanError(), semantics="forall")
Exists = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMean(), semantics="exists")

formula_aggregator = ltn.Wrapper_Formula_Aggregator(
    ltn.fuzzy_ops.Aggreg_pMeanError(p=2)
)


@tf.function()
def axioms(features_param, *args):
    x = ltn.Variable("x", features_param)
    x_hotel = ltn.Variable("x_hotel", features_param[args[CLASSES.index("Hotel")] == 1])
    x_not_hotel = ltn.Variable(
        "x_not_hotel", features_param[args[CLASSES.index("Hotel")] == 0]
    )
    x_cafe = ltn.Variable("x_cafe", features_param[args[CLASSES.index("Cafe")] == 1])
    x_not_cafe = ltn.Variable(
        "x_not_cafe", features_param[args[CLASSES.index("Cafe")] == 0]
    )
    x_store = ltn.Variable("x_store", features_param[args[CLASSES.index("Store")] == 1])
    x_not_store = ltn.Variable(
        "x_not_store", features_param[args[CLASSES.index("Store")] == 0]
    )
    x_misc_commercial = ltn.Variable(
        "x_misc_commercial", features_param[args[CLASSES.index("MiscCommercial")] == 1]
    )
    x_not_misc_commercial = ltn.Variable(
        "x_not_misc_commercial",
        features_param[args[CLASSES.index("MiscCommercial")] == 0],
    )
    axioms = [
        # Grounded Predicates
        Forall(x_hotel, p([x_hotel, class_hotel])),
        Forall(x_not_hotel, Not(p([x_not_hotel, class_hotel]))),
        Forall(x_cafe, p([x_cafe, class_cafe])),
        Forall(x_not_cafe, Not(p([x_not_cafe, class_cafe]))),
        Forall(x_store, p([x_store, class_store])),
        Forall(x_not_store, Not(p([x_not_store, class_store]))),
        Forall(x_misc_commercial, p([x_misc_commercial, class_misc_commercial])),
        Forall(
            x_not_misc_commercial,
            Not(p([x_not_misc_commercial, class_misc_commercial])),
        ),
        # Axioms
        # Cafe = Statue or Vending Machine
        Forall(
            x,
            Equiv(
                p([x, class_cafe]),
                Or(p([x, class_statue]), p([x, class_vending_machine])),
            ),
        ),
        # Hotel = Wall Sign
        Forall(
            x,
            Equiv(p([x, class_hotel]), p([x, class_wall_sign])),
        ),
        # MiscCommercial = Awning and Table
        Forall(
            x,
            Equiv(
                p([x, class_misc_commercial]),
                And(p([x, class_awning]), p([x, class_table])),
            ),
        ),
        # Store = Billboard
        Forall(
            x,
            Equiv(p([x, class_store]), p([x, class_billboard])),
        ),
    ]
    sat_level = formula_aggregator(axioms).tensor
    return sat_level


def __get_elements_from_batch(batch_elements):
    features_calculated = batch_elements[0]
    inverted_labels_list = batch_elements[1]
    labels_list = np.transpose(inverted_labels_list)
    return features_calculated, labels_list


for _ in range(len(train_generator)):
    batch_elements = next(train_generator)
    features, labels_list = __get_elements_from_batch(batch_elements)
    print("Initial sat level %.5f" % axioms(batch_elements[0], *labels_list))
    break


""" TRAINING FUNCTIONS """

metrics_dict = {
    "train_sat_kb": keras.metrics.Mean(name="train_sat_kb"),
    "test_sat_kb": keras.metrics.Mean(name="test_sat_kb"),
}


class_train_accuracy_metric = {
    f"{class_name.lower()}_train_accuracy": keras.metrics.Mean(
        name=f"{class_name.lower()}_train_accuracy"
    )
    for class_name in CLASSES
}
metrics_dict.update(class_train_accuracy_metric)


optimizer = keras.optimizers.legacy.Adam(learning_rate=LEARNING_RATE)


@tf.function()
def train_step(*args):
    features_param = args[0]
    labels_list = args[1:]
    with tf.GradientTape() as tape:
        sat = axioms(features_param, *labels_list)
        loss = 1.0 - sat
    gradients = tape.gradient(loss, p.trainable_variables)
    optimizer.apply_gradients(zip(gradients, p.trainable_variables))
    tf.print("Sat Function Result: ", sat)
    metrics_dict["train_sat_kb"](sat)
    predictions = logits_model([features_param])
    calculate_accuracy_for_each_class(predictions, labels_list)


@tf.function()
def test_step(*args):
    features_param = args[0]
    labels_list = args[1:]
    sat_kb = axioms(features_param, *labels_list)
    metrics_dict["test_sat_kb"](sat_kb)


""" AUXILIARY FUNCTIONS """


def calculate_accuracy_for_each_class(predictions, labels_param):
    for i, class_name in enumerate(CLASSES):
        class_labels = labels_param[i]
        y_pred = predictions[:, i]
        y_pred = tf.math.sigmoid(y_pred)
        y_pred = tf.cast(y_pred > 0.5, tf.float32)
        cast_class_labels = tf.cast(class_labels, tf.float32)
        accuracy = tf.reduce_mean(
            tf.cast(tf.equal(y_pred, cast_class_labels), dtype=tf.float32)
        )
        metrics_dict[f"{class_name.lower()}_train_accuracy"](accuracy)


def write_results_description_in_txt(txt_file):
    txt_file.write("Results Description\n")
    txt_file.write("Number of examples: %d\n" % TOTAL_EXAMPLES)
    txt_file.write("Train/Test limit: %d\n" % TRAIN_TEST_LIMIT)
    txt_file.write("Epochs: %d\n" % EPOCHS)
    txt_file.write("Complexity of the Dataset: %d\n" % COMPLEXITY_DATASET)
    txt_file.write("Batch size: %d\n" % BATCH_SIZE)
    txt_file.write("Learning rate: %.5f\n" % LEARNING_RATE)
    txt_file.write("Number of classes: %d\n" % len(CLASSES))
    txt_file.write("\n")


""" SCRIPT """

# scheduled_parameters = defaultdict(lambda: {}) -> This can be used to schedule hyperparameters
template = "Epoch {}"
for metrics_label in metrics_dict.keys():
    template += ", %s: {:.5f}" % metrics_label
if csv_path is not None:
    txt_file = open(txt_path, "w+")
    write_results_description_in_txt(txt_file)
    csv_file = open(csv_path, "w+")
    headers = ",".join(["Epoch"] + list(metrics_dict.keys()))
    csv_template = ",".join(["{}" for _ in range(len(metrics_dict) + 1)])
    csv_file.write(headers + "\n")

for epoch in range(EPOCHS):
    for metrics in metrics_dict.values():
        metrics.reset_states()
    for _ in range(len(train_generator)):
        batch_elements = train_generator.next()
        features_param, labels_list = __get_elements_from_batch(batch_elements)
        train_step(features_param, *labels_list)
    for _ in range(len(test_generator)):
        batch_elements = test_generator.next()
        features_param, labels_list = __get_elements_from_batch(batch_elements)
        test_step(features_param, *labels_list)

    metrics_results = [metrics.result() for metrics in metrics_dict.values()]
    print(template.format(epoch, *metrics_results))
    if csv_path is not None:
        csv_file.write(csv_template.format(epoch, *metrics_results) + "\n")
        csv_file.flush()
if csv_path is not None:
    csv_file.close()
    txt_file.close()



Found 12000 validated image filenames.
Found 3000 validated image filenames.


/home/camib/dissertation-workspace/venv/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


ValueError: in user code:

    File "/tmp/ipykernel_22896/2743121795.py", line 154, in axioms  *
        axioms = [
    File "/home/camib/dissertation-workspace/venv/lib/python3.12/site-packages/ltn/core.py", line 189, in __call__  *
        expr = super().__call__(inputs, *args, **kwargs)
    File "/home/camib/dissertation-workspace/venv/lib/python3.12/site-packages/ltn/core.py", line 159, in __call__  *
        t_outputs = self.model(as_tensors(flat_inputs), *args, **kwargs)
    File "/home/camib/dissertation-workspace/venv/lib/python3.12/site-packages/keras/src/utils/traceback_utils.py", line 122, in error_handler  **
        raise e.with_traceback(filtered_tb) from None
    File "/usr/lib/python3.12/contextlib.py", line 138, in __enter__
        except StopIteration:

    ValueError: '__softmax_tf_model' is not a valid root scope name. A root scope name has to match the following pattern: ^[A-Za-z0-9.][A-Za-z0-9_.\\/>-]*$


Teste 2

In [ ]:
""" DATASET """
data_path = Path(DATA_ROOT, "C" + str(COMPLEXITY_DATASET))
labels = Path(data_path, "labels.csv")
images = Path(data_path, "images")

df = pd.read_csv(labels, index_col=0)
df["id"] = df["id"].apply(lambda x: f"{x}.jpg")
# CLASSES = df.columns.tolist()[1:]
CLASSES = [
    "Residential",
    "CountryHouse",
    "Suburban",
    "Store",
    "Billboard",
    "PowerPlant",
    "Chimney",
    "Pipe",
]

datagen = image.ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=0,
    width_shift_range=0,
    height_shift_range=0,
    shear_range=0,
    zoom_range=0,
    horizontal_flip=False,
    fill_mode="nearest",
)

train_generator = datagen.flow_from_dataframe(
    dataframe=df[:TRAIN_TEST_LIMIT],
    directory=str(images),
    x_col="id",
    y_col=CLASSES,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False,
)

test_generator = datagen.flow_from_dataframe(
    dataframe=df[TRAIN_TEST_LIMIT:TOTAL_EXAMPLES],
    directory=str(images),
    x_col="id",
    y_col=CLASSES,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False,
)


"""CNN MODEL"""


class CNNModel(keras.Model):

    def __init__(self, num_classes):
        super(CNNModel, self).__init__()
        self.conv1 = keras.layers.Conv2D(
            16, kernel_size=(3, 3), activation="relu", input_shape=(224, 224, 3)
        )
        self.conv2 = keras.layers.Conv2D(32, kernel_size=(3, 3), activation="relu")
        self.conv3 = keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu")
        self.maxpool = keras.layers.MaxPooling2D(pool_size=(2, 2))
        self.flatten = keras.layers.Flatten()
        self.dropout = keras.layers.Dropout(0.5)
        self.dense1 = keras.layers.Dense(128, activation="relu")
        self.dense2 = keras.layers.Dense(64, activation="relu")
        self.concept_layer = keras.layers.Dense(32, activation="relu")
        self.output_layer = keras.layers.Dense(num_classes)

    def call(self, inputs):
        x = self.conv1(inputs[0])
        x = self.maxpool(x)
        x = self.conv2(x)
        x = self.maxpool(x)
        x = self.conv3(x)
        x = self.maxpool(x)
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.dense1(x)
        x = self.dropout(x)
        x = self.dense2(x)
        y = self.concept_layer(x)
        x = self.output_layer(y)
        result = tf.concat([x, y], axis=1)
        return result


""" LTN MODEL """

# Predicates
logits_model = CNNModel(num_classes=4)
p = ltn.Predicate.FromLogits(
    logits_model, activation_function="sigmoid", with_class_indexing=True
)
# pf = ltn.Predicate.FromLogits(logits_model, activation_function="softmax")

# Constants
class_residential = ltn.Constant(CLASSES.index("Residential"), trainable=False)
class_country_house = ltn.Constant(CLASSES.index("CountryHouse"), trainable=False)
class_suburban = ltn.Constant(CLASSES.index("Suburban"), trainable=False)
class_store = ltn.Constant(CLASSES.index("Store"), trainable=False)
class_billboard = ltn.Constant(CLASSES.index("Billboard"), trainable=False)
class_power_plant = ltn.Constant(CLASSES.index("PowerPlant"), trainable=False)
class_chimney = ltn.Constant(CLASSES.index("Chimney"), trainable=False)
class_pipe = ltn.Constant(CLASSES.index("Pipe"), trainable=False)


# Operators
Not = ltn.Wrapper_Connective(ltn.fuzzy_ops.Not_Std())
And = ltn.Wrapper_Connective(ltn.fuzzy_ops.And_Prod())
Or = ltn.Wrapper_Connective(ltn.fuzzy_ops.Or_ProbSum())
Implies = ltn.Wrapper_Connective(ltn.fuzzy_ops.Implies_Reichenbach())
Equiv = ltn.Wrapper_Connective(
    ltn.fuzzy_ops.Equiv(ltn.fuzzy_ops.And_Prod(), ltn.fuzzy_ops.Implies_Reichenbach())
)
Forall = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMeanError(), semantics="forall")
Exists = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMean(), semantics="exists")

formula_aggregator = ltn.Wrapper_Formula_Aggregator(
    ltn.fuzzy_ops.Aggreg_pMeanError(p=2)
)


@tf.function()
def axioms(features_param, *args):
    x = ltn.Variable("x", features_param)
    x_residential = ltn.Variable(
        "x_residential", features_param[args[CLASSES.index("Residential")] == 1]
    )
    x_not_residential = ltn.Variable(
        "x_not_residential", features_param[args[CLASSES.index("Residential")] == 0]
    )
    x_store = ltn.Variable("x_store", features_param[args[CLASSES.index("Store")] == 1])
    x_not_store = ltn.Variable(
        "x_not_store", features_param[args[CLASSES.index("Store")] == 0]
    )
    x_power_plant = ltn.Variable(
        "x_power_plant", features_param[args[CLASSES.index("PowerPlant")] == 1]
    )
    x_not_power_plant = ltn.Variable(
        "x_not_power_plant",
        features_param[args[CLASSES.index("PowerPlant")] == 0],
    )
    axioms = [
        # Grounded Predicates
        Forall(x_residential, p([x_residential, class_residential])),
        Forall(x_not_residential, Not(p([x_not_residential, class_residential]))),
        Forall(x_store, p([x_store, class_store])),
        Forall(x_not_store, Not(p([x_not_store, class_store]))),
        Forall(x_power_plant, p([x_power_plant, class_power_plant])),
        Forall(
            x_not_power_plant,
            Not(p([x_not_power_plant, class_power_plant])),
        ),
        # Axioms
        # Residential = Country House or Misc Residential or Suburban
        Forall(
            x,
            Equiv(
                p([x, class_residential]),
                Or(p([x, class_country_house]), p([x, class_suburban])),
            ),
        ),
        # Store = Billboard
        Forall(
            x,
            Equiv(p([x, class_store]), p([x, class_billboard])),
        ),
        # Power Plant = Chimney and Pipe
        Forall(
            x,
            Equiv(
                p([x, class_power_plant]),
                And(p([x, class_chimney]), p([x, class_pipe])),
            ),
        ),
    ]
    sat_level = formula_aggregator(axioms).tensor
    return sat_level


def __get_elements_from_batch(batch_elements):
    features_calculated = batch_elements[0]
    inverted_labels_list = batch_elements[1]
    labels_list = np.transpose(inverted_labels_list)
    return features_calculated, labels_list


for _ in range(len(train_generator)):
    batch_elements = train_generator.next()
    features, labels_list = __get_elements_from_batch(batch_elements)
    print("Initial sat level %.5f" % axioms(batch_elements[0], *labels_list))
    break


""" TRAINING FUNCTIONS """

metrics_dict = {
    "train_sat_kb": keras.metrics.Mean(name="train_sat_kb"),
    "test_sat_kb": keras.metrics.Mean(name="test_sat_kb"),
}


class_train_accuracy_metric = {
    f"{class_name.lower()}_train_accuracy": keras.metrics.Mean(
        name=f"{class_name.lower()}_train_accuracy"
    )
    for class_name in CLASSES
}
metrics_dict.update(class_train_accuracy_metric)


optimizer = keras.optimizers.legacy.Adam(learning_rate=LEARNING_RATE)


@tf.function()
def train_step(*args):
    features_param = args[0]
    labels_list = args[1:]
    with tf.GradientTape() as tape:
        sat = axioms(features_param, *labels_list)
        loss = 1.0 - sat
    gradients = tape.gradient(loss, p.trainable_variables)
    optimizer.apply_gradients(zip(gradients, p.trainable_variables))
    tf.print("Sat Function Result: ", sat)
    metrics_dict["train_sat_kb"](sat)
    predictions = logits_model([features_param])
    calculate_accuracy_for_each_class(predictions, labels_list)


@tf.function()
def test_step(*args):
    features_param = args[0]
    labels_list = args[1:]
    sat_kb = axioms(features_param, *labels_list)
    metrics_dict["test_sat_kb"](sat_kb)


""" AUXILIARY FUNCTIONS """


def calculate_accuracy_for_each_class(predictions, labels_param):
    for i, class_name in enumerate(CLASSES):
        class_labels = labels_param[i]
        y_pred = predictions[:, i]
        y_pred = tf.math.sigmoid(y_pred)
        y_pred = tf.cast(y_pred > 0.5, tf.float32)
        cast_class_labels = tf.cast(class_labels, tf.float32)
        accuracy = tf.reduce_mean(
            tf.cast(tf.equal(y_pred, cast_class_labels), dtype=tf.float32)
        )
        metrics_dict[f"{class_name.lower()}_train_accuracy"](accuracy)


def write_results_description_in_txt(txt_file):
    txt_file.write("Results Description\n")
    txt_file.write("Number of examples: %d\n" % TOTAL_EXAMPLES)
    txt_file.write("Train/Test limit: %d\n" % TRAIN_TEST_LIMIT)
    txt_file.write("Epochs: %d\n" % EPOCHS)
    txt_file.write("Complexity of the Dataset: %d\n" % COMPLEXITY_DATASET)
    txt_file.write("Batch size: %d\n" % BATCH_SIZE)
    txt_file.write("Learning rate: %.5f\n" % LEARNING_RATE)
    txt_file.write("Number of classes: %d\n" % len(CLASSES))
    txt_file.write("\n")


""" SCRIPT """

# scheduled_parameters = defaultdict(lambda: {}) -> This can be used to schedule hyperparameters
template = "Epoch {}"
for metrics_label in metrics_dict.keys():
    template += ", %s: {:.5f}" % metrics_label
if csv_path is not None:
    txt_file = open(txt_path, "w+")
    write_results_description_in_txt(txt_file)
    csv_file = open(csv_path, "w+")
    headers = ",".join(["Epoch"] + list(metrics_dict.keys()))
    csv_template = ",".join(["{}" for _ in range(len(metrics_dict) + 1)])
    csv_file.write(headers + "\n")

for epoch in range(EPOCHS):
    for metrics in metrics_dict.values():
        metrics.reset_states()
    for _ in range(len(train_generator)):
        batch_elements = train_generator.next()
        features_param, labels_list = __get_elements_from_batch(batch_elements)
        train_step(features_param, *labels_list)
    for _ in range(len(test_generator)):
        batch_elements = test_generator.next()
        features_param, labels_list = __get_elements_from_batch(batch_elements)
        test_step(features_param, *labels_list)

    metrics_results = [metrics.result() for metrics in metrics_dict.values()]
    print(template.format(epoch, *metrics_results))
    if csv_path is not None:
        csv_file.write(csv_template.format(epoch, *metrics_results) + "\n")
        csv_file.flush()
if csv_path is not None:
    csv_file.close()
    txt_file.close()


Found 750 validated image filenames.
Found 250 validated image filenames.
Initial sat level 0.51627
Sat Function Result:  0.516235
Sat Function Result:  0.517699897
Sat Function Result:  0.519908071
Sat Function Result:  0.521244287
Sat Function Result:  0.522861838
Sat Function Result:  0.526200056
Sat Function Result:  0.528105438
Sat Function Result:  0.532566786
Sat Function Result:  0.530959368
Sat Function Result:  0.524809241
Sat Function Result:  0.544982433
Sat Function Result:  0.535998583
Epoch 0, train_sat_kb: 0.52680, test_sat_kb: 0.54277, residential_train_accuracy: 0.35388, countryhouse_train_accuracy: 0.86260, suburban_train_accuracy: 0.22254, store_train_accuracy: 0.62845, billboard_train_accuracy: 0.73970, powerplant_train_accuracy: 0.85349, chimney_train_accuracy: 0.26806, pipe_train_accuracy: 0.37381
Sat Function Result:  0.544493198
Sat Function Result:  0.551213503
Sat Function Result:  0.553655
Sat Function Result:  0.567760408
Sat Function Result:  0.558557272
S

Repetição do teste 1 com sat metric

In [ ]:
""" DATASET """
data_path = Path(DATA_ROOT, "C" + str(COMPLEXITY_DATASET))
labels = Path(data_path, "labels.csv")
images = Path(data_path, "images")

df = pd.read_csv(labels, index_col=0)
df["id"] = df["id"].apply(lambda x: f"{x}.jpg")
# CLASSES = df.columns.tolist()[1:]
CLASSES = [
    "Cafe",
    "Hotel",
    "Store",
    "MiscCommercial",
    "Awning",
    "Billboard",
    "VendingMachine",
    "Statue",
    "Table",
    "WallSign",
]

datagen = image.ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=0,
    width_shift_range=0,
    height_shift_range=0,
    shear_range=0,
    zoom_range=0,
    horizontal_flip=False,
    fill_mode="nearest",
)

train_generator = datagen.flow_from_dataframe(
    dataframe=df[:TRAIN_TEST_LIMIT],
    directory=str(images),
    x_col="id",
    y_col=CLASSES,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False,
)

test_generator = datagen.flow_from_dataframe(
    dataframe=df[TRAIN_TEST_LIMIT:TOTAL_EXAMPLES],
    directory=str(images),
    x_col="id",
    y_col=CLASSES,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False,
)


"""CNN MODEL"""


class CNNModel(keras.Model):

    def __init__(self, num_classes):
        super(CNNModel, self).__init__()
        self.conv1 = keras.layers.Conv2D(
            16, kernel_size=(3, 3), activation="relu", input_shape=(224, 224, 3)
        )
        self.conv2 = keras.layers.Conv2D(32, kernel_size=(3, 3), activation="relu")
        self.conv3 = keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu")
        self.maxpool = keras.layers.MaxPooling2D(pool_size=(2, 2))
        self.flatten = keras.layers.Flatten()
        self.dropout = keras.layers.Dropout(0.5)
        self.dense1 = keras.layers.Dense(128, activation="relu")
        self.dense2 = keras.layers.Dense(64, activation="relu")
        self.concept_layer = keras.layers.Dense(32, activation="relu")
        self.output_layer = keras.layers.Dense(num_classes)

    def call(self, inputs):
        x = self.conv1(inputs[0])
        x = self.maxpool(x)
        x = self.conv2(x)
        x = self.maxpool(x)
        x = self.conv3(x)
        x = self.maxpool(x)
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.dense1(x)
        x = self.dropout(x)
        x = self.dense2(x)
        y = self.concept_layer(x)
        x = self.output_layer(y)
        result = tf.concat([x, y], axis=1)
        return result


""" LTN MODEL """

# Predicates
logits_model = CNNModel(num_classes=4)
p = ltn.Predicate.FromLogits(
    logits_model, activation_function="sigmoid", with_class_indexing=True
)
# pf = ltn.Predicate.FromLogits(logits_model, activation_function="softmax")

# Constants
class_cafe = ltn.Constant(CLASSES.index("Cafe"), trainable=False)
class_hotel = ltn.Constant(CLASSES.index("Hotel"), trainable=False)
class_store = ltn.Constant(CLASSES.index("Store"), trainable=False)
class_misc_commercial = ltn.Constant(CLASSES.index("MiscCommercial"), trainable=False)
class_awning = ltn.Constant(CLASSES.index("Awning"), trainable=False)
class_billboard = ltn.Constant(CLASSES.index("Billboard"), trainable=False)
class_vending_machine = ltn.Constant(CLASSES.index("VendingMachine"), trainable=False)
class_statue = ltn.Constant(CLASSES.index("Statue"), trainable=False)
class_table = ltn.Constant(CLASSES.index("Table"), trainable=False)
class_wall_sign = ltn.Constant(CLASSES.index("WallSign"), trainable=False)


# Operators
Not = ltn.Wrapper_Connective(ltn.fuzzy_ops.Not_Std())
And = ltn.Wrapper_Connective(ltn.fuzzy_ops.And_Prod())
Or = ltn.Wrapper_Connective(ltn.fuzzy_ops.Or_ProbSum())
Implies = ltn.Wrapper_Connective(ltn.fuzzy_ops.Implies_Reichenbach())
Equiv = ltn.Wrapper_Connective(
    ltn.fuzzy_ops.Equiv(ltn.fuzzy_ops.And_Prod(), ltn.fuzzy_ops.Implies_Reichenbach())
)
Forall = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMeanError(), semantics="forall")
Exists = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMean(), semantics="exists")

formula_aggregator = ltn.Wrapper_Formula_Aggregator(
    ltn.fuzzy_ops.Aggreg_pMeanError(p=2)
)


@tf.function()
def axioms(features_param, *args):
    x = ltn.Variable("x", features_param)
    x_hotel = ltn.Variable("x_hotel", features_param[args[CLASSES.index("Hotel")] == 1])
    x_not_hotel = ltn.Variable(
        "x_not_hotel", features_param[args[CLASSES.index("Hotel")] == 0]
    )
    x_cafe = ltn.Variable("x_cafe", features_param[args[CLASSES.index("Cafe")] == 1])
    x_not_cafe = ltn.Variable(
        "x_not_cafe", features_param[args[CLASSES.index("Cafe")] == 0]
    )
    x_store = ltn.Variable("x_store", features_param[args[CLASSES.index("Store")] == 1])
    x_not_store = ltn.Variable(
        "x_not_store", features_param[args[CLASSES.index("Store")] == 0]
    )
    x_misc_commercial = ltn.Variable(
        "x_misc_commercial", features_param[args[CLASSES.index("MiscCommercial")] == 1]
    )
    x_not_misc_commercial = ltn.Variable(
        "x_not_misc_commercial",
        features_param[args[CLASSES.index("MiscCommercial")] == 0],
    )
    axioms = [
        # Grounded Predicates
        Forall(x_hotel, p([x_hotel, class_hotel])),
        Forall(x_not_hotel, Not(p([x_not_hotel, class_hotel]))),
        Forall(x_cafe, p([x_cafe, class_cafe])),
        Forall(x_not_cafe, Not(p([x_not_cafe, class_cafe]))),
        Forall(x_store, p([x_store, class_store])),
        Forall(x_not_store, Not(p([x_not_store, class_store]))),
        Forall(x_misc_commercial, p([x_misc_commercial, class_misc_commercial])),
        Forall(
            x_not_misc_commercial,
            Not(p([x_not_misc_commercial, class_misc_commercial])),
        ),
        # Axioms
        # Cafe = Statue or Vending Machine
        Forall(
            x,
            Equiv(
                p([x, class_cafe]),
                Or(p([x, class_statue]), p([x, class_vending_machine])),
            ),
        ),
        # Hotel = Wall Sign
        Forall(
            x,
            Equiv(p([x, class_hotel]), p([x, class_wall_sign])),
        ),
        # MiscCommercial = Awning and Table
        Forall(
            x,
            Equiv(
                p([x, class_misc_commercial]),
                And(p([x, class_awning]), p([x, class_table])),
            ),
        ),
        # Store = Billboard
        Forall(
            x,
            Equiv(p([x, class_store]), p([x, class_billboard])),
        ),
    ]
    sat_level = formula_aggregator(axioms).tensor
    return sat_level


@tf.function()
def sat_phi_class(features_param, labels_list, class_chosen):
    x_class = ltn.Variable("x_class", features_param[labels_list == 1])
    x_not_class = ltn.Variable("x_class", features_param[labels_list == 1])
    axioms = [
        Forall(x_class, p([x_class, class_chosen])),
        Forall(x_not_class, Not(p([x_not_class, class_chosen]))),
    ]
    sat_level = formula_aggregator(axioms).tensor
    return sat_level


def __get_elements_from_batch(batch_elements):
    features_calculated = batch_elements[0]
    inverted_labels_list = batch_elements[1]
    labels_list = np.transpose(inverted_labels_list)
    return features_calculated, labels_list


for _ in range(len(train_generator)):
    batch_elements = train_generator.next()
    features, labels_list = __get_elements_from_batch(batch_elements)
    print("Initial sat level %.5f" % axioms(batch_elements[0], *labels_list))
    break


""" TRAINING FUNCTIONS """

metrics_dict = {
    "train_sat_kb": keras.metrics.Mean(name="train_sat_kb"),
    "test_sat_kb": keras.metrics.Mean(name="test_sat_kb"),
}


class_train_accuracy_metric = {
    f"{class_name.lower()}_train_accuracy": keras.metrics.Mean(
        name=f"{class_name.lower()}_train_accuracy"
    )
    for class_name in CLASSES
}
metrics_dict.update(class_train_accuracy_metric)

class_train_sat_metric = {
    f"{class_name.lower()}_sat_accuracy": keras.metrics.Mean(
        name=f"{class_name.lower()}_sat_accuracy"
    )
    for class_name in CLASSES
}
metrics_dict.update(class_train_sat_metric)


optimizer = keras.optimizers.legacy.Adam(learning_rate=LEARNING_RATE)


@tf.function()
def train_step(*args):
    features_param = args[0]
    labels_list = args[1:]
    with tf.GradientTape() as tape:
        sat = axioms(features_param, *labels_list)
        loss = 1.0 - sat
    gradients = tape.gradient(loss, p.trainable_variables)
    optimizer.apply_gradients(zip(gradients, p.trainable_variables))
    tf.print("Sat Function Result: ", sat)
    metrics_dict["train_sat_kb"](sat)
    predictions = logits_model([features_param])
    calculate_accuracy_for_each_class(predictions, labels_list)
    calculate_sat_kb_for_each_class(features_param, labels_list)


@tf.function()
def test_step(*args):
    features_param = args[0]
    labels_list = args[1:]
    sat_kb = axioms(features_param, *labels_list)
    metrics_dict["test_sat_kb"](sat_kb)


""" AUXILIARY FUNCTIONS """

CLASSES = [
    "Cafe",
    "Hotel",
    "Store",
    "MiscCommercial",
    "Awning",
    "Billboard",
    "VendingMachine",
    "Statue",
    "Table",
    "WallSign",
]


def calculate_sat_kb_for_each_class(features_param, labels_param):
    metrics_dict["cafe_sat_accuracy"](
        sat_phi_class(features_param, labels_param[CLASSES.index("Cafe")], class_cafe)
    )
    metrics_dict["hotel_sat_accuracy"](
        sat_phi_class(features_param, labels_param[CLASSES.index("Hotel")], class_hotel)
    )
    metrics_dict["store_sat_accuracy"](
        sat_phi_class(features_param, labels_param[CLASSES.index("Store")], class_store)
    )
    metrics_dict["misccommercial_sat_accuracy"](
        sat_phi_class(
            features_param,
            labels_param[CLASSES.index("MiscCommercial")],
            class_misc_commercial,
        )
    )
    metrics_dict["awning_sat_accuracy"](
        sat_phi_class(
            features_param, labels_param[CLASSES.index("Awning")], class_awning
        )
    )
    metrics_dict["billboard_sat_accuracy"](
        sat_phi_class(
            features_param, labels_param[CLASSES.index("Billboard")], class_billboard
        )
    )
    metrics_dict["vendingmachine_sat_accuracy"](
        sat_phi_class(
            features_param,
            labels_param[CLASSES.index("VendingMachine")],
            class_vending_machine,
        )
    )
    metrics_dict["statue_sat_accuracy"](
        sat_phi_class(
            features_param, labels_param[CLASSES.index("Statue")], class_statue
        )
    )
    metrics_dict["table_sat_accuracy"](
        sat_phi_class(features_param, labels_param[CLASSES.index("Table")], class_table)
    )
    metrics_dict["wallsign_sat_accuracy"](
        sat_phi_class(
            features_param, labels_param[CLASSES.index("WallSign")], class_wall_sign
        )
    )


def calculate_accuracy_for_each_class(predictions, labels_param):
    for i, class_name in enumerate(CLASSES):
        class_labels = labels_param[i]
        y_pred = predictions[:, i]
        y_pred = tf.math.sigmoid(y_pred)
        y_pred = tf.cast(y_pred > 0.5, tf.float32)
        cast_class_labels = tf.cast(class_labels, tf.float32)
        accuracy = tf.reduce_mean(
            tf.cast(tf.equal(y_pred, cast_class_labels), dtype=tf.float32)
        )
        metrics_dict[f"{class_name.lower()}_train_accuracy"](accuracy)


def write_results_description_in_txt(txt_file):
    txt_file.write("Results Description\n")
    txt_file.write("Number of examples: %d\n" % TOTAL_EXAMPLES)
    txt_file.write("Train/Test limit: %d\n" % TRAIN_TEST_LIMIT)
    txt_file.write("Epochs: %d\n" % EPOCHS)
    txt_file.write("Complexity of the Dataset: %d\n" % COMPLEXITY_DATASET)
    txt_file.write("Batch size: %d\n" % BATCH_SIZE)
    txt_file.write("Learning rate: %.5f\n" % LEARNING_RATE)
    txt_file.write("Number of classes: %d\n" % len(CLASSES))
    txt_file.write("\n")


""" SCRIPT """

# scheduled_parameters = defaultdict(lambda: {}) -> This can be used to schedule hyperparameters
template = "Epoch {}"
for metrics_label in metrics_dict.keys():
    template += ", %s: {:.5f}" % metrics_label
if csv_path is not None:
    txt_file = open(txt_path, "w+")
    write_results_description_in_txt(txt_file)
    csv_file = open(csv_path, "w+")
    headers = ",".join(["Epoch"] + list(metrics_dict.keys()))
    csv_template = ",".join(["{}" for _ in range(len(metrics_dict) + 1)])
    csv_file.write(headers + "\n")

for epoch in range(EPOCHS):
    for metrics in metrics_dict.values():
        metrics.reset_states()
    for _ in range(len(train_generator)):
        batch_elements = train_generator.next()
        features_param, labels_list = __get_elements_from_batch(batch_elements)
        train_step(features_param, *labels_list)
    for _ in range(len(test_generator)):
        batch_elements = test_generator.next()
        features_param, labels_list = __get_elements_from_batch(batch_elements)
        test_step(features_param, *labels_list)

    metrics_results = [metrics.result() for metrics in metrics_dict.values()]
    print(template.format(epoch, *metrics_results))
    if csv_path is not None:
        csv_file.write(csv_template.format(epoch, *metrics_results) + "\n")
        csv_file.flush()
if csv_path is not None:
    csv_file.close()
    txt_file.close()


Found 750 validated image filenames.
Found 250 validated image filenames.
Initial sat level 0.51668
Sat Function Result:  0.516866744
Sat Function Result:  0.513144255
Sat Function Result:  0.515630186
Sat Function Result:  0.52005434
Sat Function Result:  0.519758344
Sat Function Result:  0.522044778
Sat Function Result:  0.522282958
Sat Function Result:  0.524325907
Sat Function Result:  0.528995395
Sat Function Result:  0.536400676
Sat Function Result:  0.538812578
Sat Function Result:  0.549955368
Epoch 0, train_sat_kb: 0.52569, test_sat_kb: 0.54202, cafe_train_accuracy: 0.25532, hotel_train_accuracy: 0.20613, store_train_accuracy: 0.66231, misccommercial_train_accuracy: 0.41576, awning_train_accuracy: 0.45414, billboard_train_accuracy: 0.69390, vendingmachine_train_accuracy: 0.77978, statue_train_accuracy: 0.26478, table_train_accuracy: 0.37591, wallsign_train_accuracy: 0.20613, cafe_sat_accuracy: 0.48496, hotel_sat_accuracy: 0.47563, store_sat_accuracy: 0.49527, misccommercial_sa

# Testes dia 2.6

In [ ]:

""" DATASET """
data_path = Path(DATA_ROOT, "C" + str(COMPLEXITY_DATASET))
labels = Path(data_path, "labels.csv")
images = Path(data_path, "images")

df = pd.read_csv(labels, index_col=0)
df["id"] = df["id"].apply(lambda x: f"{x}.jpg")
# CLASSES = df.columns.tolist()[1:]

CLASSES = [
    "Cafe",
    "Store",
    "MiscCommercial",
    "Awning",
    "Billboard",
    "VendingMachine",
    "Statue",
    "Table",
]

datagen = image.ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=0,
    width_shift_range=0,
    height_shift_range=0,
    shear_range=0,
    zoom_range=0,
    horizontal_flip=False,
    fill_mode="nearest",
)

train_generator = datagen.flow_from_dataframe(
    dataframe=df[:TRAIN_TEST_LIMIT],
    directory=str(images),
    x_col="id",
    y_col=CLASSES,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False,
)

test_generator = datagen.flow_from_dataframe(
    dataframe=df[TRAIN_TEST_LIMIT:TOTAL_EXAMPLES],
    directory=str(images),
    x_col="id",
    y_col=CLASSES,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False,
)


"""CNN MODEL"""


class CNNModel(keras.Model):

    def __init__(self, num_classes):
        super(CNNModel, self).__init__()
        self.conv1 = keras.layers.Conv2D(
            16, kernel_size=(3, 3), activation="relu", input_shape=(224, 224, 3)
        )
        self.conv2 = keras.layers.Conv2D(32, kernel_size=(3, 3), activation="relu")
        self.conv3 = keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu")
        self.maxpool = keras.layers.MaxPooling2D(pool_size=(2, 2))
        self.flatten = keras.layers.Flatten()
        self.dropout = keras.layers.Dropout(0.5)
        self.dense1 = keras.layers.Dense(128, activation="relu")
        self.dense2 = keras.layers.Dense(64, activation="relu")
        self.concept_layer = keras.layers.Dense(32, activation="relu")
        self.output_layer = keras.layers.Dense(num_classes)

    def call(self, inputs):
        x = self.conv1(inputs[0])
        x = self.maxpool(x)
        x = self.conv2(x)
        x = self.maxpool(x)
        x = self.conv3(x)
        x = self.maxpool(x)
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.dense1(x)
        x = self.dropout(x)
        x = self.dense2(x)
        y = self.concept_layer(x)
        x = self.output_layer(y)
        result = tf.concat([x, y], axis=1)
        return result


""" LTN MODEL """

# Predicates
logits_model = CNNModel(num_classes=4)
p = ltn.Predicate.FromLogits(
    logits_model, activation_function="sigmoid", with_class_indexing=True
)
# pf = ltn.Predicate.FromLogits(logits_model, activation_function="softmax")

# Constants
class_cafe = ltn.Constant(CLASSES.index("Cafe"), trainable=False)
class_store = ltn.Constant(CLASSES.index("Store"), trainable=False)
class_misc_commercial = ltn.Constant(CLASSES.index("MiscCommercial"), trainable=False)
class_awning = ltn.Constant(CLASSES.index("Awning"), trainable=False)
class_billboard = ltn.Constant(CLASSES.index("Billboard"), trainable=False)
class_vending_machine = ltn.Constant(CLASSES.index("VendingMachine"), trainable=False)
class_statue = ltn.Constant(CLASSES.index("Statue"), trainable=False)
class_table = ltn.Constant(CLASSES.index("Table"), trainable=False)


# Operators
Not = ltn.Wrapper_Connective(ltn.fuzzy_ops.Not_Std())
And = ltn.Wrapper_Connective(ltn.fuzzy_ops.And_Prod())
Or = ltn.Wrapper_Connective(ltn.fuzzy_ops.Or_ProbSum())
Implies = ltn.Wrapper_Connective(ltn.fuzzy_ops.Implies_Reichenbach())
Equiv = ltn.Wrapper_Connective(
    ltn.fuzzy_ops.Equiv(ltn.fuzzy_ops.And_Prod(), ltn.fuzzy_ops.Implies_Reichenbach())
)
Forall = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMeanError(), semantics="forall")
Exists = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMean(), semantics="exists")

formula_aggregator = ltn.Wrapper_Formula_Aggregator(
    ltn.fuzzy_ops.Aggreg_pMeanError(p=2)
)


@tf.function()
def axioms(features_param, *args):
    x = ltn.Variable("x", features_param)
    x_cafe = ltn.Variable("x_cafe", features_param[args[CLASSES.index("Cafe")] == 1])
    x_not_cafe = ltn.Variable(
        "x_not_cafe", features_param[args[CLASSES.index("Cafe")] == 0]
    )
    x_store = ltn.Variable("x_store", features_param[args[CLASSES.index("Store")] == 1])
    x_not_store = ltn.Variable(
        "x_not_store", features_param[args[CLASSES.index("Store")] == 0]
    )
    x_misc_commercial = ltn.Variable(
        "x_misc_commercial", features_param[args[CLASSES.index("MiscCommercial")] == 1]
    )
    x_not_misc_commercial = ltn.Variable(
        "x_not_misc_commercial",
        features_param[args[CLASSES.index("MiscCommercial")] == 0],
    )
    axioms = [
        # Grounded Predicates
        Forall(x_cafe, p([x_cafe, class_cafe])),
        Forall(x_not_cafe, Not(p([x_not_cafe, class_cafe]))),
        Forall(x_store, p([x_store, class_store])),
        Forall(x_not_store, Not(p([x_not_store, class_store]))),
        Forall(x_misc_commercial, p([x_misc_commercial, class_misc_commercial])),
        Forall(
            x_not_misc_commercial,
            Not(p([x_not_misc_commercial, class_misc_commercial])),
        ),
        # Axioms
        # Cafe = Statue or Vending Machine
        Forall(
            x,
            Equiv(
                p([x, class_cafe]),
                Or(p([x, class_statue]), p([x, class_vending_machine])),
            ),
        ),
        # MiscCommercial = Awning and Table
        Forall(
            x,
            Equiv(
                p([x, class_misc_commercial]),
                And(p([x, class_awning]), p([x, class_table])),
            ),
        ),
        # Store = Billboard
        Forall(
            x,
            Equiv(p([x, class_store]), p([x, class_billboard])),
        ),
    ]
    sat_level = formula_aggregator(axioms).tensor
    return sat_level


def __get_elements_from_batch(batch_elements):
    features_calculated = batch_elements[0]
    inverted_labels_list = batch_elements[1]
    labels_list = np.transpose(inverted_labels_list)
    return features_calculated, labels_list


for _ in range(len(train_generator)):
    batch_elements = train_generator.next()
    features, labels_list = __get_elements_from_batch(batch_elements)
    print("Initial sat level %.5f" % axioms(batch_elements[0], *labels_list))
    break


""" TRAINING FUNCTIONS """

metrics_dict = {
    "train_sat_kb": keras.metrics.Mean(name="train_sat_kb"),
    "test_sat_kb": keras.metrics.Mean(name="test_sat_kb"),
}


class_train_accuracy_metric = {
    f"{class_name.lower()}_train_accuracy": keras.metrics.Mean(
        name=f"{class_name.lower()}_train_accuracy"
    )
    for class_name in CLASSES
}
metrics_dict.update(class_train_accuracy_metric)

class_test_accuracy_metric = {
    f"{class_name.lower()}_test_accuracy": keras.metrics.Mean(
        name=f"{class_name.lower()}_test_accuracy"
    )
    for class_name in CLASSES
}
metrics_dict.update(class_test_accuracy_metric)


optimizer = keras.optimizers.legacy.Adam(learning_rate=LEARNING_RATE)


@tf.function()
def train_step(*args):
    features_param = args[0]
    labels_list = args[1:]
    with tf.GradientTape() as tape:
        sat = axioms(features_param, *labels_list)
        loss = 1.0 - sat
    gradients = tape.gradient(loss, p.trainable_variables)
    optimizer.apply_gradients(zip(gradients, p.trainable_variables))
    tf.print("Sat Function Result: ", sat)
    metrics_dict["train_sat_kb"](sat)
    predictions = logits_model([features_param])
    calculate_accuracy_for_each_class("train", predictions, labels_list)


@tf.function()
def test_step(*args):
    features_param = args[0]
    labels_list = args[1:]
    sat_kb = axioms(features_param, *labels_list)
    metrics_dict["test_sat_kb"](sat_kb)
    predictions = logits_model([features_param])
    calculate_accuracy_for_each_class("test", predictions, labels_list)


""" AUXILIARY FUNCTIONS """


def calculate_accuracy_for_each_class(accuracy_type: str, predictions, labels_param):
    for i, class_name in enumerate(CLASSES):
        class_labels = labels_param[i]
        y_pred = predictions[:, i]
        y_pred = tf.math.sigmoid(y_pred)
        y_pred = tf.cast(y_pred > 0.5, tf.float32)
        cast_class_labels = tf.cast(class_labels, tf.float32)
        accuracy = tf.reduce_mean(
            tf.cast(tf.equal(y_pred, cast_class_labels), dtype=tf.float32)
        )
        metrics_dict[f"{class_name.lower()}_{accuracy_type}_accuracy"](accuracy)


def write_results_description_in_txt(txt_file):
    txt_file.write("Results Description\n")
    txt_file.write("Number of examples: %d\n" % TOTAL_EXAMPLES)
    txt_file.write("Train/Test limit: %d\n" % TRAIN_TEST_LIMIT)
    txt_file.write("Epochs: %d\n" % EPOCHS)
    txt_file.write("Complexity of the Dataset: %d\n" % COMPLEXITY_DATASET)
    txt_file.write("Batch size: %d\n" % BATCH_SIZE)
    txt_file.write("Learning rate: %.5f\n" % LEARNING_RATE)
    txt_file.write("Number of classes: %d\n" % len(CLASSES))
    txt_file.write("\n")


""" SCRIPT """

# scheduled_parameters = defaultdict(lambda: {}) -> This can be used to schedule hyperparameters
template = "Epoch {}"
for metrics_label in metrics_dict.keys():
    template += ", %s: {:.5f}" % metrics_label
if csv_path is not None:
    txt_file = open(txt_path, "w+")
    write_results_description_in_txt(txt_file)
    csv_file = open(csv_path, "w+")
    headers = ",".join(["Epoch"] + list(metrics_dict.keys()))
    csv_template = ",".join(["{}" for _ in range(len(metrics_dict) + 1)])
    csv_file.write(headers + "\n")

for epoch in range(EPOCHS):
    for metrics in metrics_dict.values():
        metrics.reset_states()
    for _ in range(len(train_generator)):
        batch_elements = train_generator.next()
        features_param, labels_list = __get_elements_from_batch(batch_elements)
        train_step(features_param, *labels_list)
    for _ in range(len(test_generator)):
        batch_elements = test_generator.next()
        features_param, labels_list = __get_elements_from_batch(batch_elements)
        test_step(features_param, *labels_list)

    metrics_results = [metrics.result() for metrics in metrics_dict.values()]
    print(template.format(epoch, *metrics_results))
    if csv_path is not None:
        csv_file.write(csv_template.format(epoch, *metrics_results) + "\n")
        csv_file.flush()
if csv_path is not None:
    csv_file.close()
    txt_file.close()


Found 12000 validated image filenames.
Found 3000 validated image filenames.
Initial sat level 0.51659
Sat Function Result:  0.516741633
Sat Function Result:  0.498308897
Sat Function Result:  0.515639305
Sat Function Result:  0.52494514
Sat Function Result:  0.524309754
Sat Function Result:  0.530881405
Sat Function Result:  0.531981289
Sat Function Result:  0.532692134
Sat Function Result:  0.549686372
Sat Function Result:  0.558936656
Sat Function Result:  0.561136603
Sat Function Result:  0.566790938
Sat Function Result:  0.574310184
Sat Function Result:  0.581258893
Sat Function Result:  0.573911786
Sat Function Result:  0.593373895
Sat Function Result:  0.58497262
Sat Function Result:  0.609953165
Sat Function Result:  0.604068398
Sat Function Result:  0.58038789
Sat Function Result:  0.604342699
Sat Function Result:  0.610316753
Sat Function Result:  0.599585772
Sat Function Result:  0.598720193
Sat Function Result:  0.596016884
Sat Function Result:  0.622704625
Sat Function Res

KeyboardInterrupt: 

In [ ]:
""" DATASET """
data_path = Path(DATA_ROOT, "C" + str(COMPLEXITY_DATASET))
labels = Path(data_path, "labels.csv")
images = Path(data_path, "images")

df = pd.read_csv(labels, index_col=0)
df["id"] = df["id"].apply(lambda x: f"{x}.jpg")
# CLASSES = df.columns.tolist()[1:]

CLASSES = [
    "Cafe",
    "Store",
    "MiscCommercial",
    "Awning",
    "Billboard",
    "VendingMachine",
    "Statue",
    "Table",
]

datagen = image.ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=0,
    width_shift_range=0,
    height_shift_range=0,
    shear_range=0,
    zoom_range=0,
    horizontal_flip=False,
    fill_mode="nearest",
)

train_generator = datagen.flow_from_dataframe(
    dataframe=df[:TRAIN_TEST_LIMIT],
    directory=str(images),
    x_col="id",
    y_col=CLASSES,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False,
)

test_generator = datagen.flow_from_dataframe(
    dataframe=df[TRAIN_TEST_LIMIT:TOTAL_EXAMPLES],
    directory=str(images),
    x_col="id",
    y_col=CLASSES,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False,
)


"""CNN MODEL"""


class CNNModel(keras.Model):

    def __init__(self, num_classes):
        super(CNNModel, self).__init__()
        self.conv1 = keras.layers.Conv2D(
            16, kernel_size=(3, 3), activation="relu", input_shape=(224, 224, 3)
        )
        self.conv2 = keras.layers.Conv2D(32, kernel_size=(3, 3), activation="relu")
        self.conv3 = keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu")
        self.maxpool = keras.layers.MaxPooling2D(pool_size=(2, 2))
        self.flatten = keras.layers.Flatten()
        self.dropout = keras.layers.Dropout(0.5)
        self.dense1 = keras.layers.Dense(128, activation="relu")
        self.dense2 = keras.layers.Dense(64, activation="relu")
        self.concept_layer = keras.layers.Dense(32, activation="relu")
        self.output_layer = keras.layers.Dense(num_classes)

    def call(self, inputs):
        x = self.conv1(inputs[0])
        x = self.maxpool(x)
        x = self.conv2(x)
        x = self.maxpool(x)
        x = self.conv3(x)
        x = self.maxpool(x)
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.dense1(x)
        x = self.dropout(x)
        x = self.dense2(x)
        y = self.concept_layer(x)
        x = self.output_layer(y)
        result = tf.concat([x, y], axis=1)
        return result


""" LTN MODEL """

# Predicates
logits_model = CNNModel(num_classes=4)
p = ltn.Predicate.FromLogits(
    logits_model, activation_function="sigmoid", with_class_indexing=True
)
# pf = ltn.Predicate.FromLogits(logits_model, activation_function="softmax")

# Constants
class_cafe = ltn.Constant(CLASSES.index("Cafe"), trainable=False)
class_store = ltn.Constant(CLASSES.index("Store"), trainable=False)
class_misc_commercial = ltn.Constant(CLASSES.index("MiscCommercial"), trainable=False)
class_awning = ltn.Constant(CLASSES.index("Awning"), trainable=False)
class_billboard = ltn.Constant(CLASSES.index("Billboard"), trainable=False)
class_vending_machine = ltn.Constant(CLASSES.index("VendingMachine"), trainable=False)
class_statue = ltn.Constant(CLASSES.index("Statue"), trainable=False)
class_table = ltn.Constant(CLASSES.index("Table"), trainable=False)


# Operators
Not = ltn.Wrapper_Connective(ltn.fuzzy_ops.Not_Std())
And = ltn.Wrapper_Connective(ltn.fuzzy_ops.And_Prod())
Or = ltn.Wrapper_Connective(ltn.fuzzy_ops.Or_ProbSum())
Implies = ltn.Wrapper_Connective(ltn.fuzzy_ops.Implies_Reichenbach())
Equiv = ltn.Wrapper_Connective(
    ltn.fuzzy_ops.Equiv(ltn.fuzzy_ops.And_Prod(), ltn.fuzzy_ops.Implies_Reichenbach())
)
Forall = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMeanError(), semantics="forall")
Exists = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMean(), semantics="exists")

formula_aggregator = ltn.Wrapper_Formula_Aggregator(
    ltn.fuzzy_ops.Aggreg_pMeanError(p=2)
)


@tf.function()
def axioms(features_param, *args):
    x = ltn.Variable("x", features_param)
    x_cafe = ltn.Variable("x_cafe", features_param[args[CLASSES.index("Cafe")] == 1])
    x_not_cafe = ltn.Variable(
        "x_not_cafe", features_param[args[CLASSES.index("Cafe")] == 0]
    )
    x_store = ltn.Variable("x_store", features_param[args[CLASSES.index("Store")] == 1])
    x_not_store = ltn.Variable(
        "x_not_store", features_param[args[CLASSES.index("Store")] == 0]
    )
    x_misc_commercial = ltn.Variable(
        "x_misc_commercial", features_param[args[CLASSES.index("MiscCommercial")] == 1]
    )
    x_not_misc_commercial = ltn.Variable(
        "x_not_misc_commercial",
        features_param[args[CLASSES.index("MiscCommercial")] == 0],
    )
    axioms = [
        # Grounded Predicates
        Forall(x_cafe, p([x_cafe, class_cafe])),
        Forall(x_not_cafe, Not(p([x_not_cafe, class_cafe]))),
        Forall(x_store, p([x_store, class_store])),
        Forall(x_not_store, Not(p([x_not_store, class_store]))),
        Forall(x_misc_commercial, p([x_misc_commercial, class_misc_commercial])),
        Forall(
            x_not_misc_commercial,
            Not(p([x_not_misc_commercial, class_misc_commercial])),
        ),
        # Axioms
        # Cafe = Statue or Vending Machine
        Forall(
            x,
            Equiv(
                p([x, class_cafe]),
                Or(p([x, class_statue]), p([x, class_vending_machine])),
            ),
        ),
        # MiscCommercial = Awning and Table
        Forall(
            x,
            Equiv(
                p([x, class_misc_commercial]),
                And(p([x, class_awning]), p([x, class_table])),
            ),
        ),
        # Store = Billboard
        Forall(
            x,
            Equiv(p([x, class_store]), p([x, class_billboard])),
        ),
    ]
    sat_level = formula_aggregator(axioms).tensor
    return sat_level


def __get_elements_from_batch(batch_elements):
    features_calculated = batch_elements[0]
    inverted_labels_list = batch_elements[1]
    labels_list = np.transpose(inverted_labels_list)
    return features_calculated, labels_list


for _ in range(len(train_generator)):
    batch_elements = train_generator.next()
    features, labels_list = __get_elements_from_batch(batch_elements)
    print("Initial sat level %.5f" % axioms(batch_elements[0], *labels_list))
    break


""" TRAINING FUNCTIONS """

metrics_dict = {
    "train_sat_kb": keras.metrics.Mean(name="train_sat_kb"),
    "test_sat_kb": keras.metrics.Mean(name="test_sat_kb"),
}

classes_for_confusion_matrix = ["Awning", "Table", "Statue", "VendingMachine"]

metricts_conf_matrix = {
    f"{class_name}_train_conf_matrix": tf.Variable(
        initial_value=np.zeros((2, 2), dtype=np.int32)
    )
    for class_name in classes_for_confusion_matrix
}
metricts_conf_matrix_test = {
    f"{class_name}_test_conf_matrix": tf.Variable(
        initial_value=np.zeros((2, 2), dtype=np.int32)
    )
    for class_name in classes_for_confusion_matrix
}
metricts_conf_matrix.update(metricts_conf_matrix_test)


class_train_accuracy_metric = {
    f"{class_name.lower()}_train_accuracy": keras.metrics.Mean(
        name=f"{class_name.lower()}_train_accuracy"
    )
    for class_name in CLASSES
}
metrics_dict.update(class_train_accuracy_metric)

class_test_accuracy_metric = {
    f"{class_name.lower()}_test_accuracy": keras.metrics.Mean(
        name=f"{class_name.lower()}_test_accuracy"
    )
    for class_name in CLASSES
}
metrics_dict.update(class_test_accuracy_metric)


optimizer = keras.optimizers.legacy.Adam(learning_rate=LEARNING_RATE)


@tf.function()
def train_step(*args):
    features_param = args[0]
    labels_list = args[1:]
    with tf.GradientTape() as tape:
        sat = axioms(features_param, *labels_list)
        loss = 1.0 - sat
    gradients = tape.gradient(loss, p.trainable_variables)
    optimizer.apply_gradients(zip(gradients, p.trainable_variables))
    tf.print("Sat Function Result: ", sat)
    metrics_dict["train_sat_kb"](sat)
    predictions = logits_model([features_param])
    calculate_accuracy_for_each_class("train", predictions, labels_list)
    update_confusion_matrix("train", predictions, labels_list)


@tf.function()
def test_step(*args):
    features_param = args[0]
    labels_list = args[1:]
    sat_kb = axioms(features_param, *labels_list)
    metrics_dict["test_sat_kb"](sat_kb)
    predictions = logits_model([features_param])
    calculate_accuracy_for_each_class("test", predictions, labels_list)
    update_confusion_matrix("test", predictions, labels_list)


""" AUXILIARY FUNCTIONS """


def calculate_accuracy_for_each_class(phase: str, predictions, labels_param):
    for i, class_name in enumerate(CLASSES):
        class_labels = labels_param[i]
        y_pred = predictions[:, i]
        y_pred = tf.math.sigmoid(y_pred)
        y_pred = tf.cast(y_pred > 0.5, tf.float32)
        cast_class_labels = tf.cast(class_labels, tf.float32)
        accuracy = tf.reduce_mean(
            tf.cast(tf.equal(y_pred, cast_class_labels), dtype=tf.float32)
        )
        metrics_dict[f"{class_name.lower()}_{phase}_accuracy"](accuracy)


def update_confusion_matrix(phase: str, predictions, labels_param):
    for class_name in classes_for_confusion_matrix:
        class_labels = tf.cast(labels_param[CLASSES.index(class_name)], tf.int32)
        y_pred = predictions[:, CLASSES.index(class_name)]
        y_preds = tf.cast(y_pred > 0.5, tf.int32)
        confusion_matrix = tf.math.confusion_matrix(
            class_labels, y_preds, num_classes=2
        )
        metricts_conf_matrix[f"{class_name}_{phase}_conf_matrix"].assign_add(
            confusion_matrix
        )


def write_results_description_in_txt(txt_file):
    txt_file.write("Results Description\n")
    txt_file.write("Number of examples: %d\n" % TOTAL_EXAMPLES)
    txt_file.write("Train/Test limit: %d\n" % TRAIN_TEST_LIMIT)
    txt_file.write("Epochs: %d\n" % EPOCHS)
    txt_file.write("Complexity of the Dataset: %d\n" % COMPLEXITY_DATASET)
    txt_file.write("Batch size: %d\n" % BATCH_SIZE)
    txt_file.write("Learning rate: %.5f\n" % LEARNING_RATE)
    txt_file.write("Number of classes: %d\n" % len(CLASSES))
    txt_file.write("\n")


def write_conf_matrix(confusion_matrix_path, epoch):
    with open(confusion_matrix_path, "a") as conf_matrix_file:
        conf_matrix_file.write(f"Epoch {epoch}\n")
        for class_name in classes_for_confusion_matrix:
            
            conf_matrix_file.write(
                f"{class_name} Train Confusion Matrix:\n{metricts_conf_matrix[f'{class_name}_train_conf_matrix'].numpy()}\n"
            )
            conf_matrix_file.write(
                f"{class_name} Test Confusion Matrix:\n{metricts_conf_matrix[f'{class_name}_test_conf_matrix'].numpy()}\n"
            )
        conf_matrix_file.write("\n")

""" SCRIPT """

# scheduled_parameters = defaultdict(lambda: {}) -> This can be used to schedule hyperparameters
template = "Epoch {}"
for metrics_label in metrics_dict.keys():
    template += ", %s: {:.5f}" % metrics_label
if csv_path is not None:
    txt_file = open(txt_path, "w+")
    write_results_description_in_txt(txt_file)
    csv_file = open(csv_path, "w+")

    headers = ",".join(["Epoch"] + list(metrics_dict.keys()))
    csv_template = ",".join(["{}" for _ in range(len(metrics_dict) + 1)])
    csv_file.write(headers + "\n")

for epoch in range(EPOCHS):
    for metrics in metrics_dict.values():
        metrics.reset_states()
    for metric in metricts_conf_matrix.values():
        metric.assign(np.zeros((2, 2), dtype=np.int32))
    for _ in range(len(train_generator)):
        batch_elements = train_generator.next()
        features_param, labels_list = __get_elements_from_batch(batch_elements)
        train_step(features_param, *labels_list)
    for _ in range(len(test_generator)):
        batch_elements = test_generator.next()
        features_param, labels_list = __get_elements_from_batch(batch_elements)
        test_step(features_param, *labels_list)

    metrics_results = [metrics.result() for metrics in metrics_dict.values()]
    print(template.format(epoch, *metrics_results))
    write_conf_matrix(confusion_matrix_path, epoch)
    if csv_path is not None:
        csv_file.write(csv_template.format(epoch, *metrics_results) + "\n")
        csv_file.flush()
if csv_path is not None:
    csv_file.close()
    txt_file.close()


A saída de streaming foi truncada nas últimas 5000 linhas.
Sat Function Result:  0.746585369
Sat Function Result:  0.744154334
Sat Function Result:  0.743588924
Sat Function Result:  0.740993142
Sat Function Result:  0.72547996
Sat Function Result:  0.739408612
Sat Function Result:  0.743307352
Sat Function Result:  0.749307156
Sat Function Result:  0.75412637
Sat Function Result:  0.747719049
Sat Function Result:  0.742061317
Sat Function Result:  0.740634143
Sat Function Result:  0.746648192
Sat Function Result:  0.745886803
Sat Function Result:  0.742405772
Sat Function Result:  0.749577105
Sat Function Result:  0.742579103
Sat Function Result:  0.742474198
Sat Function Result:  0.746928453
Sat Function Result:  0.740464926
Sat Function Result:  0.745992
Sat Function Result:  0.752767742
Sat Function Result:  0.747064
Sat Function Result:  0.74617815
Sat Function Result:  0.74725008
Sat Function Result:  0.749032319
Sat Function Result:  0.741245627
Sat Function Result:  0.743185163

Testes dia 4.5

# Testes dia 3.6

In [ ]:

""" DATASET """
data_path = Path(DATA_ROOT, "C" + str(COMPLEXITY_DATASET))
labels = Path(data_path, "labels.csv")
images = Path(data_path, "images")

df = pd.read_csv(labels, index_col=0)
df["id"] = df["id"].apply(lambda x: f"{x}.jpg")
# CLASSES = df.columns.tolist()[1:]

CLASSES = [
    "Restaurant",
    "Sign",
    "Truck",
    "Car",
]

datagen = image.ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=0,
    width_shift_range=0,
    height_shift_range=0,
    shear_range=0,
    zoom_range=0,
    horizontal_flip=False,
    fill_mode="nearest",
)

train_generator = datagen.flow_from_dataframe(
    dataframe=df[:TRAIN_TEST_LIMIT],
    directory=str(images),
    x_col="id",
    y_col=CLASSES,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False,
)

test_generator = datagen.flow_from_dataframe(
    dataframe=df[TRAIN_TEST_LIMIT:TOTAL_EXAMPLES],
    directory=str(images),
    x_col="id",
    y_col=CLASSES,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False,
)


"""CNN MODEL"""


class CNNModel(keras.Model):

    def __init__(self, num_classes):
        super(CNNModel, self).__init__()
        self.conv1 = keras.layers.Conv2D(
            16, kernel_size=(3, 3), activation="relu", input_shape=(224, 224, 3)
        )
        self.conv2 = keras.layers.Conv2D(32, kernel_size=(3, 3), activation="relu")
        self.conv3 = keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu")
        self.maxpool = keras.layers.MaxPooling2D(pool_size=(2, 2))
        self.flatten = keras.layers.Flatten()
        self.dropout = keras.layers.Dropout(0.5)
        self.dense1 = keras.layers.Dense(128, activation="relu")
        self.dense2 = keras.layers.Dense(64, activation="relu")
        self.concept_layer = keras.layers.Dense(32, activation="relu")
        self.output_layer = keras.layers.Dense(num_classes)

    def call(self, inputs):
        x = self.conv1(inputs[0])
        x = self.maxpool(x)
        x = self.conv2(x)
        x = self.maxpool(x)
        x = self.conv3(x)
        x = self.maxpool(x)
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.dense1(x)
        x = self.dropout(x)
        x = self.dense2(x)
        y = self.concept_layer(x)
        x = self.output_layer(y)
        result = tf.concat([x, y], axis=1)
        return result


""" LTN MODEL """

# Predicates
logits_model = CNNModel(num_classes=1)
p = ltn.Predicate.FromLogits(
    logits_model, activation_function="sigmoid", with_class_indexing=True
)
# pf = ltn.Predicate.FromLogits(logits_model, activation_function="softmax")

# Constants
class_restaurant = ltn.Constant(CLASSES.index("Restaurant"), trainable=False)
class_sign = ltn.Constant(CLASSES.index("Sign"), trainable=False)
class_car = ltn.Constant(CLASSES.index("Car"), trainable=False)
class_truck = ltn.Constant(CLASSES.index("Truck"), trainable=False)


# Operators
Not = ltn.Wrapper_Connective(ltn.fuzzy_ops.Not_Std())
And = ltn.Wrapper_Connective(ltn.fuzzy_ops.And_Prod())
Or = ltn.Wrapper_Connective(ltn.fuzzy_ops.Or_ProbSum())
Implies = ltn.Wrapper_Connective(ltn.fuzzy_ops.Implies_Reichenbach())
Equiv = ltn.Wrapper_Connective(
    ltn.fuzzy_ops.Equiv(ltn.fuzzy_ops.And_Prod(), ltn.fuzzy_ops.Implies_Reichenbach())
)
Forall = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMeanError(), semantics="forall")
Exists = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMean(), semantics="exists")

formula_aggregator = ltn.Wrapper_Formula_Aggregator(
    ltn.fuzzy_ops.Aggreg_pMeanError(p=2)
)


@tf.function()
def axioms(features_param, *args):
    x = ltn.Variable("x", features_param)
    x_restaurant = ltn.Variable(
        "x_restaurant", features_param[args[CLASSES.index("Restaurant")] == 1]
    )
    x_not_restaurant = ltn.Variable(
        "x_not_restaurant", features_param[args[CLASSES.index("Restaurant")] == 0]
    )
    axioms = [
        # Grounded Predicates
        Forall(x_restaurant, p([x_restaurant, class_restaurant])),
        Forall(x_not_restaurant, Not(p([x_not_restaurant, class_restaurant]))),
        # Axioms
        # Not (Car and Truck)
        Forall(x, Not(And(p([x, class_car]), p([x, class_truck])))),
        # Restaurant = (Car or Truck) and Sign
        Forall(
            x,
            Equiv(
                p([x, class_restaurant]),
                And(Or(p([x, class_car]), p([x, class_truck])), p([x, class_sign])),
            ),
        ),
    ]
    sat_level = formula_aggregator(axioms).tensor
    return sat_level


def __get_elements_from_batch(batch_elements):
    features_calculated = batch_elements[0]
    inverted_labels_list = batch_elements[1]
    labels_list = np.transpose(inverted_labels_list)
    return features_calculated, labels_list


for _ in range(len(train_generator)):
    batch_elements = train_generator.next()
    features, labels_list = __get_elements_from_batch(batch_elements)
    print("Initial sat level %.5f" % axioms(batch_elements[0], *labels_list))
    break


""" TRAINING FUNCTIONS """

metrics_dict = {
    "train_sat_kb": keras.metrics.Mean(name="train_sat_kb"),
    "test_sat_kb": keras.metrics.Mean(name="test_sat_kb"),
}

classes_for_confusion_matrix = [
    "Sign",
    "Truck",
    "Car",
]

metricts_conf_matrix = {
    f"{class_name}_train_conf_matrix": tf.Variable(
        initial_value=np.zeros((2, 2), dtype=np.int32)
    )
    for class_name in classes_for_confusion_matrix
}
metricts_conf_matrix_test = {
    f"{class_name}_test_conf_matrix": tf.Variable(
        initial_value=np.zeros((2, 2), dtype=np.int32)
    )
    for class_name in classes_for_confusion_matrix
}
metricts_conf_matrix.update(metricts_conf_matrix_test)


class_train_accuracy_metric = {
    f"{class_name.lower()}_train_accuracy": keras.metrics.Mean(
        name=f"{class_name.lower()}_train_accuracy"
    )
    for class_name in CLASSES
}
metrics_dict.update(class_train_accuracy_metric)

class_test_accuracy_metric = {
    f"{class_name.lower()}_test_accuracy": keras.metrics.Mean(
        name=f"{class_name.lower()}_test_accuracy"
    )
    for class_name in CLASSES
}
metrics_dict.update(class_test_accuracy_metric)


optimizer = keras.optimizers.legacy.Adam(learning_rate=LEARNING_RATE)


@tf.function()
def train_step(*args):
    features_param = args[0]
    labels_list = args[1:]
    with tf.GradientTape() as tape:
        sat = axioms(features_param, *labels_list)
        loss = 1.0 - sat
    gradients = tape.gradient(loss, p.trainable_variables)
    optimizer.apply_gradients(zip(gradients, p.trainable_variables))
    tf.print("Sat Function Result: ", sat)
    metrics_dict["train_sat_kb"](sat)
    predictions = logits_model([features_param])
    calculate_accuracy_for_each_class("train", predictions, labels_list)
    update_confusion_matrix("train", predictions, labels_list)


@tf.function()
def test_step(*args):
    features_param = args[0]
    labels_list = args[1:]
    sat_kb = axioms(features_param, *labels_list)
    metrics_dict["test_sat_kb"](sat_kb)
    predictions = logits_model([features_param])
    calculate_accuracy_for_each_class("test", predictions, labels_list)
    update_confusion_matrix("test", predictions, labels_list)


""" AUXILIARY FUNCTIONS """


def calculate_accuracy_for_each_class(phase: str, predictions, labels_param):
    for i, class_name in enumerate(CLASSES):
        class_labels = labels_param[i]
        y_pred = predictions[:, i]
        y_pred = tf.math.sigmoid(y_pred)
        y_pred = tf.cast(y_pred > 0.5, tf.float32)
        cast_class_labels = tf.cast(class_labels, tf.float32)
        accuracy = tf.reduce_mean(
            tf.cast(tf.equal(y_pred, cast_class_labels), dtype=tf.float32)
        )
        metrics_dict[f"{class_name.lower()}_{phase}_accuracy"](accuracy)


def update_confusion_matrix(phase: str, predictions, labels_param):
    for class_name in classes_for_confusion_matrix:
        class_labels = tf.cast(labels_param[CLASSES.index(class_name)], tf.int32)
        y_pred = predictions[:, CLASSES.index(class_name)]
        y_preds = tf.cast(y_pred > 0.5, tf.int32)
        confusion_matrix = tf.math.confusion_matrix(
            class_labels, y_preds, num_classes=2
        )
        metricts_conf_matrix[f"{class_name}_{phase}_conf_matrix"].assign_add(
            confusion_matrix
        )


def write_results_description_in_txt(txt_file):
    txt_file.write("Results Description\n")
    txt_file.write("Number of examples: %d\n" % TOTAL_EXAMPLES)
    txt_file.write("Train/Test limit: %d\n" % TRAIN_TEST_LIMIT)
    txt_file.write("Epochs: %d\n" % EPOCHS)
    txt_file.write("Complexity of the Dataset: %d\n" % COMPLEXITY_DATASET)
    txt_file.write("Batch size: %d\n" % BATCH_SIZE)
    txt_file.write("Learning rate: %.5f\n" % LEARNING_RATE)
    txt_file.write("Number of classes: %d\n" % len(CLASSES))
    txt_file.write("\n")


def write_conf_matrix(confusion_matrix_path, epoch):
    with open(confusion_matrix_path, "a") as conf_matrix_file:
        conf_matrix_file.write(f"Epoch {epoch}\n")
        for class_name in classes_for_confusion_matrix:
            conf_matrix_file.write(
                f"{class_name} Train Confusion Matrix:\n{metricts_conf_matrix[f'{class_name}_train_conf_matrix'].numpy()}\n"
            )
            conf_matrix_file.write(
                f"{class_name} Test Confusion Matrix:\n{metricts_conf_matrix[f'{class_name}_test_conf_matrix'].numpy()}\n"
            )
        conf_matrix_file.write("\n")


""" SCRIPT """

# scheduled_parameters = defaultdict(lambda: {}) -> This can be used to schedule hyperparameters
template = "Epoch {}"
for metrics_label in metrics_dict.keys():
    template += ", %s: {:.5f}" % metrics_label
if csv_path is not None:
    txt_file = open(txt_path, "w+")
    write_results_description_in_txt(txt_file)
    csv_file = open(csv_path, "w+")

    headers = ",".join(["Epoch"] + list(metrics_dict.keys()))
    csv_template = ",".join(["{}" for _ in range(len(metrics_dict) + 1)])
    csv_file.write(headers + "\n")

for epoch in range(EPOCHS):
    for metrics in metrics_dict.values():
        metrics.reset_states()
    for metric in metricts_conf_matrix.values():
        metric.assign(np.zeros((2, 2), dtype=np.int32))
    for _ in range(len(train_generator)):
        batch_elements = train_generator.next()
        features_param, labels_list = __get_elements_from_batch(batch_elements)
        train_step(features_param, *labels_list)
    for _ in range(len(test_generator)):
        batch_elements = test_generator.next()
        features_param, labels_list = __get_elements_from_batch(batch_elements)
        test_step(features_param, *labels_list)

    metrics_results = [metrics.result() for metrics in metrics_dict.values()]
    print(template.format(epoch, *metrics_results))
    write_conf_matrix(confusion_matrix_path, epoch)
    if csv_path is not None:
        csv_file.write(csv_template.format(epoch, *metrics_results) + "\n")
        csv_file.flush()
if csv_path is not None:
    csv_file.close()
    txt_file.close()


A saída de streaming foi truncada nas últimas 5000 linhas.
Sat Function Result:  0.754649162
Sat Function Result:  0.758478045
Sat Function Result:  0.765916765
Sat Function Result:  0.76592207
Sat Function Result:  0.756601155
Sat Function Result:  0.760293782
Sat Function Result:  0.758423746
Sat Function Result:  0.751088202
Sat Function Result:  0.76225543
Sat Function Result:  0.76225
Sat Function Result:  0.75657481
Sat Function Result:  0.758435249
Sat Function Result:  0.762017
Sat Function Result:  0.760349393
Sat Function Result:  0.752834558
Sat Function Result:  0.762215436
Sat Function Result:  0.76607877
Sat Function Result:  0.76224494
Sat Function Result:  0.764197707
Sat Function Result:  0.758370638
Sat Function Result:  0.758457124
Sat Function Result:  0.749183238
Sat Function Result:  0.768099189
Sat Function Result:  0.762184441
Sat Function Result:  0.766155303
Sat Function Result:  0.762239277
Sat Function Result:  0.760325313
Sat Function Result:  0.760324538
S

KeyboardInterrupt: 

# Testes dia 4.6

In [ ]:

""" DATASET """
data_path = Path(DATA_ROOT, "C" + str(COMPLEXITY_DATASET))
labels = Path(data_path, "labels.csv")
images = Path(data_path, "images")

df = pd.read_csv(labels, index_col=0)
df["id"] = df["id"].apply(lambda x: f"{x}.jpg")
# CLASSES = df.columns.tolist()[1:]

CLASSES = [
    "Cafe",
    "MiscCommercial",
    "Awning",
    "Table",
    "VendingMachine",
    "Statue",
]

datagen = image.ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=0,
    width_shift_range=0,
    height_shift_range=0,
    shear_range=0,
    zoom_range=0,
    horizontal_flip=False,
    fill_mode="nearest",
)

train_generator = datagen.flow_from_dataframe(
    dataframe=df[:TRAIN_TEST_LIMIT],
    directory=str(images),
    x_col="id",
    y_col=CLASSES,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False,
)

test_generator = datagen.flow_from_dataframe(
    dataframe=df[TRAIN_TEST_LIMIT:TOTAL_EXAMPLES],
    directory=str(images),
    x_col="id",
    y_col=CLASSES,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False,
)


"""CNN MODEL"""


class CNNModel(keras.Model):

    def __init__(self, num_classes):
        super(CNNModel, self).__init__()
        self.conv1 = keras.layers.Conv2D(
            16, kernel_size=(3, 3), activation="relu", input_shape=(224, 224, 3)
        )
        self.conv2 = keras.layers.Conv2D(32, kernel_size=(3, 3), activation="relu")
        self.conv3 = keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu")
        self.maxpool = keras.layers.MaxPooling2D(pool_size=(2, 2))
        self.flatten = keras.layers.Flatten()
        self.dropout = keras.layers.Dropout(0.5)
        self.dense1 = keras.layers.Dense(128, activation="relu")
        self.dense2 = keras.layers.Dense(64, activation="relu")
        self.concept_layer = keras.layers.Dense(32, activation="relu")
        self.output_layer = keras.layers.Dense(num_classes)

    def call(self, inputs):
        x = self.conv1(inputs[0])
        x = self.maxpool(x)
        x = self.conv2(x)
        x = self.maxpool(x)
        x = self.conv3(x)
        x = self.maxpool(x)
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.dense1(x)
        x = self.dropout(x)
        x = self.dense2(x)
        y = self.concept_layer(x)
        x = self.output_layer(y)
        result = tf.concat([x, y], axis=1)
        return result


""" LTN MODEL """

# Predicates
logits_model = CNNModel(num_classes=2)
p = ltn.Predicate.FromLogits(
    logits_model, activation_function="sigmoid", with_class_indexing=True
)
# pf = ltn.Predicate.FromLogits(logits_model, activation_function="softmax")

# Constants
class_cafe = ltn.Constant(CLASSES.index("Cafe"), trainable=False)
class_misc_commercial = ltn.Constant(CLASSES.index("MiscCommercial"), trainable=False)
class_awning = ltn.Constant(CLASSES.index("Awning"), trainable=False)
class_vending_machine = ltn.Constant(CLASSES.index("VendingMachine"), trainable=False)
class_statue = ltn.Constant(CLASSES.index("Statue"), trainable=False)
class_table = ltn.Constant(CLASSES.index("Table"), trainable=False)


# Operators
Not = ltn.Wrapper_Connective(ltn.fuzzy_ops.Not_Std())
And = ltn.Wrapper_Connective(ltn.fuzzy_ops.And_Prod())
Or = ltn.Wrapper_Connective(ltn.fuzzy_ops.Or_ProbSum())
Implies = ltn.Wrapper_Connective(ltn.fuzzy_ops.Implies_Reichenbach())
Equiv = ltn.Wrapper_Connective(
    ltn.fuzzy_ops.Equiv(ltn.fuzzy_ops.And_Prod(), ltn.fuzzy_ops.Implies_Reichenbach())
)
Forall = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMeanError(), semantics="forall")
Exists = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMean(), semantics="exists")

formula_aggregator = ltn.Wrapper_Formula_Aggregator(
    ltn.fuzzy_ops.Aggreg_pMeanError(p=2)
)


@tf.function()
def axioms(features_param, *args):
    x = ltn.Variable("x", features_param)
    x_cafe = ltn.Variable("x_cafe", features_param[args[CLASSES.index("Cafe")] == 1])
    x_not_cafe = ltn.Variable(
        "x_not_cafe", features_param[args[CLASSES.index("Cafe")] == 0]
    )
    x_misc_commercial = ltn.Variable(
        "x_misc_commercial", features_param[args[CLASSES.index("MiscCommercial")] == 1]
    )
    x_not_misc_commercial = ltn.Variable(
        "x_not_misc_commercial",
        features_param[args[CLASSES.index("MiscCommercial")] == 0],
    )
    axioms = [
        # Grounded Predicates
        Forall(x_cafe, p([x_cafe, class_cafe])),
        Forall(x_not_cafe, Not(p([x_not_cafe, class_cafe]))),
        Forall(x_misc_commercial, p([x_misc_commercial, class_misc_commercial])),
        Forall(
            x_not_misc_commercial,
            Not(p([x_not_misc_commercial, class_misc_commercial])),
        ),
        # Axioms
        # Cafe = Statue or Vending Machine
        Forall(
            x,
            Equiv(
                p([x, class_cafe]),
                Or(p([x, class_statue]), p([x, class_vending_machine])),
            ),
        ),
        # MiscCommercial = Awning and Table
        Forall(
            x,
            Equiv(
                p([x, class_misc_commercial]),
                And(p([x, class_awning]), p([x, class_table])),
            ),
        ),
    ]
    sat_level = formula_aggregator(axioms).tensor
    return sat_level


def __get_elements_from_batch(batch_elements):
    features_calculated = batch_elements[0]
    inverted_labels_list = batch_elements[1]
    labels_list = np.transpose(inverted_labels_list)
    return features_calculated, labels_list


for _ in range(len(train_generator)):
    batch_elements = train_generator.next()
    features, labels_list = __get_elements_from_batch(batch_elements)
    print("Initial sat level %.5f" % axioms(batch_elements[0], *labels_list))
    break


""" TRAINING FUNCTIONS """

metrics_dict = {
    "train_sat_kb": keras.metrics.Mean(name="train_sat_kb"),
    "test_sat_kb": keras.metrics.Mean(name="test_sat_kb"),
}

classes_for_confusion_matrix = [
    "Awning",
    "Table",
    "VendingMachine",
    "Statue",
]

metricts_conf_matrix = {
    f"{class_name}_train_conf_matrix": tf.Variable(
        initial_value=np.zeros((2, 2), dtype=np.int32)
    )
    for class_name in classes_for_confusion_matrix
}
metricts_conf_matrix_test = {
    f"{class_name}_test_conf_matrix": tf.Variable(
        initial_value=np.zeros((2, 2), dtype=np.int32)
    )
    for class_name in classes_for_confusion_matrix
}
metricts_conf_matrix.update(metricts_conf_matrix_test)


class_train_accuracy_metric = {
    f"{class_name.lower()}_train_accuracy": keras.metrics.Mean(
        name=f"{class_name.lower()}_train_accuracy"
    )
    for class_name in CLASSES
}
metrics_dict.update(class_train_accuracy_metric)

class_test_accuracy_metric = {
    f"{class_name.lower()}_test_accuracy": keras.metrics.Mean(
        name=f"{class_name.lower()}_test_accuracy"
    )
    for class_name in CLASSES
}
metrics_dict.update(class_test_accuracy_metric)


optimizer = keras.optimizers.legacy.Adam(learning_rate=LEARNING_RATE)


@tf.function()
def train_step(*args):
    features_param = args[0]
    labels_list = args[1:]
    with tf.GradientTape() as tape:
        sat = axioms(features_param, *labels_list)
        loss = 1.0 - sat
    gradients = tape.gradient(loss, p.trainable_variables)
    optimizer.apply_gradients(zip(gradients, p.trainable_variables))
    tf.print("Sat Function Result: ", sat)
    metrics_dict["train_sat_kb"](sat)
    predictions = logits_model([features_param])
    calculate_accuracy_for_each_class("train", predictions, labels_list)
    update_confusion_matrix("train", predictions, labels_list)


@tf.function()
def test_step(*args):
    features_param = args[0]
    labels_list = args[1:]
    sat_kb = axioms(features_param, *labels_list)
    metrics_dict["test_sat_kb"](sat_kb)
    predictions = logits_model([features_param])
    calculate_accuracy_for_each_class("test", predictions, labels_list)
    update_confusion_matrix("test", predictions, labels_list)


""" AUXILIARY FUNCTIONS """


def calculate_accuracy_for_each_class(phase: str, predictions, labels_param):
    for i, class_name in enumerate(CLASSES):
        class_labels = labels_param[i]
        y_pred = predictions[:, i]
        y_pred = tf.math.sigmoid(y_pred)
        y_pred = tf.cast(y_pred > 0.5, tf.float32)
        cast_class_labels = tf.cast(class_labels, tf.float32)
        accuracy = tf.reduce_mean(
            tf.cast(tf.equal(y_pred, cast_class_labels), dtype=tf.float32)
        )
        metrics_dict[f"{class_name.lower()}_{phase}_accuracy"](accuracy)


def update_confusion_matrix(phase: str, predictions, labels_param):
    for class_name in classes_for_confusion_matrix:
        class_labels = tf.cast(labels_param[CLASSES.index(class_name)], tf.int32)
        y_pred = predictions[:, CLASSES.index(class_name)]
        y_preds = tf.cast(y_pred > 0.5, tf.int32)
        confusion_matrix = tf.math.confusion_matrix(
            class_labels, y_preds, num_classes=2
        )
        metricts_conf_matrix[f"{class_name}_{phase}_conf_matrix"].assign_add(
            confusion_matrix
        )


def write_results_description_in_txt(txt_file):
    txt_file.write("Results Description\n")
    txt_file.write("Number of examples: %d\n" % TOTAL_EXAMPLES)
    txt_file.write("Train/Test limit: %d\n" % TRAIN_TEST_LIMIT)
    txt_file.write("Epochs: %d\n" % EPOCHS)
    txt_file.write("Complexity of the Dataset: %d\n" % COMPLEXITY_DATASET)
    txt_file.write("Batch size: %d\n" % BATCH_SIZE)
    txt_file.write("Learning rate: %.5f\n" % LEARNING_RATE)
    txt_file.write("Number of classes: %d\n" % len(CLASSES))
    txt_file.write("\n")


def write_conf_matrix(confusion_matrix_path, epoch):
    with open(confusion_matrix_path, "a") as conf_matrix_file:
        conf_matrix_file.write(f"Epoch {epoch}\n")
        for class_name in classes_for_confusion_matrix:
            conf_matrix_file.write(
                f"{class_name} Train Confusion Matrix:\n{metricts_conf_matrix[f'{class_name}_train_conf_matrix'].numpy()}\n"
            )
            conf_matrix_file.write(
                f"{class_name} Test Confusion Matrix:\n{metricts_conf_matrix[f'{class_name}_test_conf_matrix'].numpy()}\n"
            )
        conf_matrix_file.write("\n")


""" SCRIPT """

# scheduled_parameters = defaultdict(lambda: {}) -> This can be used to schedule hyperparameters
template = "Epoch {}"
for metrics_label in metrics_dict.keys():
    template += ", %s: {:.5f}" % metrics_label
if csv_path is not None:
    txt_file = open(txt_path, "w+")
    write_results_description_in_txt(txt_file)
    csv_file = open(csv_path, "w+")

    headers = ",".join(["Epoch"] + list(metrics_dict.keys()))
    csv_template = ",".join(["{}" for _ in range(len(metrics_dict) + 1)])
    csv_file.write(headers + "\n")

for epoch in range(EPOCHS):
    for metrics in metrics_dict.values():
        metrics.reset_states()
    for metric in metricts_conf_matrix.values():
        metric.assign(np.zeros((2, 2), dtype=np.int32))
    for _ in range(len(train_generator)):
        batch_elements = train_generator.next()
        features_param, labels_list = __get_elements_from_batch(batch_elements)
        train_step(features_param, *labels_list)
    for _ in range(len(test_generator)):
        batch_elements = test_generator.next()
        features_param, labels_list = __get_elements_from_batch(batch_elements)
        test_step(features_param, *labels_list)

    metrics_results = [metrics.result() for metrics in metrics_dict.values()]
    print(template.format(epoch, *metrics_results))
    write_conf_matrix(confusion_matrix_path, epoch)
    if csv_path is not None:
        csv_file.write(csv_template.format(epoch, *metrics_results) + "\n")
        csv_file.flush()
if csv_path is not None:
    csv_file.close()
    txt_file.close()


A saída de streaming foi truncada nas últimas 5000 linhas.
Sat Function Result:  0.740999
Sat Function Result:  0.734488368
Sat Function Result:  0.746145785
Sat Function Result:  0.742898881
Sat Function Result:  0.736666441
Sat Function Result:  0.741902113
Sat Function Result:  0.737846375
Sat Function Result:  0.744734168
Sat Function Result:  0.746789038
Sat Function Result:  0.745222569
Sat Function Result:  0.732009292
Sat Function Result:  0.729595304
Sat Function Result:  0.742179871
Sat Function Result:  0.746258259
Sat Function Result:  0.738901138
Sat Function Result:  0.732581
Sat Function Result:  0.735400796
Sat Function Result:  0.743075371
Sat Function Result:  0.736920357
Sat Function Result:  0.733341098
Sat Function Result:  0.746746302
Sat Function Result:  0.747100472
Sat Function Result:  0.728499055
Sat Function Result:  0.748167872
Sat Function Result:  0.728252769
Sat Function Result:  0.747633576
Sat Function Result:  0.727044821
Sat Function Result:  0.72421

In [ ]:


""" DATASET """
data_path = Path(DATA_ROOT, "C" + str(COMPLEXITY_DATASET))
labels = Path(data_path, "labels.csv")
images = Path(data_path, "images")

df = pd.read_csv(labels, index_col=0)
df["id"] = df["id"].apply(lambda x: f"{x}.jpg")
# CLASSES = df.columns.tolist()[1:]

CLASSES = [
    "Restaurant",
    "Sign",
    "Truck",
    "Car",
]

datagen = image.ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=0,
    width_shift_range=0,
    height_shift_range=0,
    shear_range=0,
    zoom_range=0,
    horizontal_flip=False,
    fill_mode="nearest",
)

train_generator = datagen.flow_from_dataframe(
    dataframe=df[:TRAIN_TEST_LIMIT],
    directory=str(images),
    x_col="id",
    y_col=CLASSES,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False,
)

test_generator = datagen.flow_from_dataframe(
    dataframe=df[TRAIN_TEST_LIMIT:TOTAL_EXAMPLES],
    directory=str(images),
    x_col="id",
    y_col=CLASSES,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False,
)


"""CNN MODEL"""


class CNNModel(keras.Model):

    def __init__(self, num_classes):
        super(CNNModel, self).__init__()
        self.conv1 = keras.layers.Conv2D(
            16, kernel_size=(3, 3), activation="relu", input_shape=(224, 224, 3)
        )
        self.conv2 = keras.layers.Conv2D(32, kernel_size=(3, 3), activation="relu")
        self.conv3 = keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu")
        self.maxpool = keras.layers.MaxPooling2D(pool_size=(2, 2))
        self.flatten = keras.layers.Flatten()
        self.dropout = keras.layers.Dropout(0.5)
        self.dense1 = keras.layers.Dense(128, activation="relu")
        self.dense2 = keras.layers.Dense(64, activation="relu")
        self.concept_layer = keras.layers.Dense(32, activation="relu")
        self.output_layer = keras.layers.Dense(num_classes)

    def call(self, inputs):
        x = self.conv1(inputs[0])
        x = self.maxpool(x)
        x = self.conv2(x)
        x = self.maxpool(x)
        x = self.conv3(x)
        x = self.maxpool(x)
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.dense1(x)
        x = self.dropout(x)
        x = self.dense2(x)
        y = self.concept_layer(x)
        x = self.output_layer(y)
        result = tf.concat([x, y], axis=1)
        return result


""" LTN MODEL """

# Predicates
logits_model = CNNModel(num_classes=1)
p = ltn.Predicate.FromLogits(
    logits_model, activation_function="sigmoid", with_class_indexing=True
)
# pf = ltn.Predicate.FromLogits(logits_model, activation_function="softmax")

# Constants
class_restaurant = ltn.Constant(CLASSES.index("Restaurant"), trainable=False)
class_sign = ltn.Constant(CLASSES.index("Sign"), trainable=False)
class_car = ltn.Constant(CLASSES.index("Car"), trainable=False)
class_truck = ltn.Constant(CLASSES.index("Truck"), trainable=False)


# Operators
Not = ltn.Wrapper_Connective(ltn.fuzzy_ops.Not_Std())
And = ltn.Wrapper_Connective(ltn.fuzzy_ops.And_Prod())
Or = ltn.Wrapper_Connective(ltn.fuzzy_ops.Or_ProbSum())
Implies = ltn.Wrapper_Connective(ltn.fuzzy_ops.Implies_Reichenbach())
Equiv = ltn.Wrapper_Connective(
    ltn.fuzzy_ops.Equiv(ltn.fuzzy_ops.And_Prod(), ltn.fuzzy_ops.Implies_Reichenbach())
)
Forall = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMeanError(), semantics="forall")
Exists = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMean(), semantics="exists")

formula_aggregator = ltn.Wrapper_Formula_Aggregator(
    ltn.fuzzy_ops.Aggreg_pMeanError(p=2)
)


@tf.function()
def axioms(features_param, *args):
    x = ltn.Variable("x", features_param)
    x_restaurant = ltn.Variable(
        "x_restaurant", features_param[args[CLASSES.index("Restaurant")] == 1]
    )
    x_not_restaurant = ltn.Variable(
        "x_not_restaurant", features_param[args[CLASSES.index("Restaurant")] == 0]
    )
    axioms = [
        # Grounded Predicates
        Forall(x_restaurant, p([x_restaurant, class_restaurant])),
        Forall(x_not_restaurant, Not(p([x_not_restaurant, class_restaurant]))),
        # Axioms
        # Restaurant = (Car or Truck) and Sign
        Forall(
            x,
            Equiv(
                p([x, class_restaurant]),
                And(Or(p([x, class_car]), p([x, class_truck])), p([x, class_sign])),
            ),
        ),
    ]
    sat_level = formula_aggregator(axioms).tensor
    return sat_level


def __get_elements_from_batch(batch_elements):
    features_calculated = batch_elements[0]
    inverted_labels_list = batch_elements[1]
    labels_list = np.transpose(inverted_labels_list)
    return features_calculated, labels_list


for _ in range(len(train_generator)):
    batch_elements = train_generator.next()
    features, labels_list = __get_elements_from_batch(batch_elements)
    print("Initial sat level %.5f" % axioms(batch_elements[0], *labels_list))
    break


""" TRAINING FUNCTIONS """

metrics_dict = {
    "train_sat_kb": keras.metrics.Mean(name="train_sat_kb"),
    "test_sat_kb": keras.metrics.Mean(name="test_sat_kb"),
}

classes_for_confusion_matrix = [
    "Sign",
    "Truck",
    "Car",
]

metricts_conf_matrix = {
    f"{class_name}_train_conf_matrix": tf.Variable(
        initial_value=np.zeros((2, 2), dtype=np.int32)
    )
    for class_name in classes_for_confusion_matrix
}
metricts_conf_matrix_test = {
    f"{class_name}_test_conf_matrix": tf.Variable(
        initial_value=np.zeros((2, 2), dtype=np.int32)
    )
    for class_name in classes_for_confusion_matrix
}
metricts_conf_matrix.update(metricts_conf_matrix_test)


class_train_accuracy_metric = {
    f"{class_name.lower()}_train_accuracy": keras.metrics.Mean(
        name=f"{class_name.lower()}_train_accuracy"
    )
    for class_name in CLASSES
}
metrics_dict.update(class_train_accuracy_metric)

class_test_accuracy_metric = {
    f"{class_name.lower()}_test_accuracy": keras.metrics.Mean(
        name=f"{class_name.lower()}_test_accuracy"
    )
    for class_name in CLASSES
}
metrics_dict.update(class_test_accuracy_metric)


optimizer = keras.optimizers.legacy.Adam(learning_rate=LEARNING_RATE)


@tf.function()
def train_step(*args):
    features_param = args[0]
    labels_list = args[1:]
    with tf.GradientTape() as tape:
        sat = axioms(features_param, *labels_list)
        loss = 1.0 - sat
    gradients = tape.gradient(loss, p.trainable_variables)
    optimizer.apply_gradients(zip(gradients, p.trainable_variables))
    tf.print("Sat Function Result: ", sat)
    metrics_dict["train_sat_kb"](sat)
    predictions = logits_model([features_param])
    calculate_accuracy_for_each_class("train", predictions, labels_list)
    update_confusion_matrix("train", predictions, labels_list)


@tf.function()
def test_step(*args):
    features_param = args[0]
    labels_list = args[1:]
    sat_kb = axioms(features_param, *labels_list)
    metrics_dict["test_sat_kb"](sat_kb)
    predictions = logits_model([features_param])
    calculate_accuracy_for_each_class("test", predictions, labels_list)
    update_confusion_matrix("test", predictions, labels_list)


""" AUXILIARY FUNCTIONS """


def calculate_accuracy_for_each_class(phase: str, predictions, labels_param):
    for i, class_name in enumerate(CLASSES):
        class_labels = labels_param[i]
        y_pred = predictions[:, i]
        y_pred = tf.math.sigmoid(y_pred)
        y_pred = tf.cast(y_pred > 0.5, tf.float32)
        cast_class_labels = tf.cast(class_labels, tf.float32)
        accuracy = tf.reduce_mean(
            tf.cast(tf.equal(y_pred, cast_class_labels), dtype=tf.float32)
        )
        metrics_dict[f"{class_name.lower()}_{phase}_accuracy"](accuracy)


def update_confusion_matrix(phase: str, predictions, labels_param):
    for class_name in classes_for_confusion_matrix:
        class_labels = tf.cast(labels_param[CLASSES.index(class_name)], tf.int32)
        y_pred = predictions[:, CLASSES.index(class_name)]
        y_preds = tf.cast(y_pred > 0.5, tf.int32)
        confusion_matrix = tf.math.confusion_matrix(
            class_labels, y_preds, num_classes=2
        )
        metricts_conf_matrix[f"{class_name}_{phase}_conf_matrix"].assign_add(
            confusion_matrix
        )


def write_results_description_in_txt(txt_file):
    txt_file.write("Results Description\n")
    txt_file.write("Number of examples: %d\n" % TOTAL_EXAMPLES)
    txt_file.write("Train/Test limit: %d\n" % TRAIN_TEST_LIMIT)
    txt_file.write("Epochs: %d\n" % EPOCHS)
    txt_file.write("Complexity of the Dataset: %d\n" % COMPLEXITY_DATASET)
    txt_file.write("Batch size: %d\n" % BATCH_SIZE)
    txt_file.write("Learning rate: %.5f\n" % LEARNING_RATE)
    txt_file.write("Number of classes: %d\n" % len(CLASSES))
    txt_file.write("\n")


def write_conf_matrix(confusion_matrix_path, epoch):
    with open(confusion_matrix_path, "a") as conf_matrix_file:
        conf_matrix_file.write(f"Epoch {epoch}\n")
        for class_name in classes_for_confusion_matrix:
            conf_matrix_file.write(
                f"{class_name} Train Confusion Matrix:\n{metricts_conf_matrix[f'{class_name}_train_conf_matrix'].numpy()}\n"
            )
            conf_matrix_file.write(
                f"{class_name} Test Confusion Matrix:\n{metricts_conf_matrix[f'{class_name}_test_conf_matrix'].numpy()}\n"
            )
        conf_matrix_file.write("\n")


""" SCRIPT """

# scheduled_parameters = defaultdict(lambda: {}) -> This can be used to schedule hyperparameters
template = "Epoch {}"
for metrics_label in metrics_dict.keys():
    template += ", %s: {:.5f}" % metrics_label
if csv_path is not None:
    txt_file = open(txt_path, "w+")
    write_results_description_in_txt(txt_file)
    csv_file = open(csv_path, "w+")

    headers = ",".join(["Epoch"] + list(metrics_dict.keys()))
    csv_template = ",".join(["{}" for _ in range(len(metrics_dict) + 1)])
    csv_file.write(headers + "\n")

for epoch in range(EPOCHS):
    for metrics in metrics_dict.values():
        metrics.reset_states()
    for metric in metricts_conf_matrix.values():
        metric.assign(np.zeros((2, 2), dtype=np.int32))
    for _ in range(len(train_generator)):
        batch_elements = train_generator.next()
        features_param, labels_list = __get_elements_from_batch(batch_elements)
        train_step(features_param, *labels_list)
    for _ in range(len(test_generator)):
        batch_elements = test_generator.next()
        features_param, labels_list = __get_elements_from_batch(batch_elements)
        test_step(features_param, *labels_list)

    metrics_results = [metrics.result() for metrics in metrics_dict.values()]
    print(template.format(epoch, *metrics_results))
    write_conf_matrix(confusion_matrix_path, epoch)
    if csv_path is not None:
        csv_file.write(csv_template.format(epoch, *metrics_results) + "\n")
        csv_file.flush()
if csv_path is not None:
    csv_file.close()
    txt_file.close()


Streaming output truncated to the last 5000 lines.
Sat Function Result:  0.773248851
Sat Function Result:  0.775704086
Sat Function Result:  0.775712311
Sat Function Result:  0.769653
Sat Function Result:  0.770813048
Sat Function Result:  0.774477482
Sat Function Result:  0.776945531
Sat Function Result:  0.76843524
Sat Function Result:  0.775682092
Sat Function Result:  0.773242831
Sat Function Result:  0.775704741
Sat Function Result:  0.769633412
Sat Function Result:  0.776904821
Sat Function Result:  0.778181612
Sat Function Result:  0.770824432
Sat Function Result:  0.774484694
Sat Function Result:  0.774482369
Sat Function Result:  0.769631326
Sat Function Result:  0.774479628
Sat Function Result:  0.775712252
Sat Function Result:  0.774466872
Sat Function Result:  0.770827472
Sat Function Result:  0.773260236
Sat Function Result:  0.775716662
Sat Function Result:  0.772037446
Sat Function Result:  0.775715709
Sat Function Result:  0.773263156
Sat Function Result:  0.769642711
S

KeyboardInterrupt: 

# Testes dia 16/6

In [ ]:

""" DATASET """
data_path = Path(DATA_ROOT, "C" + str(COMPLEXITY_DATASET))
labels = Path(data_path, "labels.csv")
images = Path(data_path, "images")

df = pd.read_csv(labels, index_col=0)
df["id"] = df["id"].apply(lambda x: f"{x}.jpg")
# CLASSES = df.columns.tolist()[1:]

CLASSES = [
    "Cafe",
    "MiscCommercial",
    "Awning",
    "Table",
    "VendingMachine",
    "Statue",
]

datagen = image.ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=0,
    width_shift_range=0,
    height_shift_range=0,
    shear_range=0,
    zoom_range=0,
    horizontal_flip=False,
    fill_mode="nearest",
)

train_generator = datagen.flow_from_dataframe(
    dataframe=df[:TRAIN_TEST_LIMIT],
    directory=str(images),
    x_col="id",
    y_col=CLASSES,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False,
)

test_generator = datagen.flow_from_dataframe(
    dataframe=df[TRAIN_TEST_LIMIT:TOTAL_EXAMPLES],
    directory=str(images),
    x_col="id",
    y_col=CLASSES,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False,
)


"""CNN MODEL"""


class CNNModel(keras.Model):

    def __init__(self, num_classes):
        super(CNNModel, self).__init__()
        self.conv1 = keras.layers.Conv2D(
            16, kernel_size=(3, 3), activation="relu", input_shape=(224, 224, 3)
        )
        self.conv2 = keras.layers.Conv2D(32, kernel_size=(3, 3), activation="relu")
        self.conv3 = keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu")
        self.maxpool = keras.layers.MaxPooling2D(pool_size=(2, 2))
        self.flatten = keras.layers.Flatten()
        self.dropout = keras.layers.Dropout(0.5)
        self.dense1 = keras.layers.Dense(128, activation="relu")
        self.dense2 = keras.layers.Dense(64, activation="relu")
        self.concept_layer = keras.layers.Dense(32, activation="relu")
        self.output_layer = keras.layers.Dense(num_classes)

    def call(self, inputs):
        x = self.conv1(inputs[0])
        x = self.maxpool(x)
        x = self.conv2(x)
        x = self.maxpool(x)
        x = self.conv3(x)
        x = self.maxpool(x)
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.dense1(x)
        x = self.dropout(x)
        x = self.dense2(x)
        y = self.concept_layer(x)
        x = self.output_layer(y)
        result = tf.concat([x, y], axis=1)
        return result


""" LTN MODEL """

# Predicates
logits_model = CNNModel(num_classes=2)
p = ltn.Predicate.FromLogits(
    logits_model, activation_function="sigmoid", with_class_indexing=True
)
# pf = ltn.Predicate.FromLogits(logits_model, activation_function="softmax")

# Constants
class_cafe = ltn.Constant(CLASSES.index("Cafe"), trainable=False)
class_misc_commercial = ltn.Constant(CLASSES.index("MiscCommercial"), trainable=False)
class_awning = ltn.Constant(CLASSES.index("Awning"), trainable=False)
class_table = ltn.Constant(CLASSES.index("Table"), trainable=False)
class_vending_machine = ltn.Constant(CLASSES.index("VendingMachine"), trainable=False)
class_statue = ltn.Constant(CLASSES.index("Statue"), trainable=False)



# Operators
Not = ltn.Wrapper_Connective(ltn.fuzzy_ops.Not_Std())
And = ltn.Wrapper_Connective(ltn.fuzzy_ops.And_Prod())
Or = ltn.Wrapper_Connective(ltn.fuzzy_ops.Or_ProbSum())
Implies = ltn.Wrapper_Connective(ltn.fuzzy_ops.Implies_Reichenbach())
Equiv = ltn.Wrapper_Connective(
    ltn.fuzzy_ops.Equiv(ltn.fuzzy_ops.And_Prod(), ltn.fuzzy_ops.Implies_Reichenbach())
)
Forall = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMeanError(), semantics="forall")
Exists = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMean(), semantics="exists")

formula_aggregator = ltn.Wrapper_Formula_Aggregator(
    ltn.fuzzy_ops.Aggreg_pMeanError(p=2)
)


@tf.function()
def axioms(features_param, *args):
    x = ltn.Variable("x", features_param)
    x_cafe = ltn.Variable("x_cafe", features_param[args[CLASSES.index("Cafe")] == 1])
    x_not_cafe = ltn.Variable(
        "x_not_cafe", features_param[args[CLASSES.index("Cafe")] == 0]
    )
    x_misc_commercial = ltn.Variable(
        "x_misc_commercial", features_param[args[CLASSES.index("MiscCommercial")] == 1]
    )
    x_not_misc_commercial = ltn.Variable(
        "x_not_misc_commercial",
        features_param[args[CLASSES.index("MiscCommercial")] == 0],
    )
    axioms = [
        # Grounded Predicates
        Forall(x_cafe, p([x_cafe, class_cafe])),
        Forall(x_not_cafe, Not(p([x_not_cafe, class_cafe]))),
        Forall(x_misc_commercial, p([x_misc_commercial, class_misc_commercial])),
        Forall(
            x_not_misc_commercial,
            Not(p([x_not_misc_commercial, class_misc_commercial])),
        ),
        # Axioms
        # Cafe = Statue or Vending Machine
        Forall(
            x,
            Equiv(
                p([x, class_cafe]),
                Or(p([x, class_statue]), p([x, class_vending_machine])),
            ),
        ),
        # MiscCommercial = Awning and Table
        Forall(
            x,
            Equiv(
                p([x, class_misc_commercial]),
                And(p([x, class_awning]), p([x, class_table])),
            ),
        ),
    ]
    sat_level = formula_aggregator(axioms).tensor
    return sat_level


def __get_elements_from_batch(batch_elements):
    features_calculated = batch_elements[0]
    inverted_labels_list = batch_elements[1]
    labels_list = np.transpose(inverted_labels_list)
    return features_calculated, labels_list


for _ in range(len(train_generator)):
    batch_elements = train_generator.next()
    features, labels_list = __get_elements_from_batch(batch_elements)
    print("Initial sat level %.5f" % axioms(batch_elements[0], *labels_list))
    break


""" TRAINING FUNCTIONS """

metrics_dict = {
    "train_sat_kb": keras.metrics.Mean(name="train_sat_kb"),
    "test_sat_kb": keras.metrics.Mean(name="test_sat_kb"),
}

metricts_conf_matrix = {
    f"{class_name}_train_conf_matrix": tf.Variable(
        initial_value=np.zeros((2, 2), dtype=np.int32)
    )
    for class_name in CLASSES
}
metricts_conf_matrix_test = {
    f"{class_name}_test_conf_matrix": tf.Variable(
        initial_value=np.zeros((2, 2), dtype=np.int32)
    )
    for class_name in CLASSES
}
metricts_conf_matrix.update(metricts_conf_matrix_test)


class_train_accuracy_metric = {
    f"{class_name.lower()}_train_accuracy": keras.metrics.Mean(
        name=f"{class_name.lower()}_train_accuracy"
    )
    for class_name in CLASSES
}
metrics_dict.update(class_train_accuracy_metric)

class_test_accuracy_metric = {
    f"{class_name.lower()}_test_accuracy": keras.metrics.Mean(
        name=f"{class_name.lower()}_test_accuracy"
    )
    for class_name in CLASSES
}
metrics_dict.update(class_test_accuracy_metric)


optimizer = keras.optimizers.legacy.Adam(learning_rate=LEARNING_RATE)


@tf.function()
def train_step(*args):
    features_param = args[0]
    labels_list = args[1:]
    with tf.GradientTape() as tape:
        sat = axioms(features_param, *labels_list)
        loss = 1.0 - sat
    gradients = tape.gradient(loss, p.trainable_variables)
    optimizer.apply_gradients(zip(gradients, p.trainable_variables))
    tf.print("Sat Function Result: ", sat)
    metrics_dict["train_sat_kb"](sat)
    predictions = logits_model([features_param])
    calculate_accuracy_for_each_class("train", predictions, labels_list)
    update_confusion_matrix("train", predictions, labels_list)


@tf.function()
def test_step(*args):
    features_param = args[0]
    labels_list = args[1:]
    sat_kb = axioms(features_param, *labels_list)
    metrics_dict["test_sat_kb"](sat_kb)
    predictions = logits_model([features_param])
    calculate_accuracy_for_each_class("test", predictions, labels_list)
    update_confusion_matrix("test", predictions, labels_list)


""" AUXILIARY FUNCTIONS """


def calculate_accuracy_for_each_class(phase: str, predictions, labels_param):
    for i, class_name in enumerate(CLASSES):
        class_labels = labels_param[i]
        y_pred = predictions[:, i]
        y_pred = tf.math.sigmoid(y_pred)
        y_pred = tf.cast(y_pred > 0.5, tf.float32)
        cast_class_labels = tf.cast(class_labels, tf.float32)
        accuracy = tf.reduce_mean(
            tf.cast(tf.equal(y_pred, cast_class_labels), dtype=tf.float32)
        )
        metrics_dict[f"{class_name.lower()}_{phase}_accuracy"](accuracy)


def update_confusion_matrix(phase: str, predictions, labels_param):
    for class_name in CLASSES:
        class_labels = tf.cast(labels_param[CLASSES.index(class_name)], tf.int32)
        y_pred = predictions[:, CLASSES.index(class_name)]
        y_preds = tf.cast(y_pred > 0.5, tf.int32)
        confusion_matrix = tf.math.confusion_matrix(
            class_labels, y_preds, num_classes=2
        )
        metricts_conf_matrix[f"{class_name}_{phase}_conf_matrix"].assign_add(
            confusion_matrix
        )


def write_results_description_in_txt(txt_file):
    txt_file.write("Results Description\n")
    txt_file.write("Number of examples: %d\n" % TOTAL_EXAMPLES)
    txt_file.write("Train/Test limit: %d\n" % TRAIN_TEST_LIMIT)
    txt_file.write("Epochs: %d\n" % EPOCHS)
    txt_file.write("Complexity of the Dataset: %d\n" % COMPLEXITY_DATASET)
    txt_file.write("Batch size: %d\n" % BATCH_SIZE)
    txt_file.write("Learning rate: %.5f\n" % LEARNING_RATE)
    txt_file.write("Number of classes: %d\n" % len(CLASSES))
    txt_file.write("\n")


def write_conf_matrix(confusion_matrix_path, epoch):
    with open(confusion_matrix_path, "a") as conf_matrix_file:
        conf_matrix_file.write(f"Epoch {epoch}\n")
        for class_name in CLASSES:
            conf_matrix_file.write(
                f"{class_name} Train Confusion Matrix:\n{metricts_conf_matrix[f'{class_name}_train_conf_matrix'].numpy()}\n"
            )
            conf_matrix_file.write(
                f"{class_name} Test Confusion Matrix:\n{metricts_conf_matrix[f'{class_name}_test_conf_matrix'].numpy()}\n"
            )
        conf_matrix_file.write("\n")


""" SCRIPT """

# scheduled_parameters = defaultdict(lambda: {}) -> This can be used to schedule hyperparameters
template = "Epoch {}"
for metrics_label in metrics_dict.keys():
    template += ", %s: {:.5f}" % metrics_label
if csv_path is not None:
    txt_file = open(txt_path, "w+")
    write_results_description_in_txt(txt_file)
    csv_file = open(csv_path, "w+")

    headers = ",".join(["Epoch"] + list(metrics_dict.keys()))
    csv_template = ",".join(["{}" for _ in range(len(metrics_dict) + 1)])
    csv_file.write(headers + "\n")

for epoch in range(EPOCHS):
    for metrics in metrics_dict.values():
        metrics.reset_states()
    for metric in metricts_conf_matrix.values():
        metric.assign(np.zeros((2, 2), dtype=np.int32))
    for _ in range(len(train_generator)):
        batch_elements = train_generator.next()
        features_param, labels_list = __get_elements_from_batch(batch_elements)
        train_step(features_param, *labels_list)
    for _ in range(len(test_generator)):
        batch_elements = test_generator.next()
        features_param, labels_list = __get_elements_from_batch(batch_elements)
        test_step(features_param, *labels_list)

    metrics_results = [metrics.result() for metrics in metrics_dict.values()]
    print(template.format(epoch, *metrics_results))
    write_conf_matrix(confusion_matrix_path, epoch)
    if csv_path is not None:
        csv_file.write(csv_template.format(epoch, *metrics_results) + "\n")
        csv_file.flush()

# Save the model using TensorFlow SavedModel format
logits_model.save('logits_model', save_format='tf')

# Save the model weights (optional)
logits_model.save_weights('logits_model_weights.h5')

if csv_path is not None:
    csv_file.close()
    txt_file.close()


Found 12000 validated image filenames.
Found 3000 validated image filenames.
Initial sat level 0.51502
Sat Function Result:  0.515292943
Sat Function Result:  0.516421914
Sat Function Result:  0.517578602
Sat Function Result:  0.525239468
Sat Function Result:  0.527584195
Sat Function Result:  0.53093785
Sat Function Result:  0.533828139
Sat Function Result:  0.544113
Sat Function Result:  0.543605447
Sat Function Result:  0.56566751
Sat Function Result:  0.562286615
Sat Function Result:  0.573437214
Sat Function Result:  0.585752249
Sat Function Result:  0.582389951
Sat Function Result:  0.561659694
Sat Function Result:  0.528374732
Sat Function Result:  0.573368609
Sat Function Result:  0.590618551
Sat Function Result:  0.57201767
Sat Function Result:  0.596524119
Sat Function Result:  0.585095167
Sat Function Result:  0.58768177
Sat Function Result:  0.580680788
Sat Function Result:  0.591904283
Sat Function Result:  0.58086
Sat Function Result:  0.606921792
Sat Function Result:  0.

In [ ]:

""" DATASET """
data_path = Path(DATA_ROOT, "C" + str(COMPLEXITY_DATASET))
labels = Path(data_path, "labels.csv")
images = Path(data_path, "images")

df = pd.read_csv(labels, index_col=0)
df["id"] = df["id"].apply(lambda x: f"{x}.jpg")
# CLASSES = df.columns.tolist()[1:]

CLASSES = [
    "Restaurant",
    "Sign",
    "Truck",
    "Car",
]

datagen = image.ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=0,
    width_shift_range=0,
    height_shift_range=0,
    shear_range=0,
    zoom_range=0,
    horizontal_flip=False,
    fill_mode="nearest",
)

train_generator = datagen.flow_from_dataframe(
    dataframe=df[:TRAIN_TEST_LIMIT],
    directory=str(images),
    x_col="id",
    y_col=CLASSES,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False,
)

test_generator = datagen.flow_from_dataframe(
    dataframe=df[TRAIN_TEST_LIMIT:TOTAL_EXAMPLES],
    directory=str(images),
    x_col="id",
    y_col=CLASSES,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False,
)


"""CNN MODEL"""


class CNNModel(keras.Model):

    def __init__(self, num_classes):
        super(CNNModel, self).__init__()
        self.conv1 = keras.layers.Conv2D(
            16, kernel_size=(3, 3), activation="relu", input_shape=(224, 224, 3)
        )
        self.conv2 = keras.layers.Conv2D(32, kernel_size=(3, 3), activation="relu")
        self.conv3 = keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu")
        self.maxpool = keras.layers.MaxPooling2D(pool_size=(2, 2))
        self.flatten = keras.layers.Flatten()
        self.dropout = keras.layers.Dropout(0.5)
        self.dense1 = keras.layers.Dense(128, activation="relu")
        self.dense2 = keras.layers.Dense(64, activation="relu")
        self.dense3 = keras.layers.Dense(32, activation="relu")
        self.concept_layer = keras.layers.Dense(3, activation="relu")
        self.dense4 = keras.layers.Dense(10, activation="relu")
        self.output_layer = keras.layers.Dense(num_classes)

    def call(self, inputs):
        x = self.conv1(inputs[0])
        x = self.maxpool(x)
        x = self.conv2(x)
        x = self.maxpool(x)
        x = self.conv3(x)
        x = self.maxpool(x)
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.dense1(x)
        x = self.dropout(x)
        x = self.dense2(x)
        x = self.dense3(x)
        y = self.concept_layer(x)
        x = self.dense4(y)
        x = self.output_layer(x)
        result = tf.concat([x, y], axis=1)
        return result


""" LTN MODEL """

# Predicates
logits_model = CNNModel(num_classes=1)
p = ltn.Predicate.FromLogits(
    logits_model, activation_function="sigmoid", with_class_indexing=True
)
# pf = ltn.Predicate.FromLogits(logits_model, activation_function="softmax")

# Constants
class_restaurant = ltn.Constant(CLASSES.index("Restaurant"), trainable=False)
class_sign = ltn.Constant(CLASSES.index("Sign"), trainable=False)
class_truck = ltn.Constant(CLASSES.index("Truck"), trainable=False)
class_car = ltn.Constant(CLASSES.index("Car"), trainable=False)


# Operators
Not = ltn.Wrapper_Connective(ltn.fuzzy_ops.Not_Std())
And = ltn.Wrapper_Connective(ltn.fuzzy_ops.And_Prod())
Or = ltn.Wrapper_Connective(ltn.fuzzy_ops.Or_ProbSum())
Implies = ltn.Wrapper_Connective(ltn.fuzzy_ops.Implies_Reichenbach())
Equiv = ltn.Wrapper_Connective(
    ltn.fuzzy_ops.Equiv(ltn.fuzzy_ops.And_Prod(), ltn.fuzzy_ops.Implies_Reichenbach())
)
Forall = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMeanError(), semantics="forall")
Exists = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMean(), semantics="exists")

formula_aggregator = ltn.Wrapper_Formula_Aggregator(
    ltn.fuzzy_ops.Aggreg_pMeanError(p=2)
)


@tf.function()
def axioms(features_param, *args):
    x = ltn.Variable("x", features_param)
    x_restaurant = ltn.Variable(
        "x_restaurant", features_param[args[CLASSES.index("Restaurant")] == 1]
    )
    x_not_restaurant = ltn.Variable(
        "x_not_restaurant", features_param[args[CLASSES.index("Restaurant")] == 0]
    )
    axioms = [
        # Grounded Predicates
        Forall(x_restaurant, p([x_restaurant, class_restaurant])),
        Forall(x_not_restaurant, Not(p([x_not_restaurant, class_restaurant]))),
        # Axioms
        # Restaurant = (Car or Truck) and Sign
        Forall(
            x,
            Equiv(
                p([x, class_restaurant]),
                And(Or(p([x, class_car]), p([x, class_truck])), p([x, class_sign])),
            ),
        ),
    ]
    sat_level = formula_aggregator(axioms).tensor
    return sat_level


def __get_elements_from_batch(batch_elements):
    features_calculated = batch_elements[0]
    inverted_labels_list = batch_elements[1]
    labels_list = np.transpose(inverted_labels_list)
    return features_calculated, labels_list


for _ in range(len(train_generator)):
    batch_elements = train_generator.next()
    features, labels_list = __get_elements_from_batch(batch_elements)
    print("Initial sat level %.5f" % axioms(batch_elements[0], *labels_list))
    break


""" TRAINING FUNCTIONS """

metrics_dict = {
    "train_sat_kb": keras.metrics.Mean(name="train_sat_kb"),
    "test_sat_kb": keras.metrics.Mean(name="test_sat_kb"),
}


metricts_conf_matrix = {
    f"{class_name}_train_conf_matrix": tf.Variable(
        initial_value=np.zeros((2, 2), dtype=np.int32)
    )
    for class_name in CLASSES
}
metricts_conf_matrix_test = {
    f"{class_name}_test_conf_matrix": tf.Variable(
        initial_value=np.zeros((2, 2), dtype=np.int32)
    )
    for class_name in CLASSES
}
metricts_conf_matrix.update(metricts_conf_matrix_test)


class_train_accuracy_metric = {
    f"{class_name.lower()}_train_accuracy": keras.metrics.Mean(
        name=f"{class_name.lower()}_train_accuracy"
    )
    for class_name in CLASSES
}
metrics_dict.update(class_train_accuracy_metric)

class_test_accuracy_metric = {
    f"{class_name.lower()}_test_accuracy": keras.metrics.Mean(
        name=f"{class_name.lower()}_test_accuracy"
    )
    for class_name in CLASSES
}
metrics_dict.update(class_test_accuracy_metric)


optimizer = keras.optimizers.legacy.Adam(learning_rate=LEARNING_RATE)


@tf.function()
def train_step(*args):
    features_param = args[0]
    labels_list = args[1:]
    with tf.GradientTape() as tape:
        sat = axioms(features_param, *labels_list)
        loss = 1.0 - sat
    gradients = tape.gradient(loss, p.trainable_variables)
    optimizer.apply_gradients(zip(gradients, p.trainable_variables))
    tf.print("Sat Function Result: ", sat)
    metrics_dict["train_sat_kb"](sat)
    predictions = logits_model([features_param])
    calculate_accuracy_for_each_class("train", predictions, labels_list)
    update_confusion_matrix("train", predictions, labels_list)


@tf.function()
def test_step(*args):
    features_param = args[0]
    labels_list = args[1:]
    sat_kb = axioms(features_param, *labels_list)
    metrics_dict["test_sat_kb"](sat_kb)
    predictions = logits_model([features_param])
    calculate_accuracy_for_each_class("test", predictions, labels_list)
    update_confusion_matrix("test", predictions, labels_list)


""" AUXILIARY FUNCTIONS """


def calculate_accuracy_for_each_class(phase: str, predictions, labels_param):
    for i, class_name in enumerate(CLASSES):
        class_labels = labels_param[i]
        y_pred = predictions[:, i]
        y_pred = tf.math.sigmoid(y_pred)
        y_pred = tf.cast(y_pred > 0.5, tf.float32)
        cast_class_labels = tf.cast(class_labels, tf.float32)
        accuracy = tf.reduce_mean(
            tf.cast(tf.equal(y_pred, cast_class_labels), dtype=tf.float32)
        )
        metrics_dict[f"{class_name.lower()}_{phase}_accuracy"](accuracy)


def update_confusion_matrix(phase: str, predictions, labels_param):
    for class_name in CLASSES:
        class_labels = tf.cast(labels_param[CLASSES.index(class_name)], tf.int32)
        y_pred = predictions[:, CLASSES.index(class_name)]
        y_preds = tf.cast(y_pred > 0.5, tf.int32)
        confusion_matrix = tf.math.confusion_matrix(
            class_labels, y_preds, num_classes=2
        )
        metricts_conf_matrix[f"{class_name}_{phase}_conf_matrix"].assign_add(
            confusion_matrix
        )


def write_results_description_in_txt(txt_file):
    txt_file.write("Results Description\n")
    txt_file.write("Number of examples: %d\n" % TOTAL_EXAMPLES)
    txt_file.write("Train/Test limit: %d\n" % TRAIN_TEST_LIMIT)
    txt_file.write("Epochs: %d\n" % EPOCHS)
    txt_file.write("Complexity of the Dataset: %d\n" % COMPLEXITY_DATASET)
    txt_file.write("Batch size: %d\n" % BATCH_SIZE)
    txt_file.write("Learning rate: %.5f\n" % LEARNING_RATE)
    txt_file.write("Number of classes: %d\n" % len(CLASSES))
    txt_file.write("\n")


def write_conf_matrix(confusion_matrix_path, epoch):
    with open(confusion_matrix_path, "a") as conf_matrix_file:
        conf_matrix_file.write(f"Epoch {epoch}\n")
        for class_name in CLASSES:
            conf_matrix_file.write(
                f"{class_name} Train Confusion Matrix:\n{metricts_conf_matrix[f'{class_name}_train_conf_matrix'].numpy()}\n"
            )
            conf_matrix_file.write(
                f"{class_name} Test Confusion Matrix:\n{metricts_conf_matrix[f'{class_name}_test_conf_matrix'].numpy()}\n"
            )
        conf_matrix_file.write("\n")


""" SCRIPT """

# scheduled_parameters = defaultdict(lambda: {}) -> This can be used to schedule hyperparameters
template = "Epoch {}"
for metrics_label in metrics_dict.keys():
    template += ", %s: {:.5f}" % metrics_label
if csv_path is not None:
    txt_file = open(txt_path, "w+")
    write_results_description_in_txt(txt_file)
    csv_file = open(csv_path, "w+")

    headers = ",".join(["Epoch"] + list(metrics_dict.keys()))
    csv_template = ",".join(["{}" for _ in range(len(metrics_dict) + 1)])
    csv_file.write(headers + "\n")

for epoch in range(EPOCHS):
    for metrics in metrics_dict.values():
        metrics.reset_states()
    for metric in metricts_conf_matrix.values():
        metric.assign(np.zeros((2, 2), dtype=np.int32))
    for _ in range(len(train_generator)):
        batch_elements = train_generator.next()
        features_param, labels_list = __get_elements_from_batch(batch_elements)
        train_step(features_param, *labels_list)
    for _ in range(len(test_generator)):
        batch_elements = test_generator.next()
        features_param, labels_list = __get_elements_from_batch(batch_elements)
        test_step(features_param, *labels_list)

    metrics_results = [metrics.result() for metrics in metrics_dict.values()]
    print(template.format(epoch, *metrics_results))
    write_conf_matrix(confusion_matrix_path, epoch)
    if csv_path is not None:
        csv_file.write(csv_template.format(epoch, *metrics_results) + "\n")
        csv_file.flush()

# Save the model using TensorFlow SavedModel format
logits_model.save('logits_model', save_format='tf')

# Save the model weights (optional)
logits_model.save_weights('logits_model_weights.h5')

if csv_path is not None:
    csv_file.close()
    txt_file.close()


Found 12000 validated image filenames.
Found 3000 validated image filenames.
Initial sat level 0.51876
Sat Function Result:  0.518796921
Sat Function Result:  0.520593405
Sat Function Result:  0.522528648
Sat Function Result:  0.522870779
Sat Function Result:  0.52323842
Sat Function Result:  0.524044275
Sat Function Result:  0.523274302
Sat Function Result:  0.523902535
Sat Function Result:  0.525948
Sat Function Result:  0.525753736
Sat Function Result:  0.524494767
Sat Function Result:  0.526908755
Sat Function Result:  0.529859185
Sat Function Result:  0.528576195
Sat Function Result:  0.532239318
Sat Function Result:  0.529401898
Sat Function Result:  0.535800576
Sat Function Result:  0.543148875
Sat Function Result:  0.551506817
Sat Function Result:  0.538702488
Sat Function Result:  0.552854955
Sat Function Result:  0.554904938
Sat Function Result:  0.546443105
Sat Function Result:  0.552384853
Sat Function Result:  0.556366444
Sat Function Result:  0.580186725
Sat Function Resu

KeyboardInterrupt: 

# Testes dia 23/6

In [ ]:
""" DATASET """

data_path = Path(DATA_ROOT, "C" + str(COMPLEXITY_DATASET))
labels = Path(data_path, "labels.csv")
images = Path(data_path, "images")

df = pd.read_csv(labels, index_col=0)
df["id"] = df["id"].apply(lambda x: f"{x}.jpg")
# CLASSES = df.columns.tolist()[1:]

CLASSES = [
    "MiscCommercial",
    "Restaurant",
    "Industrial",
    "MiscIndustrial",
    "MiscResidential",
    "Table",
    "Awning",
    "Sign",
    "Truck",
    "Car",
]

datagen = image.ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=0,
    width_shift_range=0,
    height_shift_range=0,
    shear_range=0,
    zoom_range=0,
    horizontal_flip=False,
    fill_mode="nearest",
)

train_generator = datagen.flow_from_dataframe(
    dataframe=df[:TRAIN_TEST_LIMIT],
    directory=str(images),
    x_col="id",
    y_col=CLASSES,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False,
)

test_generator = datagen.flow_from_dataframe(
    dataframe=df[TRAIN_TEST_LIMIT:TOTAL_EXAMPLES],
    directory=str(images),
    x_col="id",
    y_col=CLASSES,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False,
)


"""CNN MODEL"""


class CNNModel(keras.Model):

    def __init__(self, num_classes):
        super(CNNModel, self).__init__()
        self.conv1 = keras.layers.Conv2D(
            16, kernel_size=(3, 3), activation="relu", input_shape=(224, 224, 3)
        )
        self.conv2 = keras.layers.Conv2D(32, kernel_size=(3, 3), activation="relu")
        self.conv3 = keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu")
        self.maxpool = keras.layers.MaxPooling2D(pool_size=(2, 2))
        self.flatten = keras.layers.Flatten()
        self.dropout = keras.layers.Dropout(0.5)
        self.dense1 = keras.layers.Dense(128, activation="relu")
        self.dense2 = keras.layers.Dense(64, activation="relu")
        self.dense3 = keras.layers.Dense(32, activation="relu")
        self.concept_layer = keras.layers.Dense(12, activation="relu")
        self.output_layer = keras.layers.Dense(num_classes)

    def call(self, inputs):
        x = self.conv1(inputs[0])
        x = self.maxpool(x)
        x = self.conv2(x)
        x = self.maxpool(x)
        x = self.conv3(x)
        x = self.maxpool(x)
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.dense1(x)
        x = self.dropout(x)
        x = self.dense2(x)
        x = self.dense3(x)
        y = self.concept_layer(x)
        x = self.output_layer(y)
        result = tf.concat([x, y], axis=1)
        return result


""" LTN MODEL """

# Predicates
logits_model = CNNModel(num_classes=5)
p = ltn.Predicate.FromLogits(
    logits_model, activation_function="sigmoid", with_class_indexing=True
)
# pf = ltn.Predicate.FromLogits(logits_model, activation_function="softmax")

# Constants
class_misc_commercial = ltn.Constant(CLASSES.index("MiscCommercial"), trainable=False)
class_restaurant = ltn.Constant(CLASSES.index("Restaurant"), trainable=False)
class_industrial = ltn.Constant(CLASSES.index("Industrial"), trainable=False)
class_misc_industrial = ltn.Constant(CLASSES.index("MiscIndustrial"), trainable=False)
class_misc_residential = ltn.Constant(CLASSES.index("MiscResidential"), trainable=False)
class_table = ltn.Constant(CLASSES.index("Table"), trainable=False)
class_awning = ltn.Constant(CLASSES.index("Awning"), trainable=False)
class_sign = ltn.Constant(CLASSES.index("Sign"), trainable=False)
class_truck = ltn.Constant(CLASSES.index("Truck"), trainable=False)
class_car = ltn.Constant(CLASSES.index("Car"), trainable=False)


# Operators
Not = ltn.Wrapper_Connective(ltn.fuzzy_ops.Not_Std())
And = ltn.Wrapper_Connective(ltn.fuzzy_ops.And_Prod())
Or = ltn.Wrapper_Connective(ltn.fuzzy_ops.Or_ProbSum())
Implies = ltn.Wrapper_Connective(ltn.fuzzy_ops.Implies_Reichenbach())
Equiv = ltn.Wrapper_Connective(
    ltn.fuzzy_ops.Equiv(ltn.fuzzy_ops.And_Prod(), ltn.fuzzy_ops.Implies_Reichenbach())
)
Forall = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMeanError(), semantics="forall")
Exists = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMean(), semantics="exists")

formula_aggregator = ltn.Wrapper_Formula_Aggregator(
    ltn.fuzzy_ops.Aggreg_pMeanError(p=2)
)


@tf.function()
def axioms(features_param, *args):
    x = ltn.Variable("x", features_param)
    x_misc_commercial = ltn.Variable(
        "x_misc_commercial",
        features_param[args[CLASSES.index("MiscCommercial")] == 1],
    )
    x_not_misc_commercial = ltn.Variable(
        "x_not_misc_commercial",
        features_param[args[CLASSES.index("MiscCommercial")] == 0],
    )
    x_restaurant = ltn.Variable(
        "x_restaurant", features_param[args[CLASSES.index("Restaurant")] == 1]
    )
    x_not_restaurant = ltn.Variable(
        "x_not_restaurant", features_param[args[CLASSES.index("Restaurant")] == 0]
    )
    x_industrial = ltn.Variable(
        "x_industrial", features_param[args[CLASSES.index("Industrial")] == 1]
    )
    x_not_industrial = ltn.Variable(
        "x_not_industrial", features_param[args[CLASSES.index("Industrial")] == 0]
    )
    x_misc_industrial = ltn.Variable(
        "x_misc_industrial",
        features_param[args[CLASSES.index("MiscIndustrial")] == 1],
    )
    x_not_misc_industrial = ltn.Variable(
        "x_not_misc_industrial",
        features_param[args[CLASSES.index("MiscIndustrial")] == 0],
    )
    x_misc_residential = ltn.Variable(
        "x_misc_residential",
        features_param[args[CLASSES.index("MiscResidential")] == 1],
    )
    x_not_misc_residential = ltn.Variable(
        "x_not_misc_residential",
        features_param[args[CLASSES.index("MiscResidential")] == 0],
    )
    axioms = [
        # Grounded Predicates
        Forall(x_misc_commercial, p([x_misc_commercial, class_misc_commercial])),
        Forall(
            x_not_misc_commercial,
            Not(p([x_not_misc_commercial, class_misc_commercial])),
        ),
        Forall(x_restaurant, p([x_restaurant, class_restaurant])),
        Forall(x_not_restaurant, Not(p([x_not_restaurant, class_restaurant]))),
        Forall(x_industrial, p([x_industrial, class_industrial])),
        Forall(x_not_industrial, Not(p([x_not_industrial, class_industrial]))),
        Forall(x_misc_industrial, p([x_misc_industrial, class_misc_industrial])),
        Forall(
            x_not_misc_industrial,
            Not(p([x_not_misc_industrial, class_misc_industrial])),
        ),
        Forall(x_misc_residential, p([x_misc_residential, class_misc_residential])),
        Forall(
            x_not_misc_residential,
            Not(p([x_not_misc_residential, class_misc_residential])),
        ),
        # Axioms
        # Not (Car and Truck)
        Forall(x, Not(And(p([x, class_car]), p([x, class_truck])))),
        # MiscCommercial = Awning and Table
        Forall(
            x,
            Equiv(
                p([x, class_misc_commercial]),
                And(p([x, class_awning]), p([x, class_table])),
            ),
        ),
        # Restaurant = (Car or Truck) and Sign
        Forall(
            x,
            Equiv(
                p([x, class_restaurant]),
                And(Or(p([x, class_car]), p([x, class_truck])), p([x, class_sign])),
            ),
        ),
        # Industrial = not Table
        Forall(
            x,
            Equiv(p([x, class_industrial]), Not(p([x, class_table]))),
        ),
        # Misc Industrial = (not Awning) and Truck
        Forall(
            x,
            Equiv(
                p([x, class_misc_industrial]),
                And(Not(p([x, class_awning])), p([x, class_truck])),
            ),
        ),
        # Misc Residential = not (awning and table) and tiled roof
        Forall(
            x,
            Equiv(
                p([x, class_misc_residential]),
                Not(And(p([x, class_awning]), p([x, class_table]))),
            ),
        ),
    ]
    sat_level = formula_aggregator(axioms).tensor
    return sat_level


def __get_elements_from_batch(batch_elements):
    features_calculated = batch_elements[0]
    inverted_labels_list = batch_elements[1]
    labels_list = np.transpose(inverted_labels_list)
    return features_calculated, labels_list


for _ in range(len(train_generator)):
    batch_elements = train_generator.next()
    features, labels_list = __get_elements_from_batch(batch_elements)
    print("Initial sat level %.5f" % axioms(batch_elements[0], *labels_list))
    break


""" TRAINING FUNCTIONS """

metrics_dict = {
    "train_sat_kb": keras.metrics.Mean(name="train_sat_kb"),
    "test_sat_kb": keras.metrics.Mean(name="test_sat_kb"),
}


metricts_conf_matrix = {
    f"{class_name}_train_conf_matrix": tf.Variable(
        initial_value=np.zeros((2, 2), dtype=np.int32)
    )
    for class_name in CLASSES
}
metricts_conf_matrix_test = {
    f"{class_name}_test_conf_matrix": tf.Variable(
        initial_value=np.zeros((2, 2), dtype=np.int32)
    )
    for class_name in CLASSES
}
metricts_conf_matrix.update(metricts_conf_matrix_test)


class_train_accuracy_metric = {
    f"{class_name.lower()}_train_accuracy": keras.metrics.Mean(
        name=f"{class_name.lower()}_train_accuracy"
    )
    for class_name in CLASSES
}
metrics_dict.update(class_train_accuracy_metric)

class_test_accuracy_metric = {
    f"{class_name.lower()}_test_accuracy": keras.metrics.Mean(
        name=f"{class_name.lower()}_test_accuracy"
    )
    for class_name in CLASSES
}
metrics_dict.update(class_test_accuracy_metric)


optimizer = keras.optimizers.legacy.Adam(learning_rate=LEARNING_RATE)


@tf.function()
def train_step(*args):
    features_param = args[0]
    labels_list = args[1:]
    with tf.GradientTape() as tape:
        sat = axioms(features_param, *labels_list)
        loss = 1.0 - sat
    gradients = tape.gradient(loss, p.trainable_variables)
    optimizer.apply_gradients(zip(gradients, p.trainable_variables))
    tf.print("Sat Function Result: ", sat)
    metrics_dict["train_sat_kb"](sat)
    predictions = logits_model([features_param])
    calculate_accuracy_for_each_class("train", predictions, labels_list)
    update_confusion_matrix("train", predictions, labels_list)


@tf.function()
def test_step(*args):
    features_param = args[0]
    labels_list = args[1:]
    sat_kb = axioms(features_param, *labels_list)
    metrics_dict["test_sat_kb"](sat_kb)
    predictions = logits_model([features_param])
    calculate_accuracy_for_each_class("test", predictions, labels_list)
    update_confusion_matrix("test", predictions, labels_list)


""" AUXILIARY FUNCTIONS """


def calculate_accuracy_for_each_class(phase: str, predictions, labels_param):
    for i, class_name in enumerate(CLASSES):
        class_labels = labels_param[i]
        y_pred = predictions[:, i]
        y_pred = tf.math.sigmoid(y_pred)
        y_pred = tf.cast(y_pred > 0.5, tf.float32)
        cast_class_labels = tf.cast(class_labels, tf.float32)
        accuracy = tf.reduce_mean(
            tf.cast(tf.equal(y_pred, cast_class_labels), dtype=tf.float32)
        )
        metrics_dict[f"{class_name.lower()}_{phase}_accuracy"](accuracy)


def update_confusion_matrix(phase: str, predictions, labels_param):
    for class_name in CLASSES:
        class_labels = tf.cast(labels_param[CLASSES.index(class_name)], tf.int32)
        y_pred = predictions[:, CLASSES.index(class_name)]
        y_preds = tf.cast(y_pred > 0.5, tf.int32)
        confusion_matrix = tf.math.confusion_matrix(
            class_labels, y_preds, num_classes=2
        )
        metricts_conf_matrix[f"{class_name}_{phase}_conf_matrix"].assign_add(
            confusion_matrix
        )


def write_results_description_in_txt(txt_file):
    txt_file.write("Results Description\n")
    txt_file.write("Number of examples: %d\n" % TOTAL_EXAMPLES)
    txt_file.write("Train/Test limit: %d\n" % TRAIN_TEST_LIMIT)
    txt_file.write("Epochs: %d\n" % EPOCHS)
    txt_file.write("Complexity of the Dataset: %d\n" % COMPLEXITY_DATASET)
    txt_file.write("Batch size: %d\n" % BATCH_SIZE)
    txt_file.write("Learning rate: %.5f\n" % LEARNING_RATE)
    txt_file.write("Number of classes: %d\n" % len(CLASSES))
    txt_file.write("\n")


def write_conf_matrix(confusion_matrix_path, epoch):
    with open(confusion_matrix_path, "a") as conf_matrix_file:
        conf_matrix_file.write(f"Epoch {epoch}\n")
        for class_name in CLASSES:
            conf_matrix_file.write(
                f"{class_name} Train Confusion Matrix:\n{metricts_conf_matrix[f'{class_name}_train_conf_matrix'].numpy()}\n"
            )
            conf_matrix_file.write(
                f"{class_name} Test Confusion Matrix:\n{metricts_conf_matrix[f'{class_name}_test_conf_matrix'].numpy()}\n"
            )
        conf_matrix_file.write("\n")


""" SCRIPT """

# scheduled_parameters = defaultdict(lambda: {}) -> This can be used to schedule hyperparameters
template = "Epoch {}"
for metrics_label in metrics_dict.keys():
    template += ", %s: {:.5f}" % metrics_label
if csv_path is not None:
    txt_file = open(txt_path, "w+")
    write_results_description_in_txt(txt_file)
    csv_file = open(csv_path, "w+")

    headers = ",".join(["Epoch"] + list(metrics_dict.keys()))
    csv_template = ",".join(["{}" for _ in range(len(metrics_dict) + 1)])
    csv_file.write(headers + "\n")

for epoch in range(EPOCHS):
    for metrics in metrics_dict.values():
        metrics.reset_states()
    for metric in metricts_conf_matrix.values():
        metric.assign(np.zeros((2, 2), dtype=np.int32))
    for _ in range(len(train_generator)):
        batch_elements = train_generator.next()
        features_param, labels_list = __get_elements_from_batch(batch_elements)
        train_step(features_param, *labels_list)
    for _ in range(len(test_generator)):
        batch_elements = test_generator.next()
        features_param, labels_list = __get_elements_from_batch(batch_elements)
        test_step(features_param, *labels_list)

    metrics_results = [metrics.result() for metrics in metrics_dict.values()]
    print(template.format(epoch, *metrics_results))
    write_conf_matrix(confusion_matrix_path, epoch)
    if csv_path is not None:
        csv_file.write(csv_template.format(epoch, *metrics_results) + "\n")
        csv_file.flush()

logits_model.save_weights("logits_model_weights.h5")

if csv_path is not None:
    csv_file.close()
    txt_file.close()


Found 12000 validated image filenames.
Found 3000 validated image filenames.
Initial sat level 0.52810
Sat Function Result:  0.528160572
Sat Function Result:  0.523783803
Sat Function Result:  0.520551264
Sat Function Result:  0.528316617
Sat Function Result:  0.531965554
Sat Function Result:  0.532332778
Sat Function Result:  0.53279984
Sat Function Result:  0.533212543
Sat Function Result:  0.53385222
Sat Function Result:  0.53449285
Sat Function Result:  0.535809755
Sat Function Result:  0.53769052
Sat Function Result:  0.539083481
Sat Function Result:  0.541224122
Sat Function Result:  0.544148088
Sat Function Result:  0.546126
Sat Function Result:  0.544475675
Sat Function Result:  0.547771692
Sat Function Result:  0.553378284
Sat Function Result:  0.551700354
Sat Function Result:  0.557722
Sat Function Result:  0.55882591
Sat Function Result:  0.559411168
Sat Function Result:  0.560417652
Sat Function Result:  0.557699859
Sat Function Result:  0.568563282
Sat Function Result:  0.

In [ ]:

""" DATASET """

data_path = Path(DATA_ROOT, "C" + str(COMPLEXITY_DATASET))
labels = Path(data_path, "labels.csv")
images = Path(data_path, "images")

df = pd.read_csv(labels, index_col=0)
df["id"] = df["id"].apply(lambda x: f"{x}.jpg")
# CLASSES = df.columns.tolist()[1:]

CLASSES = [
    "MiscIndustrial",
    "PowerPlant",
    "WaterTreatment",
    "Awning",
    "Truck",
    "Chimney",
    "Pipe",
]

datagen = image.ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=0,
    width_shift_range=0,
    height_shift_range=0,
    shear_range=0,
    zoom_range=0,
    horizontal_flip=False,
    fill_mode="nearest",
)

train_generator = datagen.flow_from_dataframe(
    dataframe=df[:TRAIN_TEST_LIMIT],
    directory=str(images),
    x_col="id",
    y_col=CLASSES,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False,
)

test_generator = datagen.flow_from_dataframe(
    dataframe=df[TRAIN_TEST_LIMIT:TOTAL_EXAMPLES],
    directory=str(images),
    x_col="id",
    y_col=CLASSES,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False,
)


"""CNN MODEL"""


class CNNModel(keras.Model):

    def __init__(self, num_classes):
        super(CNNModel, self).__init__()
        self.conv1 = keras.layers.Conv2D(
            16, kernel_size=(3, 3), activation="relu", input_shape=(224, 224, 3)
        )
        self.conv2 = keras.layers.Conv2D(32, kernel_size=(3, 3), activation="relu")
        self.conv3 = keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu")
        self.maxpool = keras.layers.MaxPooling2D(pool_size=(2, 2))
        self.flatten = keras.layers.Flatten()
        self.dropout = keras.layers.Dropout(0.5)
        self.dense1 = keras.layers.Dense(128, activation="relu")
        self.dense2 = keras.layers.Dense(64, activation="relu")
        self.dense3 = keras.layers.Dense(32, activation="relu")
        self.concept_layer = keras.layers.Dense(12, activation="relu")
        self.output_layer = keras.layers.Dense(num_classes)

    def call(self, inputs):
        x = self.conv1(inputs[0])
        x = self.maxpool(x)
        x = self.conv2(x)
        x = self.maxpool(x)
        x = self.conv3(x)
        x = self.maxpool(x)
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.dense1(x)
        x = self.dropout(x)
        x = self.dense2(x)
        x = self.dense3(x)
        y = self.concept_layer(x)
        x = self.output_layer(y)
        result = tf.concat([x, y], axis=1)
        return result


""" LTN MODEL """

# Predicates
logits_model = CNNModel(num_classes=3)
p = ltn.Predicate.FromLogits(
    logits_model, activation_function="sigmoid", with_class_indexing=True
)
# pf = ltn.Predicate.FromLogits(logits_model, activation_function="softmax")

# Constants
class_misc_industrial = ltn.Constant(CLASSES.index("MiscIndustrial"), trainable=False)
class_power_plant = ltn.Constant(CLASSES.index("PowerPlant"), trainable=False)
class_water_treatment = ltn.Constant(CLASSES.index("WaterTreatment"), trainable=False)
class_awning = ltn.Constant(CLASSES.index("Awning"), trainable=False)
class_truck = ltn.Constant(CLASSES.index("Truck"), trainable=False)
class_chimney = ltn.Constant(CLASSES.index("Chimney"), trainable=False)
class_pipe = ltn.Constant(CLASSES.index("Pipe"), trainable=False)


# Operators
Not = ltn.Wrapper_Connective(ltn.fuzzy_ops.Not_Std())
And = ltn.Wrapper_Connective(ltn.fuzzy_ops.And_Prod())
Or = ltn.Wrapper_Connective(ltn.fuzzy_ops.Or_ProbSum())
Implies = ltn.Wrapper_Connective(ltn.fuzzy_ops.Implies_Reichenbach())
Equiv = ltn.Wrapper_Connective(
    ltn.fuzzy_ops.Equiv(ltn.fuzzy_ops.And_Prod(), ltn.fuzzy_ops.Implies_Reichenbach())
)
Forall = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMeanError(), semantics="forall")
Exists = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMean(), semantics="exists")

formula_aggregator = ltn.Wrapper_Formula_Aggregator(
    ltn.fuzzy_ops.Aggreg_pMeanError(p=2)
)


@tf.function()
def axioms(features_param, *args):
    x = ltn.Variable("x", features_param)
    x_misc_industrial = ltn.Variable(
        "x_misc_industrial",
        features_param[args[CLASSES.index("MiscIndustrial")] == 1],
    )
    x_not_misc_industrial = ltn.Variable(
        "x_not_misc_industrial",
        features_param[args[CLASSES.index("MiscIndustrial")] == 0],
    )
    x_power_plant = ltn.Variable(
        "x_power_plant", features_param[args[CLASSES.index("PowerPlant")] == 1]
    )
    x_not_power_plant = ltn.Variable(
        "x_not_power_plant", features_param[args[CLASSES.index("PowerPlant")] == 0]
    )
    x_water_treatment = ltn.Variable(
        "x_water_treatment",
        features_param[args[CLASSES.index("WaterTreatment")] == 1],
    )
    x_not_water_treatment = ltn.Variable(
        "x_not_water_treatment",
        features_param[args[CLASSES.index("WaterTreatment")] == 0],
    )
    axioms = [
        # Grounded Predicates
        Forall(x_misc_industrial, p([x_misc_industrial, class_misc_industrial])),
        Forall(
            x_not_misc_industrial,
            Not(p([x_not_misc_industrial, class_misc_industrial])),
        ),
        Forall(x_power_plant, p([x_power_plant, class_power_plant])),
        Forall(x_not_power_plant, Not(p([x_not_power_plant, class_power_plant]))),
        Forall(x_water_treatment, p([x_water_treatment, class_water_treatment])),
        Forall(
            x_not_water_treatment,
            Not(p([x_not_water_treatment, class_water_treatment])),
        ),
        # Axioms
        # Misc Industrial = (not Awning) and Truck
        Forall(
            x,
            Equiv(
                p([x, class_misc_industrial]),
                And(Not(p([x, class_awning])), p([x, class_truck])),
            ),
        ),
        # Power Plant = Chimney and Pipe
        Forall(
            x,
            Equiv(
                p([x, class_power_plant]),
                And(p([x, class_chimney]), p([x, class_pipe])),
            ),
        ),
        # Water Treatment = Pipe and Truck
        Forall(
            x,
            Equiv(
                p([x, class_water_treatment]),
                And(p([x, class_pipe]), p([x, class_truck])),
            ),
        ),
    ]
    sat_level = formula_aggregator(axioms).tensor
    return sat_level


def __get_elements_from_batch(batch_elements):
    features_calculated = batch_elements[0]
    inverted_labels_list = batch_elements[1]
    labels_list = np.transpose(inverted_labels_list)
    return features_calculated, labels_list


for _ in range(len(train_generator)):
    batch_elements = train_generator.next()
    features, labels_list = __get_elements_from_batch(batch_elements)
    print("Initial sat level %.5f" % axioms(batch_elements[0], *labels_list))
    break


""" TRAINING FUNCTIONS """

metrics_dict = {
    "train_sat_kb": keras.metrics.Mean(name="train_sat_kb"),
    "test_sat_kb": keras.metrics.Mean(name="test_sat_kb"),
}


metricts_conf_matrix = {
    f"{class_name}_train_conf_matrix": tf.Variable(
        initial_value=np.zeros((2, 2), dtype=np.int32)
    )
    for class_name in CLASSES
}
metricts_conf_matrix_test = {
    f"{class_name}_test_conf_matrix": tf.Variable(
        initial_value=np.zeros((2, 2), dtype=np.int32)
    )
    for class_name in CLASSES
}
metricts_conf_matrix.update(metricts_conf_matrix_test)


class_train_accuracy_metric = {
    f"{class_name.lower()}_train_accuracy": keras.metrics.Mean(
        name=f"{class_name.lower()}_train_accuracy"
    )
    for class_name in CLASSES
}
metrics_dict.update(class_train_accuracy_metric)

class_test_accuracy_metric = {
    f"{class_name.lower()}_test_accuracy": keras.metrics.Mean(
        name=f"{class_name.lower()}_test_accuracy"
    )
    for class_name in CLASSES
}
metrics_dict.update(class_test_accuracy_metric)


optimizer = keras.optimizers.legacy.Adam(learning_rate=LEARNING_RATE)


@tf.function()
def train_step(*args):
    features_param = args[0]
    labels_list = args[1:]
    with tf.GradientTape() as tape:
        sat = axioms(features_param, *labels_list)
        loss = 1.0 - sat
    gradients = tape.gradient(loss, p.trainable_variables)
    optimizer.apply_gradients(zip(gradients, p.trainable_variables))
    tf.print("Sat Function Result: ", sat)
    metrics_dict["train_sat_kb"](sat)
    predictions = logits_model([features_param])
    calculate_accuracy_for_each_class("train", predictions, labels_list)
    update_confusion_matrix("train", predictions, labels_list)


@tf.function()
def test_step(*args):
    features_param = args[0]
    labels_list = args[1:]
    sat_kb = axioms(features_param, *labels_list)
    metrics_dict["test_sat_kb"](sat_kb)
    predictions = logits_model([features_param])
    calculate_accuracy_for_each_class("test", predictions, labels_list)
    update_confusion_matrix("test", predictions, labels_list)


""" AUXILIARY FUNCTIONS """


def calculate_accuracy_for_each_class(phase: str, predictions, labels_param):
    for i, class_name in enumerate(CLASSES):
        class_labels = labels_param[i]
        y_pred = predictions[:, i]
        y_pred = tf.math.sigmoid(y_pred)
        y_pred = tf.cast(y_pred > 0.5, tf.float32)
        cast_class_labels = tf.cast(class_labels, tf.float32)
        accuracy = tf.reduce_mean(
            tf.cast(tf.equal(y_pred, cast_class_labels), dtype=tf.float32)
        )
        metrics_dict[f"{class_name.lower()}_{phase}_accuracy"](accuracy)


def update_confusion_matrix(phase: str, predictions, labels_param):
    for class_name in CLASSES:
        class_labels = tf.cast(labels_param[CLASSES.index(class_name)], tf.int32)
        y_pred = predictions[:, CLASSES.index(class_name)]
        y_preds = tf.cast(y_pred > 0.5, tf.int32)
        confusion_matrix = tf.math.confusion_matrix(
            class_labels, y_preds, num_classes=2
        )
        metricts_conf_matrix[f"{class_name}_{phase}_conf_matrix"].assign_add(
            confusion_matrix
        )


def write_results_description_in_txt(txt_file):
    txt_file.write("Results Description\n")
    txt_file.write("Number of examples: %d\n" % TOTAL_EXAMPLES)
    txt_file.write("Train/Test limit: %d\n" % TRAIN_TEST_LIMIT)
    txt_file.write("Epochs: %d\n" % EPOCHS)
    txt_file.write("Complexity of the Dataset: %d\n" % COMPLEXITY_DATASET)
    txt_file.write("Batch size: %d\n" % BATCH_SIZE)
    txt_file.write("Learning rate: %.5f\n" % LEARNING_RATE)
    txt_file.write("Number of classes: %d\n" % len(CLASSES))
    txt_file.write("\n")


def write_conf_matrix(confusion_matrix_path, epoch):
    with open(confusion_matrix_path, "a") as conf_matrix_file:
        conf_matrix_file.write(f"Epoch {epoch}\n")
        for class_name in CLASSES:
            conf_matrix_file.write(
                f"{class_name} Train Confusion Matrix:\n{metricts_conf_matrix[f'{class_name}_train_conf_matrix'].numpy()}\n"
            )
            conf_matrix_file.write(
                f"{class_name} Test Confusion Matrix:\n{metricts_conf_matrix[f'{class_name}_test_conf_matrix'].numpy()}\n"
            )
        conf_matrix_file.write("\n")


""" SCRIPT """

# scheduled_parameters = defaultdict(lambda: {}) -> This can be used to schedule hyperparameters
template = "Epoch {}"
for metrics_label in metrics_dict.keys():
    template += ", %s: {:.5f}" % metrics_label
if csv_path is not None:
    txt_file = open(txt_path, "w+")
    write_results_description_in_txt(txt_file)
    csv_file = open(csv_path, "w+")

    headers = ",".join(["Epoch"] + list(metrics_dict.keys()))
    csv_template = ",".join(["{}" for _ in range(len(metrics_dict) + 1)])
    csv_file.write(headers + "\n")

for epoch in range(EPOCHS):
    for metrics in metrics_dict.values():
        metrics.reset_states()
    for metric in metricts_conf_matrix.values():
        metric.assign(np.zeros((2, 2), dtype=np.int32))
    for _ in range(len(train_generator)):
        batch_elements = train_generator.next()
        features_param, labels_list = __get_elements_from_batch(batch_elements)
        train_step(features_param, *labels_list)
    for _ in range(len(test_generator)):
        batch_elements = test_generator.next()
        features_param, labels_list = __get_elements_from_batch(batch_elements)
        test_step(features_param, *labels_list)

    metrics_results = [metrics.result() for metrics in metrics_dict.values()]
    print(template.format(epoch, *metrics_results))
    write_conf_matrix(confusion_matrix_path, epoch)
    if csv_path is not None:
        csv_file.write(csv_template.format(epoch, *metrics_results) + "\n")
        csv_file.flush()

logits_model.save_weights("logits_model_weights.h5")

if csv_path is not None:
    csv_file.close()
    txt_file.close()


A saída de streaming foi truncada nas últimas 5000 linhas.
Sat Function Result:  0.838837862
Sat Function Result:  0.849625289
Sat Function Result:  0.810853
Sat Function Result:  0.852351367
Sat Function Result:  0.8317132
Sat Function Result:  0.854626894
Sat Function Result:  0.856662929
Sat Function Result:  0.835597038
Sat Function Result:  0.839102745
Sat Function Result:  0.828982353
Sat Function Result:  0.840716481
Sat Function Result:  0.844612956
Sat Function Result:  0.847975791
Sat Function Result:  0.827749491
Sat Function Result:  0.810276389
Sat Function Result:  0.828705549
Sat Function Result:  0.849695504
Sat Function Result:  0.840714574
Sat Function Result:  0.838198543
Sat Function Result:  0.836433649
Sat Function Result:  0.835553586
Sat Function Result:  0.813812
Sat Function Result:  0.844304
Sat Function Result:  0.825441658
Sat Function Result:  0.83445251
Sat Function Result:  0.83616364
Sat Function Result:  0.835200667
Sat Function Result:  0.827997446
Sa

# Testes dia 8/7

In [ ]:
logits_model = None
p = None

""" DATASET """

data_path = Path(DATA_ROOT, "C" + str(COMPLEXITY_DATASET))
labels = Path(data_path, "labels.csv")
images = Path(data_path, "images")

df = pd.read_csv(labels, index_col=0)
df["id"] = df["id"].apply(lambda x: f"{x}.jpg")
# CLASSES = df.columns.tolist()[1:]

CLASSES = [
    "Cafe",
    "MiscCommercial",
    "Awning",
    "Table",
    "VendingMachine",
    "Statue",
]

datagen = image.ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=0,
    width_shift_range=0,
    height_shift_range=0,
    shear_range=0,
    zoom_range=0,
    horizontal_flip=False,
    fill_mode="nearest",
)

train_generator = datagen.flow_from_dataframe(
    dataframe=df[:TRAIN_TEST_LIMIT],
    directory=str(images),
    x_col="id",
    y_col=CLASSES,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False,
)

test_generator = datagen.flow_from_dataframe(
    dataframe=df[TRAIN_TEST_LIMIT:TOTAL_EXAMPLES],
    directory=str(images),
    x_col="id",
    y_col=CLASSES,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False,
)


"""CNN MODEL"""


class CNNModel(keras.Model):

    def __init__(self, num_classes):
        super(CNNModel, self).__init__()
        self.conv1 = keras.layers.Conv2D(
            16, kernel_size=(3, 3), activation="relu", input_shape=(224, 224, 3)
        )
        self.conv2 = keras.layers.Conv2D(32, kernel_size=(3, 3), activation="relu")
        self.conv3 = keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu")
        self.maxpool = keras.layers.MaxPooling2D(pool_size=(2, 2))
        self.flatten = keras.layers.Flatten()
        self.dropout = keras.layers.Dropout(0.5)
        self.dense1 = keras.layers.Dense(64, activation="relu")
        self.dense2 = keras.layers.Dense(32, activation="relu")
        self.dense3 = keras.layers.Dense(16, activation="relu")
        self.concept_layer = keras.layers.Dense(4)
        self.output_layer = keras.layers.Dense(2)

    def call(self, inputs):
        x = self.conv1(inputs[0])
        x = self.maxpool(x)
        x = self.conv2(x)
        x = self.maxpool(x)
        x = self.conv3(x)
        x = self.maxpool(x)
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.dense1(x)
        x = self.dropout(x)
        x = self.dense2(x)
        x = self.dense3(x)
        concept_output = self.concept_layer(x)
        output = self.output_layer(concept_output)
        result = tf.concat([output, concept_output], axis=1)
        return result


""" LTN MODEL """

# Predicates
logits_model = CNNModel(num_classes=1)
p = ltn.Predicate.FromLogits(
    logits_model, activation_function="sigmoid", with_class_indexing=True
)
# pf = ltn.Predicate.FromLogits(logits_model, activation_function="softmax")

# Constants
class_cafe = ltn.Constant(CLASSES.index("Cafe"), trainable=False)
class_misc_commercial = ltn.Constant(CLASSES.index("MiscCommercial"), trainable=False)
class_awning = ltn.Constant(CLASSES.index("Awning"), trainable=False)
class_vending_machine = ltn.Constant(CLASSES.index("VendingMachine"), trainable=False)
class_statue = ltn.Constant(CLASSES.index("Statue"), trainable=False)
class_table = ltn.Constant(CLASSES.index("Table"), trainable=False)


# Operators
Not = ltn.Wrapper_Connective(ltn.fuzzy_ops.Not_Std())
And = ltn.Wrapper_Connective(ltn.fuzzy_ops.And_Prod())
Or = ltn.Wrapper_Connective(ltn.fuzzy_ops.Or_ProbSum())
Implies = ltn.Wrapper_Connective(ltn.fuzzy_ops.Implies_Reichenbach())
Equiv = ltn.Wrapper_Connective(
    ltn.fuzzy_ops.Equiv(ltn.fuzzy_ops.And_Prod(), ltn.fuzzy_ops.Implies_Reichenbach())
)
Forall = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMeanError(), semantics="forall")
Exists = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMean(), semantics="exists")

formula_aggregator = ltn.Wrapper_Formula_Aggregator(
    ltn.fuzzy_ops.Aggreg_pMeanError(p=2)
)


@tf.function()
def axioms(features_param, *args):
    x = ltn.Variable("x", features_param)
    x_cafe = ltn.Variable("x_cafe", features_param[args[CLASSES.index("Cafe")] == 1])
    x_not_cafe = ltn.Variable(
        "x_not_cafe", features_param[args[CLASSES.index("Cafe")] == 0]
    )
    x_misc_commercial = ltn.Variable(
        "x_misc_commercial", features_param[args[CLASSES.index("MiscCommercial")] == 1]
    )
    x_not_misc_commercial = ltn.Variable(
        "x_not_misc_commercial",
        features_param[args[CLASSES.index("MiscCommercial")] == 0],
    )
    axioms = [
        # Grounded Predicates
        Forall(x_cafe, p([x_cafe, class_cafe])),
        Forall(x_not_cafe, Not(p([x_not_cafe, class_cafe]))),
        Forall(x_misc_commercial, p([x_misc_commercial, class_misc_commercial])),
        Forall(
            x_not_misc_commercial,
            Not(p([x_not_misc_commercial, class_misc_commercial])),
        ),
        # Axioms
        # Cafe = Statue or Vending Machine
        Forall(
            x,
            Equiv(
                p([x, class_cafe]),
                Or(p([x, class_statue]), p([x, class_vending_machine])),
            ),
        ),
        # MiscCommercial = Awning and Table
        Forall(
            x,
            Equiv(
                p([x, class_misc_commercial]),
                And(p([x, class_awning]), p([x, class_table])),
            ),
        ),
    ]
    sat_level = formula_aggregator(axioms).tensor
    return sat_level


def __get_elements_from_batch(batch_elements):
    features_calculated = batch_elements[0]
    inverted_labels_list = batch_elements[1]
    labels_list = np.transpose(inverted_labels_list)
    return features_calculated, labels_list


for _ in range(len(train_generator)):
    batch_elements = train_generator.next()
    features, labels_list = __get_elements_from_batch(batch_elements)
    print("Initial sat level %.5f" % axioms(batch_elements[0], *labels_list))
    break


""" TRAINING FUNCTIONS """

metrics_dict = {
    "train_sat_kb": keras.metrics.Mean(name="train_sat_kb"),
    "test_sat_kb": keras.metrics.Mean(name="test_sat_kb"),
}


metricts_conf_matrix = {
    f"{class_name}_train_conf_matrix": tf.Variable(
        initial_value=np.zeros((2, 2), dtype=np.int32)
    )
    for class_name in CLASSES
}
metricts_conf_matrix_test = {
    f"{class_name}_test_conf_matrix": tf.Variable(
        initial_value=np.zeros((2, 2), dtype=np.int32)
    )
    for class_name in CLASSES
}
metricts_conf_matrix.update(metricts_conf_matrix_test)

commercial_true = {
    f"{class_name}_when_MiscCommercial_true_conf_matrix": tf.Variable(
        initial_value=np.zeros((2, 2), dtype=np.int32)
    )
    for class_name in ["Awning", "Table"]
}
metricts_conf_matrix.update(commercial_true)

cafe_True = {
    f"{class_name}_when_Cafe_true_conf_matrix": tf.Variable(
        initial_value=np.zeros((2, 2), dtype=np.int32)
    )
    for class_name in ["VendingMachine", "Statue"]
}
metricts_conf_matrix.update(cafe_True)


class_train_accuracy_metric = {
    f"{class_name.lower()}_train_accuracy": keras.metrics.Mean(
        name=f"{class_name.lower()}_train_accuracy"
    )
    for class_name in CLASSES
}
metrics_dict.update(class_train_accuracy_metric)

class_test_accuracy_metric = {
    f"{class_name.lower()}_test_accuracy": keras.metrics.Mean(
        name=f"{class_name.lower()}_test_accuracy"
    )
    for class_name in CLASSES
}
metrics_dict.update(class_test_accuracy_metric)


optimizer = keras.optimizers.legacy.Adam(learning_rate=LEARNING_RATE)


@tf.function()
def train_step(*args):
    features_param = args[0]
    labels_list = args[1:]
    with tf.GradientTape() as tape:
        sat = axioms(features_param, *labels_list)
        loss = 1.0 - sat
    gradients = tape.gradient(loss, p.trainable_variables)
    optimizer.apply_gradients(zip(gradients, p.trainable_variables))
    tf.print("Sat Function Result: ", sat)
    metrics_dict["train_sat_kb"](sat)
    predictions = logits_model([features_param])
    calculate_accuracy_for_each_class("train", predictions, labels_list)
    update_confusion_matrix("train", predictions, labels_list)


@tf.function()
def test_step(*args):
    features_param = args[0]
    labels_list = args[1:]
    sat_kb = axioms(features_param, *labels_list)
    metrics_dict["test_sat_kb"](sat_kb)
    predictions = logits_model([features_param])
    calculate_accuracy_for_each_class("test", predictions, labels_list)
    update_confusion_matrix("test", predictions, labels_list)


""" AUXILIARY FUNCTIONS """


def calculate_accuracy_for_each_class(phase: str, predictions, labels_param):
    for i, class_name in enumerate(CLASSES):
        class_labels = labels_param[i]
        y_pred = predictions[:, i]
        y_pred = tf.math.sigmoid(y_pred)
        y_pred = tf.cast(y_pred > 0.5, tf.float32)
        cast_class_labels = tf.cast(class_labels, tf.float32)
        accuracy = tf.reduce_mean(
            tf.cast(tf.equal(y_pred, cast_class_labels), dtype=tf.float32)
        )
        metrics_dict[f"{class_name.lower()}_{phase}_accuracy"](accuracy)


def update_confusion_matrix(phase: str, predictions, labels_param):
    for class_name in CLASSES:
        class_labels = tf.cast(labels_param[CLASSES.index(class_name)], tf.int32)
        y_pred = predictions[:, CLASSES.index(class_name)]
        y_pred = tf.math.sigmoid(y_pred)
        y_preds = tf.cast(y_pred > 0.5, tf.int32)
        confusion_matrix = tf.math.confusion_matrix(
            class_labels, y_preds, num_classes=2
        )
        metricts_conf_matrix[f"{class_name}_{phase}_conf_matrix"].assign_add(
            confusion_matrix
        )
        if class_name in ["Awning", "Table"] and phase == "train":
            class_labels_true = class_labels[
                labels_param[CLASSES.index("MiscCommercial")] == 1
            ]
            y_preds_true = y_preds[labels_param[CLASSES.index("MiscCommercial")] == 1]
            confusion_matrix = tf.math.confusion_matrix(
                class_labels_true, y_preds_true, num_classes=2
            )
            metricts_conf_matrix[
                f"{class_name}_when_MiscCommercial_true_conf_matrix"
            ].assign_add(confusion_matrix)
        if class_name in ["VendingMachine", "Statue"] and phase == "train":
            class_labels_true = class_labels[labels_param[CLASSES.index("Cafe")] == 1]
            y_preds_true = y_preds[labels_param[CLASSES.index("Cafe")] == 1]
            confusion_matrix = tf.math.confusion_matrix(
                class_labels_true, y_preds_true, num_classes=2
            )
            metricts_conf_matrix[f"{class_name}_when_Cafe_true_conf_matrix"].assign_add(
                confusion_matrix
            )


def write_results_description_in_txt(txt_file):
    txt_file.write("Results Description\n")
    txt_file.write("Number of examples: %d\n" % TOTAL_EXAMPLES)
    txt_file.write("Train/Test limit: %d\n" % TRAIN_TEST_LIMIT)
    txt_file.write("Epochs: %d\n" % EPOCHS)
    txt_file.write("Complexity of the Dataset: %d\n" % COMPLEXITY_DATASET)
    txt_file.write("Batch size: %d\n" % BATCH_SIZE)
    txt_file.write("Learning rate: %.5f\n" % LEARNING_RATE)
    txt_file.write("Number of classes: %d\n" % len(CLASSES))
    txt_file.write("\n")


def write_conf_matrix(confusion_matrix_path, epoch):
    with open(confusion_matrix_path, "a") as conf_matrix_file:
        conf_matrix_file.write(f"Epoch {epoch}\n")
        for class_name in CLASSES:
            conf_matrix_file.write(
                f"{class_name} Train Confusion Matrix:\n{metricts_conf_matrix[f'{class_name}_train_conf_matrix'].numpy()}\n"
            )
            conf_matrix_file.write(
                f"{class_name} Test Confusion Matrix:\n{metricts_conf_matrix[f'{class_name}_test_conf_matrix'].numpy()}\n"
            )
            if class_name in ["VendingMachine", "Statue"]:
                conf_matrix_file.write(
                    f"{class_name} Cafe=1 Matrix:\n{metricts_conf_matrix[f'{class_name}_when_Cafe_true_conf_matrix'].numpy()}\n"
                )
            if class_name in ["Awning", "Table"]:
                conf_matrix_file.write(
                    f"{class_name} MiscCommercial=1 Confusion Matrix:\n{metricts_conf_matrix[f'{class_name}_when_MiscCommercial_true_conf_matrix'].numpy()}\n"
                )
        conf_matrix_file.write("\n")


""" SCRIPT """

# scheduled_parameters = defaultdict(lambda: {}) -> This can be used to schedule hyperparameters
template = "Epoch {}"
for metrics_label in metrics_dict.keys():
    template += ", %s: {:.5f}" % metrics_label
if csv_path is not None:
    txt_file = open(txt_path, "w+")
    write_results_description_in_txt(txt_file)
    csv_file = open(csv_path, "w+")

    headers = ",".join(["Epoch"] + list(metrics_dict.keys()))
    csv_template = ",".join(["{}" for _ in range(len(metrics_dict) + 1)])
    csv_file.write(headers + "\n")

for epoch in range(EPOCHS):
    for metrics in metrics_dict.values():
        metrics.reset_states()
    for metric in metricts_conf_matrix.values():
        metric.assign(np.zeros((2, 2), dtype=np.int32))
    for _ in range(len(train_generator)):
        batch_elements = train_generator.next()
        features_param, labels_list = __get_elements_from_batch(batch_elements)
        train_step(features_param, *labels_list)
    for _ in range(len(test_generator)):
        batch_elements = test_generator.next()
        features_param, labels_list = __get_elements_from_batch(batch_elements)
        test_step(features_param, *labels_list)

    metrics_results = [metrics.result() for metrics in metrics_dict.values()]
    print(template.format(epoch, *metrics_results))
    write_conf_matrix(confusion_matrix_path, epoch)
    if csv_path is not None:
        csv_file.write(csv_template.format(epoch, *metrics_results) + "\n")
        csv_file.flush()

logits_model.save_weights("logits_model_weights.h5")

if csv_path is not None:
    csv_file.close()
    txt_file.close()


Found 12000 validated image filenames.
Found 3000 validated image filenames.


/usr/local/lib/python3.10/dist-packages/keras/src/layers/convolutional/base_conv.py:99: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(


AttributeError: 'DataFrameIterator' object has no attribute 'next'

#Testes dia 5/9

In [ ]:
logits_model = None
p = None

best_satisfiability = 0
best_epoch = 0

""" DATASET """

data_path = Path(DATA_ROOT, "C" + str(COMPLEXITY_DATASET))
labels = Path(data_path, "labels.csv")
images = Path(data_path, "images")

df = pd.read_csv(labels, index_col=0)
df["id"] = df["id"].apply(lambda x: f"{x}.jpg")
# CLASSES = df.columns.tolist()[1:]

CLASSES = [
    "Cafe",
    "MiscCommercial",
    "Awning",
    "Table",
    "VendingMachine",
    "Statue",
]

datagen = image.ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=0,
    width_shift_range=0,
    height_shift_range=0,
    shear_range=0,
    zoom_range=0,
    horizontal_flip=False,
    fill_mode="nearest",
)

train_generator = datagen.flow_from_dataframe(
    dataframe=df[:TRAIN_TEST_LIMIT],
    directory=str(images),
    x_col="id",
    y_col=CLASSES,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False,
)

test_generator = datagen.flow_from_dataframe(
    dataframe=df[TRAIN_TEST_LIMIT:TOTAL_EXAMPLES],
    directory=str(images),
    x_col="id",
    y_col=CLASSES,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False,
)


"""CNN MODEL"""


class CNNModel(keras.Model):

    def __init__(self, num_classes):
        super(CNNModel, self).__init__()
        self.conv1 = keras.layers.Conv2D(
            16, kernel_size=(3, 3), activation="relu", input_shape=(224, 224, 3)
        )
        self.conv2 = keras.layers.Conv2D(32, kernel_size=(3, 3), activation="relu")
        self.conv3 = keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu")
        self.maxpool = keras.layers.MaxPooling2D(pool_size=(2, 2))
        self.flatten = keras.layers.Flatten()
        self.dropout = keras.layers.Dropout(0.5)
        self.dense1 = keras.layers.Dense(64, activation="relu")
        self.dense2 = keras.layers.Dense(32, activation="relu")
        self.dense3 = keras.layers.Dense(16, activation="relu")
        self.concept_layer = keras.layers.Dense(4)
        self.output_layer = keras.layers.Dense(2)

    def call(self, inputs):
        x = self.conv1(inputs[0])
        x = self.maxpool(x)
        x = self.conv2(x)
        x = self.maxpool(x)
        x = self.conv3(x)
        x = self.maxpool(x)
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.dense1(x)
        x = self.dropout(x)
        x = self.dense2(x)
        x = self.dense3(x)
        concept_output = self.concept_layer(x)
        output = self.output_layer(concept_output)
        result = tf.concat([output, concept_output], axis=1)
        return result


""" LTN MODEL """

# Predicates
logits_model = CNNModel(num_classes=1)
p = ltn.Predicate.FromLogits(
    logits_model, activation_function="sigmoid", with_class_indexing=True
)
# pf = ltn.Predicate.FromLogits(logits_model, activation_function="softmax")

# Constants
class_cafe = ltn.Constant(CLASSES.index("Cafe"), trainable=False)
class_misc_commercial = ltn.Constant(CLASSES.index("MiscCommercial"), trainable=False)
class_awning = ltn.Constant(CLASSES.index("Awning"), trainable=False)
class_vending_machine = ltn.Constant(CLASSES.index("VendingMachine"), trainable=False)
class_statue = ltn.Constant(CLASSES.index("Statue"), trainable=False)
class_table = ltn.Constant(CLASSES.index("Table"), trainable=False)


# Operators
Not = ltn.Wrapper_Connective(ltn.fuzzy_ops.Not_Std())
And = ltn.Wrapper_Connective(ltn.fuzzy_ops.And_Prod())
Or = ltn.Wrapper_Connective(ltn.fuzzy_ops.Or_ProbSum())
Implies = ltn.Wrapper_Connective(ltn.fuzzy_ops.Implies_Reichenbach())
Equiv = ltn.Wrapper_Connective(
    ltn.fuzzy_ops.Equiv(ltn.fuzzy_ops.And_Prod(), ltn.fuzzy_ops.Implies_Reichenbach())
)
Forall = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMeanError(), semantics="forall")
Exists = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMean(), semantics="exists")

formula_aggregator = ltn.Wrapper_Formula_Aggregator(
    ltn.fuzzy_ops.Aggreg_pMeanError(p=2)
)


@tf.function()
def axioms(features_param, *args):
    x = ltn.Variable("x", features_param)
    x_cafe = ltn.Variable("x_cafe", features_param[args[CLASSES.index("Cafe")] == 1])
    x_not_cafe = ltn.Variable(
        "x_not_cafe", features_param[args[CLASSES.index("Cafe")] == 0]
    )
    x_misc_commercial = ltn.Variable(
        "x_misc_commercial", features_param[args[CLASSES.index("MiscCommercial")] == 1]
    )
    x_not_misc_commercial = ltn.Variable(
        "x_not_misc_commercial",
        features_param[args[CLASSES.index("MiscCommercial")] == 0],
    )
    axioms = [
        # Grounded Predicates
        Forall(x_cafe, p([x_cafe, class_cafe])),
        Forall(x_not_cafe, Not(p([x_not_cafe, class_cafe]))),
        Forall(x_misc_commercial, p([x_misc_commercial, class_misc_commercial])),
        Forall(
            x_not_misc_commercial,
            Not(p([x_not_misc_commercial, class_misc_commercial])),
        ),
        # Axioms
        # Cafe = Statue or Vending Machine
        Forall(
            x,
            Equiv(
                p([x, class_cafe]),
                Or(p([x, class_statue]), p([x, class_vending_machine])),
            ),
        ),
        # MiscCommercial = Awning and Table
        Forall(
            x,
            Equiv(
                p([x, class_misc_commercial]),
                And(p([x, class_awning]), p([x, class_table])),
            ),
        ),
    ]
    sat_level = formula_aggregator(axioms).tensor
    return sat_level


def __get_elements_from_batch(batch_elements):
    features_calculated = batch_elements[0]
    inverted_labels_list = batch_elements[1]
    labels_list = np.transpose(inverted_labels_list)
    return features_calculated, labels_list


for _ in range(len(train_generator)):
    batch_elements = train_generator.next()
    features, labels_list = __get_elements_from_batch(batch_elements)
    print("Initial sat level %.5f" % axioms(batch_elements[0], *labels_list))
    break


""" TRAINING FUNCTIONS """

metrics_dict = {
    "train_sat_kb": keras.metrics.Mean(name="train_sat_kb"),
    "test_sat_kb": keras.metrics.Mean(name="test_sat_kb"),
}


metricts_conf_matrix = {
    f"{class_name}_train_conf_matrix": tf.Variable(
        initial_value=np.zeros((2, 2), dtype=np.int32)
    )
    for class_name in CLASSES
}
metricts_conf_matrix_test = {
    f"{class_name}_test_conf_matrix": tf.Variable(
        initial_value=np.zeros((2, 2), dtype=np.int32)
    )
    for class_name in CLASSES
}
metricts_conf_matrix.update(metricts_conf_matrix_test)


class_train_accuracy_metric = {
    f"{class_name.lower()}_train_accuracy": keras.metrics.Mean(
        name=f"{class_name.lower()}_train_accuracy"
    )
    for class_name in CLASSES
}
metrics_dict.update(class_train_accuracy_metric)

class_test_accuracy_metric = {
    f"{class_name.lower()}_test_accuracy": keras.metrics.Mean(
        name=f"{class_name.lower()}_test_accuracy"
    )
    for class_name in CLASSES
}
metrics_dict.update(class_test_accuracy_metric)


optimizer = keras.optimizers.legacy.Adam(learning_rate=LEARNING_RATE)


@tf.function()
def train_step(*args):
    features_param = args[0]
    labels_list = args[1:]
    with tf.GradientTape() as tape:
        sat = axioms(features_param, *labels_list)
        loss = 1.0 - sat
    gradients = tape.gradient(loss, p.trainable_variables)
    optimizer.apply_gradients(zip(gradients, p.trainable_variables))
    tf.print("Sat Function Result: ", sat)
    metrics_dict["train_sat_kb"](sat)
    predictions = logits_model([features_param])
    calculate_accuracy_for_each_class("train", predictions, labels_list)
    update_confusion_matrix("train", predictions, labels_list)


@tf.function()
def test_step(*args):
    features_param = args[0]
    labels_list = args[1:]
    sat_kb = axioms(features_param, *labels_list)
    metrics_dict["test_sat_kb"](sat_kb)
    predictions = logits_model([features_param])
    calculate_accuracy_for_each_class("test", predictions, labels_list)
    update_confusion_matrix("test", predictions, labels_list)


""" AUXILIARY FUNCTIONS """


def calculate_accuracy_for_each_class(phase: str, predictions, labels_param):
    for i, class_name in enumerate(CLASSES):
        class_labels = labels_param[i]
        y_pred = predictions[:, i]
        y_pred = tf.math.sigmoid(y_pred)
        y_pred = tf.cast(y_pred > 0.5, tf.float32)
        cast_class_labels = tf.cast(class_labels, tf.float32)
        accuracy = tf.reduce_mean(
            tf.cast(tf.equal(y_pred, cast_class_labels), dtype=tf.float32)
        )
        metrics_dict[f"{class_name.lower()}_{phase}_accuracy"](accuracy)


def update_confusion_matrix(phase: str, predictions, labels_param):
    for class_name in CLASSES:
        class_labels = tf.cast(labels_param[CLASSES.index(class_name)], tf.int32)
        y_pred = predictions[:, CLASSES.index(class_name)]
        y_pred = tf.math.sigmoid(y_pred)
        y_preds = tf.cast(y_pred > 0.5, tf.int32)
        confusion_matrix = tf.math.confusion_matrix(
            class_labels, y_preds, num_classes=2
        )
        metricts_conf_matrix[f"{class_name}_{phase}_conf_matrix"].assign_add(
            confusion_matrix
        )


def write_results_description_in_txt(txt_file):
    txt_file.write("Results Description\n")
    txt_file.write("Number of examples: %d\n" % TOTAL_EXAMPLES)
    txt_file.write("Train/Test limit: %d\n" % TRAIN_TEST_LIMIT)
    txt_file.write("Epochs: %d\n" % EPOCHS)
    txt_file.write("Complexity of the Dataset: %d\n" % COMPLEXITY_DATASET)
    txt_file.write("Batch size: %d\n" % BATCH_SIZE)
    txt_file.write("Learning rate: %.5f\n" % LEARNING_RATE)
    txt_file.write("Number of classes: %d\n" % len(CLASSES))
    txt_file.write("\n")


def write_conf_matrix(confusion_matrix_path, epoch):
    with open(confusion_matrix_path, "a") as conf_matrix_file:
        conf_matrix_file.write(f"Epoch {epoch}\n")
        for class_name in CLASSES:
            conf_matrix_file.write(
                f"{class_name} Train Confusion Matrix:\n{metricts_conf_matrix[f'{class_name}_train_conf_matrix'].numpy()}\n"
            )
            conf_matrix_file.write(
                f"{class_name} Test Confusion Matrix:\n{metricts_conf_matrix[f'{class_name}_test_conf_matrix'].numpy()}\n"
            )
        conf_matrix_file.write("\n")


""" SCRIPT """

# scheduled_parameters = defaultdict(lambda: {}) -> This can be used to schedule hyperparameters
template = "Epoch {}"
for metrics_label in metrics_dict.keys():
    template += ", %s: {:.5f}" % metrics_label
if csv_path is not None:
    txt_file = open(txt_path, "w+")
    write_results_description_in_txt(txt_file)
    csv_file = open(csv_path, "w+")

    headers = ",".join(["Epoch"] + list(metrics_dict.keys()))
    csv_template = ",".join(["{}" for _ in range(len(metrics_dict) + 1)])
    csv_file.write(headers + "\n")

for epoch in range(EPOCHS):
    for metrics in metrics_dict.values():
        metrics.reset_states()
    for metric in metricts_conf_matrix.values():
        metric.assign(np.zeros((2, 2), dtype=np.int32))
    for _ in range(len(train_generator)):
        batch_elements = train_generator.next()
        features_param, labels_list = __get_elements_from_batch(batch_elements)
        train_step(features_param, *labels_list)
    for _ in range(len(test_generator)):
        batch_elements = test_generator.next()
        features_param, labels_list = __get_elements_from_batch(batch_elements)
        test_step(features_param, *labels_list)


    metrics_results = [metrics.result() for metrics in metrics_dict.values()]
    print(template.format(epoch, *metrics_results))
    sat_template = "{:.5f}"
    cur_sat = float(sat_template.format(metrics_results[0]))
    if cur_sat > best_satisfiability:
        logits_model.save_weights("logits_model_weights.h5",overwrite=True)
        best_epoch = epoch
        best_satisfiability = cur_sat
        print(f"NEW BEST EPOCH -> {best_epoch}")
    else:
        print(f"BEST EPOCH -> {best_epoch}")

    write_conf_matrix(confusion_matrix_path, epoch)
    if csv_path is not None:
        csv_file.write(csv_template.format(epoch, *metrics_results) + "\n")
        csv_file.flush()

if csv_path is not None:
    csv_file.close()
    txt_file.close()


A saída de streaming foi truncada nas últimas 5000 linhas.
Sat Function Result:  0.860829175
Sat Function Result:  0.959457457
Sat Function Result:  0.999691129
Sat Function Result:  0.877249181
Sat Function Result:  0.998177
Sat Function Result:  0.882119417
Sat Function Result:  0.9655056
Sat Function Result:  0.85804826
Sat Function Result:  0.916903734
Sat Function Result:  0.902838588
Sat Function Result:  0.845729053
Sat Function Result:  0.81677103
Sat Function Result:  0.999690711
Sat Function Result:  0.99943763
Sat Function Result:  0.953325629
Sat Function Result:  0.891054094
Sat Function Result:  0.903402388
Sat Function Result:  0.999690533
Sat Function Result:  0.916680813
Sat Function Result:  0.999689162
Sat Function Result:  0.999668717
Sat Function Result:  0.939771533
Sat Function Result:  0.941301942
Sat Function Result:  0.919121325
Sat Function Result:  0.912738085
Sat Function Result:  0.890860677
Sat Function Result:  0.822931647
Sat Function Result:  0.9012396

# Testes dia 9/9

In [ ]:
logits_model = None
p = None

best_satisfiability = 0
best_epoch = 0

""" DATASET """

data_path = Path(DATA_ROOT, "C" + str(COMPLEXITY_DATASET))
labels = Path(data_path, "labels.csv")
images = Path(data_path, "images")

df = pd.read_csv(labels, index_col=0)
df["id"] = df["id"].apply(lambda x: f"{x}.jpg")
# CLASSES = df.columns.tolist()[1:]

CLASSES = [
    "PowerPlant",
    "Chimney",
    "Pipe",
]

datagen = image.ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=0,
    width_shift_range=0,
    height_shift_range=0,
    shear_range=0,
    zoom_range=0,
    horizontal_flip=False,
    fill_mode="nearest",
)

train_generator = datagen.flow_from_dataframe(
    dataframe=df[:TRAIN_TEST_LIMIT],
    directory=str(images),
    x_col="id",
    y_col=CLASSES,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False,
)

test_generator = datagen.flow_from_dataframe(
    dataframe=df[TRAIN_TEST_LIMIT:TOTAL_EXAMPLES],
    directory=str(images),
    x_col="id",
    y_col=CLASSES,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False,
)


"""CNN MODEL"""


class CNNModel(keras.Model):

    def __init__(self, num_classes):
        super(CNNModel, self).__init__()
        self.conv1 = keras.layers.Conv2D(
            16, kernel_size=(3, 3), activation="relu", input_shape=(224, 224, 3)
        )
        self.conv2 = keras.layers.Conv2D(32, kernel_size=(3, 3), activation="relu")
        self.conv3 = keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu")
        self.maxpool = keras.layers.MaxPooling2D(pool_size=(2, 2))
        self.flatten = keras.layers.Flatten()
        self.dropout = keras.layers.Dropout(0.5)
        self.dense1 = keras.layers.Dense(64, activation="relu")
        self.dense2 = keras.layers.Dense(32, activation="relu")
        self.dense3 = keras.layers.Dense(16, activation="relu")
        self.concept_layer = keras.layers.Dense(2)
        self.output_layer = keras.layers.Dense(1)

    def call(self, inputs):
        x = self.conv1(inputs[0])
        x = self.maxpool(x)
        x = self.conv2(x)
        x = self.maxpool(x)
        x = self.conv3(x)
        x = self.maxpool(x)
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.dense1(x)
        x = self.dropout(x)
        x = self.dense2(x)
        x = self.dense3(x)
        concept_output = self.concept_layer(x)
        output = self.output_layer(concept_output)
        result = tf.concat([output, concept_output], axis=1)
        return result


""" LTN MODEL """

# Predicates
logits_model = CNNModel(num_classes=1)
p = ltn.Predicate.FromLogits(
    logits_model, activation_function="sigmoid", with_class_indexing=True
)
# pf = ltn.Predicate.FromLogits(logits_model, activation_function="softmax")

# Constants
class_powerplant = ltn.Constant(CLASSES.index("PowerPlant"), trainable=False)
class_chimney = ltn.Constant(CLASSES.index("Chimney"), trainable=False)
class_pipe = ltn.Constant(CLASSES.index("Pipe"), trainable=False)


# Operators
Not = ltn.Wrapper_Connective(ltn.fuzzy_ops.Not_Std())
And = ltn.Wrapper_Connective(ltn.fuzzy_ops.And_Prod())
Or = ltn.Wrapper_Connective(ltn.fuzzy_ops.Or_ProbSum())
Implies = ltn.Wrapper_Connective(ltn.fuzzy_ops.Implies_Reichenbach())
Equiv = ltn.Wrapper_Connective(
    ltn.fuzzy_ops.Equiv(ltn.fuzzy_ops.And_Prod(), ltn.fuzzy_ops.Implies_Reichenbach())
)
Forall = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMeanError(), semantics="forall")
Exists = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMean(), semantics="exists")

formula_aggregator = ltn.Wrapper_Formula_Aggregator(
    ltn.fuzzy_ops.Aggreg_pMeanError(p=2)
)


@tf.function()
def axioms(features_param, *args):
    x = ltn.Variable("x", features_param)
    x_powerplant = ltn.Variable(
        "x_powerplant", features_param[args[CLASSES.index("PowerPlant")] == 1]
    )
    x_not_powerplant = ltn.Variable(
        "x_not_powerplant",
        features_param[args[CLASSES.index("PowerPlant")] == 0],
    )
    axioms = [
        # Grounded Predicates
        Forall(x_powerplant, p([x_powerplant, class_powerplant])),
        Forall(
            x_not_powerplant,
            Not(p([x_not_powerplant, class_powerplant])),
        ),
        # Axioms
        # PowerPlant = Chimney and Pipe
        Forall(
            x,
            Equiv(
                p([x, class_powerplant]),
                And(p([x, class_chimney]), p([x, class_pipe])),
            ),
        ),
    ]
    sat_level = formula_aggregator(axioms).tensor
    return sat_level


def __get_elements_from_batch(batch_elements):
    features_calculated = batch_elements[0]
    inverted_labels_list = batch_elements[1]
    labels_list = np.transpose(inverted_labels_list)
    return features_calculated, labels_list


for _ in range(len(train_generator)):
    batch_elements = train_generator.next()
    features, labels_list = __get_elements_from_batch(batch_elements)
    print("Initial sat level %.5f" % axioms(batch_elements[0], *labels_list))
    break


""" TRAINING FUNCTIONS """

metrics_dict = {
    "train_sat_kb": keras.metrics.Mean(name="train_sat_kb"),
    "test_sat_kb": keras.metrics.Mean(name="test_sat_kb"),
}


metricts_conf_matrix = {
    f"{class_name}_train_conf_matrix": tf.Variable(
        initial_value=np.zeros((2, 2), dtype=np.int32)
    )
    for class_name in CLASSES
}
metricts_conf_matrix_test = {
    f"{class_name}_test_conf_matrix": tf.Variable(
        initial_value=np.zeros((2, 2), dtype=np.int32)
    )
    for class_name in CLASSES
}
metricts_conf_matrix.update(metricts_conf_matrix_test)


class_train_accuracy_metric = {
    f"{class_name.lower()}_train_accuracy": keras.metrics.Mean(
        name=f"{class_name.lower()}_train_accuracy"
    )
    for class_name in CLASSES
}
metrics_dict.update(class_train_accuracy_metric)

class_test_accuracy_metric = {
    f"{class_name.lower()}_test_accuracy": keras.metrics.Mean(
        name=f"{class_name.lower()}_test_accuracy"
    )
    for class_name in CLASSES
}
metrics_dict.update(class_test_accuracy_metric)


optimizer = keras.optimizers.legacy.Adam(learning_rate=LEARNING_RATE)


@tf.function()
def train_step(*args):
    features_param = args[0]
    labels_list = args[1:]
    with tf.GradientTape() as tape:
        sat = axioms(features_param, *labels_list)
        loss = 1.0 - sat
    gradients = tape.gradient(loss, p.trainable_variables)
    optimizer.apply_gradients(zip(gradients, p.trainable_variables))
    tf.print("Sat Function Result: ", sat)
    metrics_dict["train_sat_kb"](sat)
    predictions = logits_model([features_param])
    calculate_accuracy_for_each_class("train", predictions, labels_list)
    update_confusion_matrix("train", predictions, labels_list)


@tf.function()
def test_step(*args):
    features_param = args[0]
    labels_list = args[1:]
    sat_kb = axioms(features_param, *labels_list)
    metrics_dict["test_sat_kb"](sat_kb)
    predictions = logits_model([features_param])
    calculate_accuracy_for_each_class("test", predictions, labels_list)
    update_confusion_matrix("test", predictions, labels_list)


""" AUXILIARY FUNCTIONS """


def calculate_accuracy_for_each_class(phase: str, predictions, labels_param):
    for i, class_name in enumerate(CLASSES):
        class_labels = labels_param[i]
        y_pred = predictions[:, i]
        y_pred = tf.math.sigmoid(y_pred)
        y_pred = tf.cast(y_pred > 0.5, tf.float32)
        cast_class_labels = tf.cast(class_labels, tf.float32)
        accuracy = tf.reduce_mean(
            tf.cast(tf.equal(y_pred, cast_class_labels), dtype=tf.float32)
        )
        metrics_dict[f"{class_name.lower()}_{phase}_accuracy"](accuracy)


def update_confusion_matrix(phase: str, predictions, labels_param):
    for class_name in CLASSES:
        class_labels = tf.cast(labels_param[CLASSES.index(class_name)], tf.int32)
        y_pred = predictions[:, CLASSES.index(class_name)]
        y_pred = tf.math.sigmoid(y_pred)
        y_preds = tf.cast(y_pred > 0.5, tf.int32)
        confusion_matrix = tf.math.confusion_matrix(
            class_labels, y_preds, num_classes=2
        )
        metricts_conf_matrix[f"{class_name}_{phase}_conf_matrix"].assign_add(
            confusion_matrix
        )


def write_results_description_in_txt(txt_file):
    txt_file.write("Results Description\n")
    txt_file.write("Number of examples: %d\n" % TOTAL_EXAMPLES)
    txt_file.write("Train/Test limit: %d\n" % TRAIN_TEST_LIMIT)
    txt_file.write("Epochs: %d\n" % EPOCHS)
    txt_file.write("Complexity of the Dataset: %d\n" % COMPLEXITY_DATASET)
    txt_file.write("Batch size: %d\n" % BATCH_SIZE)
    txt_file.write("Learning rate: %.5f\n" % LEARNING_RATE)
    txt_file.write("Number of classes: %d\n" % len(CLASSES))
    txt_file.write("\n")


def write_conf_matrix(confusion_matrix_path, epoch):
    with open(confusion_matrix_path, "a") as conf_matrix_file:
        conf_matrix_file.write(f"Epoch {epoch}\n")
        for class_name in CLASSES:
            conf_matrix_file.write(
                f"{class_name} Train Confusion Matrix:\n{metricts_conf_matrix[f'{class_name}_train_conf_matrix'].numpy()}\n"
            )
            conf_matrix_file.write(
                f"{class_name} Test Confusion Matrix:\n{metricts_conf_matrix[f'{class_name}_test_conf_matrix'].numpy()}\n"
            )
        conf_matrix_file.write("\n")


""" SCRIPT """

# scheduled_parameters = defaultdict(lambda: {}) -> This can be used to schedule hyperparameters
template = "Epoch {}"
for metrics_label in metrics_dict.keys():
    template += ", %s: {:.5f}" % metrics_label
if csv_path is not None:
    txt_file = open(txt_path, "w+")
    write_results_description_in_txt(txt_file)
    csv_file = open(csv_path, "w+")

    headers = ",".join(["Epoch"] + list(metrics_dict.keys()))
    csv_template = ",".join(["{}" for _ in range(len(metrics_dict) + 1)])
    csv_file.write(headers + "\n")

for epoch in range(EPOCHS):
    for metrics in metrics_dict.values():
        metrics.reset_states()
    for metric in metricts_conf_matrix.values():
        metric.assign(np.zeros((2, 2), dtype=np.int32))
    for _ in range(len(train_generator)):
        batch_elements = train_generator.next()
        features_param, labels_list = __get_elements_from_batch(batch_elements)
        train_step(features_param, *labels_list)
    for _ in range(len(test_generator)):
        batch_elements = test_generator.next()
        features_param, labels_list = __get_elements_from_batch(batch_elements)
        test_step(features_param, *labels_list)


    metrics_results = [metrics.result() for metrics in metrics_dict.values()]
    print(template.format(epoch, *metrics_results))
    sat_template = "{:.5f}"
    cur_sat = float(sat_template.format(metrics_results[0]))
    if cur_sat > best_satisfiability:
        logits_model.save_weights("logits_model_weights.h5",overwrite=True)
        best_epoch = epoch
        best_satisfiability = cur_sat
        print(f"NEW BEST EPOCH -> {best_epoch}")
    else:
        print(f"BEST EPOCH -> {best_epoch}")

    write_conf_matrix(confusion_matrix_path, epoch)
    if csv_path is not None:
        csv_file.write(csv_template.format(epoch, *metrics_results) + "\n")
        csv_file.flush()

if csv_path is not None:
    csv_file.close()
    txt_file.close()


A saída de streaming foi truncada nas últimas 5000 linhas.
Sat Function Result:  0.888835
Sat Function Result:  0.917132854
Sat Function Result:  0.999709785
Sat Function Result:  0.865157366
Sat Function Result:  0.885620952
Sat Function Result:  0.921382189
Sat Function Result:  0.968061924
Sat Function Result:  0.894970894
Sat Function Result:  0.999712646
Sat Function Result:  0.888841927
Sat Function Result:  0.845654786
Sat Function Result:  0.83993876
Sat Function Result:  0.865243912
Sat Function Result:  0.999708414
Sat Function Result:  0.886718035
Sat Function Result:  0.887798667
Sat Function Result:  0.999707758
Sat Function Result:  0.999711812
Sat Function Result:  0.999710143
Sat Function Result:  0.861279786
Sat Function Result:  0.920644641
Sat Function Result:  0.891805112
Sat Function Result:  0.865115523
Sat Function Result:  0.999713957
Sat Function Result:  0.888842046
Sat Function Result:  0.859926224
Sat Function Result:  0.918300331
Sat Function Result:  0.865

KeyboardInterrupt: 